# MLP + numerical-feature Transformer: whole-domain NetFlow unlearning

This is a complete, self-contained teaching notebook for leave-one-domain-out machine unlearning. It loads four standardized NetFlow datasets through the same local-first/Kaggle fallback used previously, applies one fixed 47-feature schema and binary label convention, makes independent grouped train/validation/test splits, and trains both an MLP and an FT-style numerical-feature Transformer. For every possible forgotten dataset, it compares the untouched original model, full retained-only retraining, the earlier amnesiac rollback baseline, canonical AAAI Selective Synaptic Dampening (SSD) at a published CIFAR/ResNet threshold *and* across an explicit α sweep, original-objective SCRUB with privacy rewind (SCRUB+R) at a published CIFAR learning rate *and* across a learning-rate sweep, and the proposed class-balanced Domain-Contrastive SSD (DC-SSD). Sweep variants appear as separate rows (`ssd_canonical@alpha=2`, `scrub_r@lr=0.005`) so that no baseline is represented only by hyperparameters transplanted from image classification.

The notebook treats retained-only retraining as the counterfactual gold standard. Successful unlearning means approaching that model while preserving retained-domain utility; it does **not** mean forcing accuracy on the forgotten domain to zero, because knowledge learned from other networks can legitimately generalize there. Every method starts independently from the exact same original checkpoint. Test data and the scratch model are evaluation-only, Fisher estimates use training rows only, and SCRUB+R uses the forgotten validation split only for the rewind operation prescribed by its paper.

The published baselines remain visibly separate from the proposed method. Canonical SSD uses the full pooled-training Fisher and the full forgotten-training Fisher with the published selection/dampening equation and no fine-tuning. SCRUB+R uses complete forget and retain passes, a frozen teacher, alternating max/min epochs, and same-domain validation rewind. DC-SSD first balances Fisher importance across benign/attack labels inside each domain and then contrasts the forgotten domain against an equal-weight retained-domain reference. Timing, sample exposure, storage, selected parameters, predictive similarity to retraining, and membership-inference results are saved for every run.


## 1. Install dependencies

**What the following block does:** This cell installs the libraries used by every later cell into the active Jupyter kernel. NumPy and pandas handle arrays and tables; SciPy and scikit-learn provide metrics, confidence intervals, and the learned privacy attack; PyTorch defines and trains both neural networks; PyArrow reads Parquet data by row batch; KaggleHub is an optional fallback when the configured files are not local; psutil samples process memory; and joblib stores preprocessing metadata. It does not load data or train anything. Run it once in a fresh environment, and restart the kernel only if Jupyter explicitly asks you to do so.

In [ ]:
%pip install -q "numpy>=1.26,<2" "pandas>=2,<3" "scipy>=1.11"     "scikit-learn>=1.4" "torch>=2.2,<3" "pyarrow>=15,<24"     "kagglehub>=0.3,<1" "psutil>=5.9,<8" "joblib>=1.3,<2"

## 2. Experiment configuration types

**What the following block does:** This cell defines validated dataclasses for datasets, preprocessing, models, original training, every unlearning method, membership attacks, and runtime behavior. It also declares the fixed 47 behavioural NetFlow features (the DNS transaction ID is excluded because it is an identifier, not behaviour) before any dataset is inspected, preventing a future forgotten domain from influencing feature selection. The unlearning configuration deliberately gives each algorithm its own parameters: canonical SSD defaults to the AAAI class-unlearning values \(\alpha=10,\lambda=1\); DC-SSD uses a separately named domain-contrastive threshold; SCRUB+R uses the nearest original-paper large-scale class-unlearning schedule; and rollback retains its earlier repair controls. The validator rejects unknown or duplicated methods and invalid schedules before a long experiment starts.


In [ ]:
"""Typed JSON configuration for the NetFlow unlearning experiment."""

from __future__ import annotations

import json
import sys
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

if sys.version_info < (3, 10):
    raise RuntimeError("This notebook requires Python 3.10 or newer")

# Publicly specified NF-v3 behavioural fields.  The list is fixed before any
# experiment is loaded, so a forgotten domain cannot influence feature
# selection.  Direct identifiers, ports, and absolute timestamps are excluded;
# FLOW_DURATION and BYTES_PER_PKT are deterministic derived fields.
# DNS_QUERY_ID is deliberately excluded: it is a 16-bit DNS transaction
# identifier chosen by the resolver, not a behavioural flow statistic, and a
# capture tool's ID generator can act as a dataset fingerprint.
PUBLIC_V3_MODEL_FEATURES = [
    "PROTOCOL",
    "L7_PROTO",
    "IN_BYTES",
    "OUT_BYTES",
    "IN_PKTS",
    "OUT_PKTS",
    "FLOW_DURATION",
    "TCP_FLAGS",
    "CLIENT_TCP_FLAGS",
    "SERVER_TCP_FLAGS",
    "DURATION_IN",
    "DURATION_OUT",
    "MIN_TTL",
    "MAX_TTL",
    "LONGEST_FLOW_PKT",
    "SHORTEST_FLOW_PKT",
    "MIN_IP_PKT_LEN",
    "MAX_IP_PKT_LEN",
    "SRC_TO_DST_SECOND_BYTES",
    "DST_TO_SRC_SECOND_BYTES",
    "RETRANSMITTED_IN_BYTES",
    "RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES",
    "RETRANSMITTED_OUT_PKTS",
    "SRC_TO_DST_AVG_THROUGHPUT",
    "DST_TO_SRC_AVG_THROUGHPUT",
    "NUM_PKTS_UP_TO_128_BYTES",
    "NUM_PKTS_128_TO_256_BYTES",
    "NUM_PKTS_256_TO_512_BYTES",
    "NUM_PKTS_512_TO_1024_BYTES",
    "NUM_PKTS_1024_TO_1514_BYTES",
    "TCP_WIN_MAX_IN",
    "TCP_WIN_MAX_OUT",
    "ICMP_TYPE",
    "ICMP_IPV4_TYPE",
    "DNS_QUERY_TYPE",
    "DNS_TTL_ANSWER",
    "FTP_COMMAND_RET_CODE",
    "SRC_TO_DST_IAT_MIN",
    "SRC_TO_DST_IAT_MAX",
    "SRC_TO_DST_IAT_AVG",
    "SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN",
    "DST_TO_SRC_IAT_MAX",
    "DST_TO_SRC_IAT_AVG",
    "DST_TO_SRC_IAT_STDDEV",
    "BYTES_PER_PKT",
]


@dataclass
class DatasetConfig:
    """One standardized NetFlow domain.

    ``path`` may name a CSV/Parquet file, a directory, or a glob.  When it does
    not resolve locally, ``kaggle_slug`` is used (if downloads are enabled).
    ``file_pattern`` is applied inside a directory or downloaded Kaggle folder;
    it is especially useful when one download contains multiple datasets.
    ``sample_rows`` is ``None`` to inherit ``DataConfig.sample_rows_per_dataset``,
    ``0`` to disable the cap for this dataset only, or a positive count.
    """

    name: str
    path: str | None = None
    kaggle_slug: str | None = None
    file_pattern: str = "**/*"
    label_column: str | None = None
    sample_rows: int | None = None
    encoding: str = "utf-8"
    allow_multiple_files: bool = False

    def validate(self) -> None:
        if not self.name.strip():
            raise ValueError("Every dataset needs a non-empty name")
        if not self.path and not self.kaggle_slug:
            raise ValueError(
                f"Dataset {self.name!r} needs either 'path' or 'kaggle_slug'"
            )
        if self.sample_rows is not None and self.sample_rows != 0 and self.sample_rows < 10:
            raise ValueError(
                f"{self.name}: sample_rows must be >= 10, 0 (no cap), or null (inherit)"
            )
        if not self.encoding.strip():
            raise ValueError(f"{self.name}: encoding must be non-empty")


@dataclass
class DataConfig:
    datasets: list[DatasetConfig]
    common_features: list[str] | None = None
    train_fraction: float = 0.70
    validation_fraction: float = 0.15
    test_fraction: float = 0.15
    seed: int = 42
    sample_rows_per_dataset: int | None = 250_000
    csv_chunk_rows: int = 100_000
    scaler: str = "fixed_log"
    scaler_fit_rows: int | None = 200_000
    allow_kaggle_download: bool = True
    split_strategy: str = "group_stratified"
    group_columns: list[str] = field(
        default_factory=lambda: ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "PROTOCOL"]
    )
    drop_exact_duplicates: bool = True
    # Drop rows whose model-feature vector *and* label repeat within a domain.
    # NetFlow corpora contain many flows that differ only in endpoints; with
    # endpoint-based grouping those land on both sides of a split.
    drop_feature_duplicates: bool = False
    # Warn when train vs validation/test attack rate differ by more than this.
    split_rate_warning_threshold: float = 0.10
    hash_source_files: bool = True
    strict_protocol: bool = False

    def validate(self) -> None:
        if not 4 <= len(self.datasets) <= 5:
            raise ValueError(
                "The requested protocol needs 4 or 5 datasets; "
                f"configuration contains {len(self.datasets)}"
            )
        names = [d.name for d in self.datasets]
        if len(set(names)) != len(names):
            raise ValueError(f"Dataset names must be unique: {names}")
        for dataset in self.datasets:
            dataset.validate()
        total = self.train_fraction + self.validation_fraction + self.test_fraction
        if abs(total - 1.0) > 1e-9:
            raise ValueError("train/validation/test fractions must add to 1")
        if min(self.train_fraction, self.validation_fraction, self.test_fraction) <= 0:
            raise ValueError("All split fractions must be positive")
        if self.csv_chunk_rows < 1_000:
            raise ValueError("csv_chunk_rows must be at least 1000")
        if not 0 < self.split_rate_warning_threshold <= 1:
            raise ValueError("split_rate_warning_threshold must be in (0, 1]")
        if (
            self.sample_rows_per_dataset is not None
            and self.sample_rows_per_dataset < 10
        ):
            raise ValueError("sample_rows_per_dataset must be >= 10 or null")
        if self.common_features is not None:
            normalized = [feature.strip().upper() for feature in self.common_features]
            if not normalized or any(not feature for feature in normalized):
                raise ValueError("common_features must contain non-empty names")
            if len(set(normalized)) != len(normalized):
                raise ValueError("common_features must not contain duplicates")
        if self.scaler not in {
            "fixed_log",
            "quantile",
            "robust",
            "standard",
            "minmax",
            "none",
        }:
            raise ValueError(
                "scaler must be one of: fixed_log, quantile, robust, standard, "
                "minmax, none"
            )
        if self.split_strategy not in {"group_stratified", "row_stratified"}:
            raise ValueError(
                "split_strategy must be 'group_stratified' or 'row_stratified'"
            )
        normalized_groups = [column.strip().upper() for column in self.group_columns]
        if self.split_strategy == "group_stratified" and not normalized_groups:
            raise ValueError("group_stratified splitting needs group_columns")
        if self.strict_protocol:
            if self.common_features is None:
                raise ValueError(
                    "Strict protocol requires a predeclared common_features list"
                )
            if self.scaler != "fixed_log":
                raise ValueError(
                    "Strict protocol requires the stateless fixed_log transform"
                )
            if self.split_strategy != "group_stratified":
                raise ValueError("Strict protocol requires group_stratified splitting")


@dataclass
class ModelConfig:
    architectures: list[str] = field(default_factory=lambda: ["mlp", "numerical_feature_transformer"])
    latent_dim: int = 32
    mlp_hidden_dims: list[int] = field(default_factory=lambda: [128, 64])
    dropout: float = 0.10
    nft_d_token: int = 16
    nft_heads: int = 4
    nft_layers: int = 2
    nft_ffn_factor: int = 4

    def validate(self) -> None:
        allowed = {"mlp", "numerical_feature_transformer"}
        if not self.architectures or set(self.architectures) - allowed:
            raise ValueError(f"architectures must be a non-empty subset of {allowed}")
        if self.latent_dim < 2 or any(width < 2 for width in self.mlp_hidden_dims):
            raise ValueError("latent and hidden dimensions must be >= 2")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must be in [0, 1)")
        if self.nft_d_token < 1 or self.nft_heads < 1 or self.nft_ffn_factor < 1:
            raise ValueError(
                "Numerical-Feature Transformer token/head/FFN dimensions must be positive"
            )
        if self.nft_d_token % self.nft_heads:
            raise ValueError("nft_d_token must be divisible by nft_heads")
        if self.nft_layers < 1:
            raise ValueError("nft_layers must be >= 1")


@dataclass
class TrainingConfig:
    epochs: int = 20
    batch_size: int = 256
    learning_rate: float = 1e-3
    optimizer: str = "sgd"
    sgd_momentum: float = 0.0
    weight_decay: float = 0.0
    patience: int = 5
    min_delta: float = 1e-4
    fixed_class_weights: list[float] = field(default_factory=lambda: [1.0, 1.0])
    gradient_clip_norm: float = 5.0
    domain_sampling: str = "proportional"
    checkpoint_selection: str = "final"
    seeds: list[int] = field(default_factory=lambda: [42, 1337, 2026])
    strict_unlearning_protocol: bool = False

    def validate(self) -> None:
        if min(self.epochs, self.batch_size, self.patience) < 1:
            raise ValueError("epochs, batch_size, and patience must be positive")
        if self.learning_rate <= 0 or self.weight_decay < 0:
            raise ValueError("Invalid optimizer hyperparameters")
        if self.optimizer not in {"sgd", "adamw"}:
            raise ValueError("optimizer must be 'sgd' or 'adamw'")
        if not 0 <= self.sgd_momentum < 1:
            raise ValueError("sgd_momentum must be in [0, 1)")
        if self.min_delta < 0:
            raise ValueError("min_delta must be non-negative")
        if len(self.fixed_class_weights) != 2 or any(
            value <= 0 for value in self.fixed_class_weights
        ):
            raise ValueError("fixed_class_weights must contain two positive values")
        if self.gradient_clip_norm < 0:
            raise ValueError("gradient_clip_norm must be non-negative")
        if self.domain_sampling not in {"proportional", "balanced"}:
            raise ValueError("domain_sampling must be 'proportional' or 'balanced'")
        if self.checkpoint_selection not in {"final", "early_stopping"}:
            raise ValueError("checkpoint_selection must be 'final' or 'early_stopping'")
        if not self.seeds:
            raise ValueError("At least one seed is required")
        if len(set(self.seeds)) != len(self.seeds):
            raise ValueError("Training seeds must be unique")
        if self.strict_unlearning_protocol:
            if self.optimizer != "sgd":
                raise ValueError("Strict unlearning requires optimizer='sgd'")
            if self.sgd_momentum != 0 or self.weight_decay != 0:
                raise ValueError(
                    "Strict unlearning requires zero momentum and weight decay"
                )
            if self.domain_sampling != "proportional":
                raise ValueError(
                    "Strict unlearning requires proportional domain sampling"
                )
            if self.checkpoint_selection != "final":
                raise ValueError("Strict unlearning requires a fixed final checkpoint")


@dataclass
class UnlearningConfig:
    """Published baselines plus the explicitly proposed DC-SSD method."""

    methods: list[str] = field(
        default_factory=lambda: [
            "rollback_repair",
            "ssd_canonical",
            "dc_ssd_no_contrast",
            "dc_ssd_no_class_balance",
            "dc_ssd",
            "scrub_r",
        ]
    )

    # Earlier amnesiac baseline; retained because its failure is informative.
    rollback_scale: float = 1.0
    repair_epochs: int = 2
    repair_fraction: float = 0.25
    repair_learning_rate: float = 2e-3

    # Fisher estimator shared by canonical SSD and DC-SSD.
    fisher_batch_size: int = 64
    fisher_epsilon: float = 1e-12

    # Published CIFAR/ResNet setting, not a universal AAAI 2024 default. With a 25% deletion,
    # alpha=10 can select zero weights; the notebook reports that honestly.
    ssd_alpha: float = 10.0
    ssd_lambda: float = 1.0
    # Every value here becomes its own reported baseline row
    # (``ssd_canonical@alpha=2``), so the paper default is never the only
    # canonical SSD result. Values equal to ``ssd_alpha`` are not repeated.
    ssd_alpha_sweep: list[float] = field(default_factory=lambda: [1.0, 2.0, 3.0, 5.0])

    # Proposed class-balanced, equal-domain DC-SSD starting point.
    dc_ssd_alpha: float = 2.0
    dc_ssd_lambda: float = 1.0
    dc_ssd_reference: str = "mean"

    # SCRUB+R settings closest to the paper's ResNet/CIFAR class protocol.
    scrub_steps: int = 3
    scrub_max_steps: int = 2
    scrub_forget_batch_size: int = 32
    scrub_retain_batch_size: int = 128
    scrub_optimizer: str = "sgd"
    scrub_learning_rate: float = 5e-4
    # Extra learning rates reported as separate ``scrub_r@lr=...`` rows.  The
    # paper default is 20x below this notebook's training rate, so a sweep is
    # required to show whether SCRUB moves the weights at all.
    scrub_learning_rate_sweep: list[float] = field(
        default_factory=lambda: [5e-3, 1e-2]
    )
    scrub_momentum: float = 0.9
    scrub_weight_decay: float = 5e-4
    scrub_temperature: float = 4.0
    scrub_alpha: float = 0.001
    scrub_gamma: float = 0.99
    # Milestones are counted in SCRUB outer steps; a milestone >= scrub_steps
    # fires after the last update and is therefore rejected by validate().
    scrub_lr_milestones: list[int] = field(default_factory=lambda: [2])
    scrub_lr_decay_factor: float = 0.1

    def validate(self) -> None:
        allowed = {
            "rollback_repair",
            "ssd_canonical",
            "dc_ssd_no_contrast",
            "dc_ssd_no_class_balance",
            "dc_ssd",
            "scrub_r",
        }
        if not self.methods:
            raise ValueError("At least one unlearning method must be enabled")
        if len(set(self.methods)) != len(self.methods):
            raise ValueError("Unlearning methods must not contain duplicates")
        unknown = set(self.methods) - allowed
        if unknown:
            raise ValueError(f"Unknown unlearning methods: {sorted(unknown)}")
        if self.rollback_scale < 0:
            raise ValueError("rollback_scale must be non-negative")
        if self.repair_epochs < 0:
            raise ValueError("repair_epochs must be >= 0")
        if not 0 < self.repair_fraction <= 1:
            raise ValueError("repair_fraction must be in (0, 1]")
        if self.repair_learning_rate <= 0:
            raise ValueError("repair_learning_rate must be positive")
        if self.fisher_batch_size < 1 or self.fisher_epsilon <= 0:
            raise ValueError("Fisher batch size and epsilon must be positive")
        if min(
            self.ssd_alpha,
            self.ssd_lambda,
            self.dc_ssd_alpha,
            self.dc_ssd_lambda,
        ) <= 0:
            raise ValueError("SSD alpha/lambda parameters must be positive")
        if any(alpha <= 0 for alpha in self.ssd_alpha_sweep):
            raise ValueError("ssd_alpha_sweep values must be positive")
        if len(set(self.ssd_alpha_sweep)) != len(self.ssd_alpha_sweep):
            raise ValueError("ssd_alpha_sweep must not contain duplicates")
        if any(rate <= 0 for rate in self.scrub_learning_rate_sweep):
            raise ValueError("scrub_learning_rate_sweep values must be positive")
        if len(set(self.scrub_learning_rate_sweep)) != len(
            self.scrub_learning_rate_sweep
        ):
            raise ValueError("scrub_learning_rate_sweep must not contain duplicates")
        if self.dc_ssd_reference not in {"mean", "max"}:
            raise ValueError("dc_ssd_reference must be 'mean' or 'max'")
        if self.scrub_steps < 1:
            raise ValueError("scrub_steps must be positive")
        if not 0 <= self.scrub_max_steps <= self.scrub_steps:
            raise ValueError("scrub_max_steps must be between 0 and scrub_steps")
        if min(
            self.scrub_forget_batch_size,
            self.scrub_retain_batch_size,
        ) < 1:
            raise ValueError("SCRUB batch sizes must be positive")
        if self.scrub_optimizer not in {"sgd", "adam"}:
            raise ValueError("scrub_optimizer must be 'sgd' or 'adam'")
        if self.scrub_learning_rate <= 0 or self.scrub_weight_decay < 0:
            raise ValueError("Invalid SCRUB optimizer parameters")
        if not 0 <= self.scrub_momentum < 1:
            raise ValueError("scrub_momentum must be in [0, 1)")
        if self.scrub_temperature <= 0:
            raise ValueError("scrub_temperature must be positive")
        if min(self.scrub_alpha, self.scrub_gamma) < 0:
            raise ValueError("SCRUB loss weights must be non-negative")
        if self.scrub_alpha + self.scrub_gamma <= 0:
            raise ValueError("At least one SCRUB retain loss must be active")
        if (
            any(step < 1 for step in self.scrub_lr_milestones)
            or sorted(set(self.scrub_lr_milestones)) != self.scrub_lr_milestones
        ):
            raise ValueError("SCRUB LR milestones must be sorted unique positives")
        if any(step >= self.scrub_steps for step in self.scrub_lr_milestones):
            raise ValueError(
                "SCRUB LR milestones must be < scrub_steps; a milestone at or "
                "after the final step never affects an update"
            )
        if not 0 < self.scrub_lr_decay_factor <= 1:
            raise ValueError("scrub_lr_decay_factor must be in (0, 1]")

    def expanded_methods(self) -> list[tuple[str, str, dict[str, float]]]:
        """Return (label, base_method, overrides) including sweep variants."""

        expanded: list[tuple[str, str, dict[str, float]]] = []
        for method in self.methods:
            expanded.append((method, method, {}))
            if method == "ssd_canonical":
                for alpha in self.ssd_alpha_sweep:
                    if alpha != self.ssd_alpha:
                        expanded.append(
                            (f"ssd_canonical@alpha={alpha:g}", method, {"alpha": alpha})
                        )
            elif method == "scrub_r":
                for rate in self.scrub_learning_rate_sweep:
                    if rate != self.scrub_learning_rate:
                        expanded.append(
                            (f"scrub_r@lr={rate:g}", method, {"learning_rate": rate})
                        )
        labels = [label for label, _, _ in expanded]
        if len(set(labels)) != len(labels):
            raise ValueError(f"Expanded method labels collide: {labels}")
        return expanded

@dataclass
class AttackConfig:
    max_samples_per_class: int = 10_000
    fixed_fpr: float = 0.01
    bootstrap_repetitions: int = 1_000
    confidence_level: float = 0.95

    def validate(self) -> None:
        if self.max_samples_per_class < 10:
            raise ValueError("max_samples_per_class must be >= 10")
        if not 0 < self.fixed_fpr < 1:
            raise ValueError("fixed_fpr must be in (0, 1)")
        if self.bootstrap_repetitions < 0:
            raise ValueError("bootstrap_repetitions must be >= 0")
        if not 0 < self.confidence_level < 1:
            raise ValueError("confidence_level must be in (0, 1)")


@dataclass
class RuntimeConfig:
    output_dir: str = "artifacts/netflow_unlearning"
    device: str = "auto"
    num_workers: int = 0
    save_predictions: bool = False
    deterministic: bool = True

    def validate(self) -> None:
        if self.device not in {"auto", "cpu", "cuda", "mps"}:
            raise ValueError("device must be auto, cpu, cuda, or mps")
        if self.num_workers < 0:
            raise ValueError("num_workers must be >= 0")


@dataclass
class ExperimentConfig:
    data: DataConfig
    models: ModelConfig = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    unlearning: UnlearningConfig = field(default_factory=UnlearningConfig)
    attack: AttackConfig = field(default_factory=AttackConfig)
    runtime: RuntimeConfig = field(default_factory=RuntimeConfig)

    def validate(self) -> None:
        self.data.validate()
        self.models.validate()
        self.training.validate()
        self.unlearning.validate()
        self.attack.validate()
        self.runtime.validate()
        if self.data.strict_protocol != self.training.strict_unlearning_protocol:
            raise ValueError(
                "data.strict_protocol and training.strict_unlearning_protocol "
                "must agree"
            )
        if self.data.strict_protocol and len(self.training.seeds) < 3:
            raise ValueError(
                "Publication protocol requires at least three independent seeds"
            )

    @classmethod
    def from_dict(cls, raw: dict[str, Any]) -> ExperimentConfig:
        data_raw = dict(raw["data"])
        data_raw["datasets"] = [DatasetConfig(**d) for d in data_raw["datasets"]]
        config = cls(
            data=DataConfig(**data_raw),
            models=ModelConfig(**raw.get("models", {})),
            training=TrainingConfig(**raw.get("training", {})),
            unlearning=UnlearningConfig(**raw.get("unlearning", {})),
            attack=AttackConfig(**raw.get("attack", {})),
            runtime=RuntimeConfig(**raw.get("runtime", {})),
        )
        config.validate()
        return config

    @classmethod
    def from_json(cls, path: str | Path) -> ExperimentConfig:
        config_path = Path(path).expanduser().resolve()
        with config_path.open("r", encoding="utf-8") as handle:
            raw = json.load(handle)
        config = cls.from_dict(raw)

        # Dataset paths in a checked-in config are relative to that config.
        for dataset in config.data.datasets:
            if dataset.path and not Path(dataset.path).expanduser().is_absolute():
                dataset.path = str((config_path.parent / dataset.path).resolve())
        output = Path(config.runtime.output_dir).expanduser()
        if not output.is_absolute():
            config.runtime.output_dir = str((config_path.parent / output).resolve())
        return config

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)

    def to_json(self, path: str | Path) -> None:
        destination = Path(path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("w", encoding="utf-8") as handle:
            json.dump(self.to_dict(), handle, indent=2, sort_keys=True)


## 3. Data loading, labels, features, provenance, and splits

**What the following block does:** This cell implements the complete data pipeline. It resolves one explicit CSV or Parquet source per domain, refusing multiple matches unless the user declares that they are intentional shards; reads large files in chunks with an explicit encoding; takes a reproducible uniform row sample; records the source file and original row number; removes exact duplicate records; canonicalizes columns; and converts common label conventions to `0 = benign` and `1 = attack`. It deterministically derives duration and bytes-per-packet, requires every predeclared feature to be present and numeric, and applies the same data-independent formula, `tanh(sign(x) × log1p(abs(x)) / 10)`, to every value. Because this transform is not fitted, a forgotten domain cannot remain in scaler parameters.

The cell then forms each domain's 70/15/15 partitions with a deterministic stratified group assignment based on the bidirectional endpoint pair and protocol. All flows in one group stay in one partition, reducing repeated-connection leakage that a random row split would permit. This is endpoint-pair isolation, not host isolation: one host may still communicate with different peers across partitions. Sampling and split seeds are derived from the domain name, so removing another domain does not change retained records. The saved manifest includes file size, modification time, optional SHA-256, duplicate counts, class rates, sampled-record fingerprint, split indices, positions back into the fingerprinted sampled-record ordering, group IDs, source-record IDs, and labels. Automatic feature intersection and learned scalers remain available only for explicitly non-strict exploratory work.

In [ ]:
"""NetFlow loading, common-schema construction, and leakage-safe splitting."""

from __future__ import annotations

import glob
import hashlib
import json
import math
import re
import warnings
from collections.abc import Iterable, Iterator, Sequence
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd


SUPPORTED_SUFFIXES = {".csv", ".parquet", ".pq"}
LABEL_ALIASES = ("LABEL", "BINARY_LABEL", "TARGET", "CLASS")
BENIGN_TOKENS = {
    "0",
    "benign",
    "normal",
    "normal.",
    "background",
    "legitimate",
    "false",
    "no",
    "nonattack",
    "non_attack",
    "non-attack",
}

# Identifiers make domain recognition easy without describing flow behaviour.
# This is the same exclusion policy used by the existing notebooks, expanded
# with common spelling variants.
EXCLUDED_COLUMNS = {
    "ATTACK",
    "ATTACK_CAT",
    "ATTACK_CATEGORY",
    "DATASET",
    "DATE",
    "TIMESTAMP",
    "FLOW_ID",
    "ROW_ID",
    "ID",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "IPV6_SRC_ADDR",
    "IPV6_DST_ADDR",
    "SRC_IP",
    "DST_IP",
    "SOURCE_IP",
    "DESTINATION_IP",
    "L4_SRC_PORT",
    "L4_DST_PORT",
    "SRC_PORT",
    "DST_PORT",
    "SOURCE_PORT",
    "DESTINATION_PORT",
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "FLOW_DURATION_MILLISECONDS",
    "NF_SOURCE_FILE",
    "NF_SOURCE_ROW",
}


def canonical_name(value: object) -> str:
    """Return a stable, case-insensitive feature name."""

    name = re.sub(r"[^A-Z0-9]+", "_", str(value).strip().upper()).strip("_")
    if not name:
        raise ValueError(f"Empty column name after canonicalization: {value!r}")
    return name


def _domain_seed(base_seed: int, domain_name: str, purpose: str) -> int:
    """Derive a seed unaffected by dataset ordering or leave-one-out removal."""

    payload = f"{base_seed}\x1f{domain_name}\x1f{purpose}".encode()
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "little")


def _canonicalize_frame(frame: pd.DataFrame) -> pd.DataFrame:
    names = [canonical_name(column) for column in frame.columns]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        raise ValueError(f"Columns collide after canonicalization: {duplicates}")
    result = frame.copy()
    result.columns = names
    return result


def _files_below(root: Path, pattern: str) -> list[Path]:
    return sorted(
        {
            path.resolve()
            for path in root.glob(pattern)
            if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES
        }
    )


def resolve_dataset_files(
    spec: DatasetConfig, allow_kaggle_download: bool = True
) -> list[Path]:
    """Resolve local data first and optionally fall back to Kaggle.

    No download is attempted when at least one matching local file exists.
    """

    files: list[Path] = []
    if spec.path:
        expanded = str(Path(spec.path).expanduser())
        if any(character in expanded for character in "*?["):
            files = sorted(
                Path(match).resolve()
                for match in glob.glob(expanded, recursive=True)
                if Path(match).is_file()
                and Path(match).suffix.lower() in SUPPORTED_SUFFIXES
            )
        else:
            candidate = Path(expanded)
            if candidate.is_file() and candidate.suffix.lower() in SUPPORTED_SUFFIXES:
                files = [candidate.resolve()]
            elif candidate.is_dir():
                files = _files_below(candidate, spec.file_pattern)

    if not files and spec.kaggle_slug and allow_kaggle_download:
        try:
            import kagglehub  # type: ignore
        except ImportError as exc:
            raise RuntimeError(
                f"No local files found for {spec.name!r}. Install kagglehub or "
                "put the dataset at the configured local path."
            ) from exc
        downloaded = Path(kagglehub.dataset_download(spec.kaggle_slug))
        files = _files_below(downloaded, spec.file_pattern)

    if not files:
        local_hint = f" at {spec.path!r}" if spec.path else ""
        download_hint = (
            f" (Kaggle fallback {spec.kaggle_slug!r})" if spec.kaggle_slug else ""
        )
        raise FileNotFoundError(
            f"No CSV/Parquet files found for {spec.name!r}{local_hint}{download_hint}."
        )
    if len(files) > 1 and not spec.allow_multiple_files:
        preview = [str(path) for path in files[:8]]
        raise ValueError(
            f"{spec.name!r} matched {len(files)} files, which could silently "
            "concatenate mirrors or duplicate exports. Narrow file_pattern, or "
            "set allow_multiple_files=True only when these are intentional shards. "
            f"Matches: {preview}"
        )
    return files


def _iter_file_chunks(
    path: Path, csv_chunk_rows: int, encoding: str = "utf-8"
) -> Iterator[pd.DataFrame]:
    if path.suffix.lower() == ".csv":
        # Encoding is explicit.  Retrying a partially-consumed generator with a
        # second encoding can duplicate every chunk yielded before a late decode
        # error, so a silent fallback is deliberately forbidden.
        yield from pd.read_csv(
            path,
            chunksize=csv_chunk_rows,
            low_memory=False,
            on_bad_lines="error",
            encoding=encoding,
        )
        return

    # Prefer row-group iteration so a large Parquet file is not materialized at
    # once.  pandas remains a fallback for environments without pyarrow.
    try:
        import pyarrow.parquet as pq  # type: ignore

        parquet = pq.ParquetFile(path)
        for batch in parquet.iter_batches(batch_size=csv_chunk_rows):
            yield batch.to_pandas()
    except ImportError:
        yield pd.read_parquet(path)


def _uniform_sample_files(
    files: Sequence[Path],
    limit: int | None,
    csv_chunk_rows: int,
    seed: int,
    encoding: str = "utf-8",
) -> pd.DataFrame:
    """Read files with an exact bounded uniform sample.

    Every row receives an i.i.d. random priority and the smallest ``limit``
    priorities are retained.  Unlike taking the first N rows, this does not
    bias a time-ordered NetFlow file, and memory is O(limit + chunk size).
    """

    rng = np.random.default_rng(seed)
    priority_column = "__NF_UNLEARNING_SAMPLE_PRIORITY__"
    reservoir: pd.DataFrame | None = None
    observed = 0

    if limit is None:
        warnings.warn(
            "sample_rows=None materializes the complete corpus in memory; use a "
            "bounded, predeclared sample for ordinary experiments.",
            ResourceWarning,
            stacklevel=2,
        )

    for path in files:
        file_row_offset = 0
        for chunk in _iter_file_chunks(path, csv_chunk_rows, encoding=encoding):
            if chunk.empty:
                continue
            if priority_column in chunk.columns:
                raise ValueError(f"Reserved column exists in {path}: {priority_column}")
            reserved = {"NF_SOURCE_FILE", "NF_SOURCE_ROW"} & {
                canonical_name(column) for column in chunk.columns
            }
            if reserved:
                raise ValueError(
                    f"Reserved provenance columns exist in {path}: {reserved}"
                )
            chunk = chunk.copy()
            chunk["NF_SOURCE_FILE"] = str(path)
            chunk["NF_SOURCE_ROW"] = np.arange(
                file_row_offset, file_row_offset + len(chunk), dtype=np.int64
            )
            file_row_offset += len(chunk)
            observed += len(chunk)
            if limit is None:
                reservoir = (
                    chunk
                    if reservoir is None
                    else pd.concat([reservoir, chunk], ignore_index=True, sort=False)
                )
                continue

            priorities = rng.random(len(chunk))
            if len(chunk) > limit:
                keep = np.argpartition(priorities, limit - 1)[:limit]
                chunk = chunk.iloc[keep].copy()
                priorities = priorities[keep]
            chunk[priority_column] = priorities
            reservoir = (
                chunk
                if reservoir is None
                else pd.concat([reservoir, chunk], ignore_index=True, sort=False)
            )
            if len(reservoir) > 2 * limit:
                reservoir = reservoir.nsmallest(limit, priority_column).copy()

    if reservoir is None or reservoir.empty:
        raise ValueError(
            f"Dataset files contain no rows: {[str(path) for path in files]}"
        )
    if limit is not None:
        reservoir = reservoir.nsmallest(min(limit, len(reservoir)), priority_column)
        reservoir = reservoir.drop(columns=[priority_column])
    reservoir = reservoir.reset_index(drop=True)
    reservoir.attrs["rows_observed"] = observed
    return reservoir


def _find_label_column(frame: pd.DataFrame, requested: str | None) -> str:
    if requested:
        candidate = canonical_name(requested)
        if candidate not in frame.columns:
            raise ValueError(
                f"Configured label column {requested!r} is absent. "
                f"Columns include: {list(frame.columns)[:20]}"
            )
        return candidate
    for candidate in LABEL_ALIASES:
        if candidate in frame.columns:
            return candidate
    if "ATTACK" in frame.columns:
        return "ATTACK"
    raise ValueError(
        "No binary label found. Set label_column in the dataset configuration."
    )


def _binary_labels(
    series: pd.Series,
    dataset_name: str,
    require_zero_one_numeric: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    """Normalize common binary/named labels and return labels plus valid mask."""

    valid = ~series.isna()
    if not valid.any():
        raise ValueError(f"{dataset_name}: every label is missing")
    clean = series[valid]

    numeric = pd.to_numeric(clean, errors="coerce")
    if numeric.notna().all():
        values = np.sort(numeric.unique())
        if set(values.tolist()).issubset({0, 1}):
            encoded = numeric.astype(np.int64).to_numpy()
        elif require_zero_one_numeric:
            raise ValueError(
                f"{dataset_name}: strict protocol requires numeric labels encoded "
                f"as 0=benign and 1=attack; found {values[:10].tolist()}"
            )
        elif len(values) == 2:
            encoded = (numeric.to_numpy() == values[-1]).astype(np.int64)
        elif 0 in values:
            # Some corpora encode attack families as positive integers.
            encoded = (numeric.to_numpy() != 0).astype(np.int64)
        else:
            raise ValueError(
                f"{dataset_name}: numeric label has no benign zero and is not "
                f"binary (values start {values[:10].tolist()})"
            )
    else:
        if require_zero_one_numeric:
            examples = sorted(clean.astype(str).unique().tolist())[:10]
            raise ValueError(
                f"{dataset_name}: strict protocol requires numeric labels encoded "
                f"as 0=benign and 1=attack; found text labels {examples}"
            )
        normalized = clean.astype(str).str.strip().str.lower()
        benign_mask = normalized.isin(BENIGN_TOKENS)
        if not benign_mask.any():
            examples = sorted(normalized.unique().tolist())[:10]
            raise ValueError(
                f"{dataset_name}: cannot identify a benign label among {examples}"
            )
        encoded = (~benign_mask).astype(np.int64).to_numpy()

    labels = np.full(len(series), -1, dtype=np.int64)
    labels[np.flatnonzero(valid.to_numpy())] = encoded
    valid_mask = labels >= 0
    classes, counts = np.unique(labels[valid_mask], return_counts=True)
    if classes.tolist() != [0, 1]:
        raise ValueError(
            f"{dataset_name}: both benign (0) and attack (1) are required; "
            f"found {dict(zip(classes.tolist(), counts.tolist(), strict=True))}"
        )
    if counts.min() < 3:
        raise ValueError(
            f"{dataset_name}: each class needs at least 3 rows for train/val/test"
        )
    return labels[valid_mask], valid_mask


def _numeric_features(
    frame: pd.DataFrame,
    label_column: str,
    dataset_name: str,
    requested_features: Sequence[str] | None = None,
) -> pd.DataFrame:
    frame = frame.copy()

    # Normalize the duration name across NetFlow releases.  Prefer the direct
    # duration feature; otherwise derive it before timestamps are removed.
    if "FLOW_DURATION" not in frame.columns:
        if "FLOW_DURATION_MILLISECONDS" in frame.columns:
            frame["FLOW_DURATION"] = frame["FLOW_DURATION_MILLISECONDS"]
        elif {
            "FLOW_START_MILLISECONDS",
            "FLOW_END_MILLISECONDS",
        }.issubset(frame.columns):
            start = pd.to_numeric(frame["FLOW_START_MILLISECONDS"], errors="coerce")
            end = pd.to_numeric(frame["FLOW_END_MILLISECONDS"], errors="coerce")
            frame["FLOW_DURATION"] = (end - start).clip(lower=0)

    if {"IN_BYTES", "IN_PKTS"}.issubset(frame.columns):
        in_bytes = pd.to_numeric(frame["IN_BYTES"], errors="coerce")
        in_packets = pd.to_numeric(frame["IN_PKTS"], errors="coerce")
        frame["BYTES_PER_PKT"] = in_bytes / (in_packets + 1e-5)

    excluded = set(EXCLUDED_COLUMNS) | set(LABEL_ALIASES) | {label_column}
    if requested_features is not None:
        candidates = [canonical_name(column) for column in requested_features]
        missing = sorted(set(candidates) - set(frame.columns))
        if missing:
            raise ValueError(
                f"{dataset_name}: predeclared model features are missing: {missing}"
            )
        forbidden = sorted(set(candidates) & excluded)
        if forbidden:
            raise ValueError(
                f"{dataset_name}: requested features contain identifiers or labels: "
                f"{forbidden}"
            )
    else:
        candidates = [column for column in frame.columns if column not in excluded]
    numeric: dict[str, pd.Series] = {}
    dropped: list[str] = []
    for column in candidates:
        converted = pd.to_numeric(frame[column], errors="coerce")
        original_nonmissing = int(frame[column].notna().sum())
        convertible = int(converted.notna().sum())
        ratio = convertible / max(original_nonmissing, 1)
        if ratio >= 0.99:
            numeric[column] = converted
        elif requested_features is not None:
            raise ValueError(
                f"{dataset_name}: predeclared feature {column!r} is only "
                f"{ratio:.1%} numeric; fix the source data instead of selecting "
                "features from observed values"
            )
        else:
            dropped.append(column)

    if dropped:
        warnings.warn(
            f"{dataset_name}: dropping nonnumeric columns {dropped}",
            RuntimeWarning,
            stacklevel=2,
        )
    if not numeric:
        raise ValueError(f"{dataset_name}: no numeric NetFlow features remain")

    result = pd.DataFrame(numeric, index=frame.index)
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return result


@dataclass
class DomainSplit:
    name: str
    x_train: np.ndarray
    y_train: np.ndarray
    x_val: np.ndarray
    y_val: np.ndarray
    x_test: np.ndarray
    y_test: np.ndarray
    train_indices: np.ndarray
    val_indices: np.ndarray
    test_indices: np.ndarray
    train_record_ids: np.ndarray | None = None
    val_record_ids: np.ndarray | None = None
    test_record_ids: np.ndarray | None = None
    train_group_ids: np.ndarray | None = None
    val_group_ids: np.ndarray | None = None
    test_group_ids: np.ndarray | None = None

    @property
    def X_train(self) -> np.ndarray:  # compatibility with notebook notation
        return self.x_train

    @property
    def X_val(self) -> np.ndarray:
        return self.x_val

    @property
    def X_test(self) -> np.ndarray:
        return self.x_test


@dataclass
class PreparedData:
    domains: dict[str, DomainSplit]
    common_features: list[str]
    scaler: FeatureScaler
    source_files: dict[str, list[str]]
    rows_observed: dict[str, int]
    source_metadata: dict[str, list[dict[str, object]]]
    sample_fingerprints: dict[str, str]
    duplicates_removed: dict[str, int]
    feature_duplicates_removed: dict[str, int] = field(default_factory=dict)
    feature_overlap: dict[str, dict[str, float]] = field(default_factory=dict)
    split_rate_gaps: dict[str, dict[str, float]] = field(default_factory=dict)

    @property
    def input_dim(self) -> int:
        return len(self.common_features)


def _stratified_indices(
    labels: np.ndarray,
    train_fraction: float,
    validation_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    partitions: dict[str, list[np.ndarray]] = {"train": [], "val": [], "test": []}
    for label in (0, 1):
        indices = np.flatnonzero(labels == label)
        rng.shuffle(indices)
        count = len(indices)
        train_count = max(1, math.floor(count * train_fraction))
        val_count = max(1, math.floor(count * validation_fraction))
        if train_count + val_count >= count:
            # The earlier class-count validation guarantees at least three.
            train_count, val_count = count - 2, 1
        partitions["train"].append(indices[:train_count])
        partitions["val"].append(indices[train_count : train_count + val_count])
        partitions["test"].append(indices[train_count + val_count :])

    outputs: list[np.ndarray] = []
    for key in ("train", "val", "test"):
        joined = np.concatenate(partitions[key]).astype(np.int64, copy=False)
        rng.shuffle(joined)
        outputs.append(joined)
    return outputs[0], outputs[1], outputs[2]


def _make_group_ids(frame: pd.DataFrame, columns: Sequence[str]) -> np.ndarray:
    """Create stable record groups used to keep related flows in one split."""

    normalized = [canonical_name(column) for column in columns]
    missing = sorted(set(normalized) - set(frame.columns))
    if missing:
        raise ValueError(f"Configured group columns are missing: {missing}")
    pieces = [frame[column].fillna("<NA>").astype(str) for column in normalized]
    if normalized[:2] == ["IPV4_SRC_ADDR", "IPV4_DST_ADDR"]:
        # Treat the two directions of the same endpoint pair as one conversation
        # group; otherwise A->B could be in training while B->A is in test.
        source = pieces[0].to_numpy(dtype=str)
        destination = pieces[1].to_numpy(dtype=str)
        first = np.where(source <= destination, source, destination)
        second = np.where(source <= destination, destination, source)
        pieces[:2] = [
            pd.Series(first, index=frame.index),
            pd.Series(second, index=frame.index),
        ]
    group = pieces[0]
    for piece in pieces[1:]:
        group = group.str.cat(piece, sep="|")
    return group.to_numpy(dtype=str)


def _stratified_group_indices(
    labels: np.ndarray,
    groups: np.ndarray,
    train_fraction: float,
    validation_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Split whole groups while approximately preserving size and class rate."""

    labels = np.asarray(labels, dtype=np.int64)
    groups = np.asarray(groups)
    if len(labels) != len(groups):
        raise ValueError("labels and groups have different lengths")
    unique_groups, group_number = np.unique(groups, return_inverse=True)
    if len(unique_groups) < 3:
        raise ValueError("At least three distinct groups are required")

    fractions = np.array(
        [
            train_fraction,
            validation_fraction,
            1.0 - train_fraction - validation_fraction,
        ],
        dtype=np.float64,
    )
    group_class_counts = np.zeros((len(unique_groups), 2), dtype=np.int64)
    np.add.at(group_class_counts, (group_number, labels), 1)
    group_sizes = group_class_counts.sum(axis=1)
    class_totals = group_class_counts.sum(axis=0)
    target_class_counts = fractions[:, None] * class_totals[None, :]
    target_row_counts = fractions * len(labels)
    target_rate = float(labels.mean())
    best: tuple[float, tuple[np.ndarray, np.ndarray, np.ndarray]] | None = None

    # A greedy group assignment is linear in the number of records and avoids
    # the very high cost of repeatedly constructing many StratifiedGroupKFold
    # folds on million-row corpora. Random restarts vary equal-sized group order;
    # the best legal assignment is retained.
    rng = np.random.default_rng(seed)
    # One greedy pass is sufficient for large corpora (and keeps runtime
    # linear); a few random tie-order restarts improve small teaching datasets.
    restart_count = 8 if len(unique_groups) <= 1_000 else 1
    for _ in range(restart_count):
        jitter = rng.random(len(unique_groups))
        order = np.lexsort((jitter, -group_sizes))
        assigned_class_counts = np.zeros((3, 2), dtype=np.float64)
        assigned_row_counts = np.zeros(3, dtype=np.float64)
        group_assignment = np.full(len(unique_groups), -1, dtype=np.int8)
        for current_group in order:
            costs = []
            for split_number in range(3):
                proposed_classes = assigned_class_counts.copy()
                proposed_rows = assigned_row_counts.copy()
                proposed_classes[split_number] += group_class_counts[current_group]
                proposed_rows[split_number] += group_sizes[current_group]
                class_error = np.square(
                    (proposed_classes - target_class_counts)
                    / np.maximum(target_class_counts, 1.0)
                ).sum()
                row_error = np.square(
                    (proposed_rows - target_row_counts)
                    / np.maximum(target_row_counts, 1.0)
                ).sum()
                costs.append(float(class_error + 0.25 * row_error))
            minimum = min(costs)
            choices = np.flatnonzero(np.isclose(costs, minimum))
            selected_split = int(rng.choice(choices))
            group_assignment[current_group] = selected_split
            assigned_class_counts[selected_split] += group_class_counts[current_group]
            assigned_row_counts[selected_split] += group_sizes[current_group]

        candidate = tuple(
            np.flatnonzero(group_assignment[group_number] == split_number).astype(
                np.int64, copy=False
            )
            for split_number in range(3)
        )
        if any(
            len(part) == 0
            or set(np.unique(labels[part]).tolist()) != {0, 1}
            or np.bincount(labels[part], minlength=2).min() < 2
            for part in candidate
        ):
            continue
        sizes = np.array([len(part) / len(labels) for part in candidate])
        rates = np.array([labels[part].mean() for part in candidate])
        score = float(np.abs(sizes - fractions).sum())
        score += float(0.25 * np.abs(rates - target_rate).sum())
        if best is None or score < best[0]:
            best = (score, candidate)

    if best is None:
        raise ValueError(
            "Could not form grouped train/validation/test partitions containing "
            "at least two rows from each class. Choose stronger group columns or "
            "increase the sample size."
        )
    rng = np.random.default_rng(seed)
    outputs = []
    for part in best[1]:
        result = part.copy()
        rng.shuffle(result)
        outputs.append(result)
    return outputs[0], outputs[1], outputs[2]


class FeatureScaler:
    """Feature transform with a publication-safe stateless option.

    ``fixed_log`` applies ``tanh(sign(x) * log1p(abs(x)) / 10)`` using no fitted
    data.  Therefore the preprocessing state is identical whether or not any
    domain exists.  Learned scalers remain available only for explicitly
    non-strict exploratory runs.
    """

    def __init__(self, kind: str, seed: int = 42):
        self.kind = kind
        self.seed = seed
        self.transformer = None

    def fit(self, values: np.ndarray) -> FeatureScaler:
        if self.kind in {"fixed_log", "none"}:
            return self
        try:
            from sklearn.preprocessing import (
                MinMaxScaler,
                QuantileTransformer,
                RobustScaler,
                StandardScaler,
            )
        except ImportError as exc:
            raise RuntimeError("scikit-learn is required for feature scaling") from exc

        if self.kind == "quantile":
            self.transformer = QuantileTransformer(
                n_quantiles=min(1_000, len(values)),
                output_distribution="uniform",
                random_state=self.seed,
                subsample=min(200_000, len(values)),
            )
        elif self.kind == "robust":
            self.transformer = RobustScaler(quantile_range=(5.0, 95.0))
        elif self.kind == "standard":
            self.transformer = StandardScaler()
        elif self.kind == "minmax":
            self.transformer = MinMaxScaler(feature_range=(-1.0, 1.0))
        else:
            raise ValueError(f"Unknown scaler: {self.kind}")
        self.transformer.fit(values)
        return self

    def transform(self, values: np.ndarray) -> np.ndarray:
        if self.kind == "fixed_log":
            finite = np.nan_to_num(
                np.asarray(values, dtype=np.float64),
                nan=0.0,
                posinf=np.finfo(np.float64).max,
                neginf=-np.finfo(np.float64).max,
            )
            output = np.tanh(np.sign(finite) * np.log1p(np.abs(finite)) / 10.0)
        elif self.kind == "none":
            output = np.asarray(values)
        else:
            if self.transformer is None:
                raise RuntimeError("FeatureScaler must be fitted before transform")
            output = self.transformer.transform(values)
            if self.kind == "quantile":
                output = 2.0 * output - 1.0
            elif self.kind == "robust":
                output = np.clip(output / 4.0, -1.0, 1.0)
            elif self.kind == "minmax":
                output = np.clip(output, -1.0, 1.0)
        return np.nan_to_num(output, nan=0.0, posinf=0.0, neginf=0.0).astype(
            np.float32, copy=False
        )


def _fit_rows(arrays: Iterable[np.ndarray], limit: int | None, seed: int) -> np.ndarray:
    combined = np.concatenate(list(arrays), axis=0)
    if limit is None or len(combined) <= limit:
        return combined
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(combined), size=limit, replace=False)
    return combined[indices]


def _safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(8 * 1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def _source_file_metadata(
    files: Sequence[Path], include_sha256: bool
) -> list[dict[str, object]]:
    output: list[dict[str, object]] = []
    for path in files:
        stat = path.stat()
        row: dict[str, object] = {
            "path": str(path),
            "size_bytes": int(stat.st_size),
            "modified_time_ns": int(stat.st_mtime_ns),
        }
        if include_sha256:
            row["sha256"] = _sha256_file(path)
        output.append(row)
    return output


def _sample_fingerprint(record_ids: np.ndarray, labels: np.ndarray) -> str:
    digest = hashlib.sha256()
    for record_id, label in zip(record_ids, labels, strict=True):
        digest.update(str(record_id).encode("utf-8"))
        digest.update(b"\0")
        digest.update(str(int(label)).encode("ascii"))
        digest.update(b"\n")
    return digest.hexdigest()


def _row_keys(values: np.ndarray) -> np.ndarray:
    """Hashable per-row keys for exact feature-vector comparison."""

    contiguous = np.ascontiguousarray(values)
    return contiguous.view(np.dtype((np.void, contiguous.dtype.itemsize * contiguous.shape[1]))).ravel()


def _feature_overlap(
    values: np.ndarray, train_idx: np.ndarray, other_idx: np.ndarray
) -> float:
    """Fraction of ``other_idx`` rows whose feature vector also occurs in train."""

    if len(other_idx) == 0:
        return 0.0
    train_keys = set(_row_keys(values[train_idx]).tolist())
    other_keys = _row_keys(values[other_idx])
    return float(np.mean([key in train_keys for key in other_keys.tolist()]))


def prepare_data(
    config: DataConfig, output_dir: str | Path | None = None
) -> PreparedData:
    """Load all domains and produce reproducible, leakage-resistant splits."""

    config.validate()
    feature_frames: dict[str, pd.DataFrame] = {}
    labels_by_domain: dict[str, np.ndarray] = {}
    groups_by_domain: dict[str, np.ndarray] = {}
    record_ids_by_domain: dict[str, np.ndarray] = {}
    source_files: dict[str, list[str]] = {}
    rows_observed: dict[str, int] = {}
    source_metadata: dict[str, list[dict[str, object]]] = {}
    sample_fingerprints: dict[str, str] = {}
    duplicates_removed: dict[str, int] = {}
    feature_duplicates_removed: dict[str, int] = {}
    feature_overlap: dict[str, dict[str, float]] = {}
    split_rate_gaps: dict[str, dict[str, float]] = {}
    fingerprinted_sample_positions: dict[str, dict[str, np.ndarray]] = {}
    requested = (
        [canonical_name(name) for name in config.common_features]
        if config.common_features is not None
        else None
    )

    for spec in config.datasets:
        files = resolve_dataset_files(spec, config.allow_kaggle_download)
        limit = spec.sample_rows
        if limit is None:
            limit = config.sample_rows_per_dataset
        elif limit == 0:
            limit = None  # explicit per-dataset "no cap"
        raw = _uniform_sample_files(
            files,
            limit=limit,
            csv_chunk_rows=config.csv_chunk_rows,
            seed=_domain_seed(config.seed, spec.name, "row-sampling"),
            encoding=spec.encoding,
        )
        observed = int(raw.attrs.get("rows_observed", len(raw)))
        raw = _canonicalize_frame(raw)
        before_deduplication = len(raw)
        if config.drop_exact_duplicates:
            duplicate_columns = [
                column
                for column in raw.columns
                if column not in {"NF_SOURCE_FILE", "NF_SOURCE_ROW"}
            ]
            raw = raw.drop_duplicates(subset=duplicate_columns, keep="first")
            raw = raw.reset_index(drop=True)
        duplicates_removed[spec.name] = before_deduplication - len(raw)
        label_column = _find_label_column(raw, spec.label_column)
        labels, valid_mask = _binary_labels(
            raw[label_column],
            spec.name,
            require_zero_one_numeric=config.strict_protocol,
        )
        raw = raw.loc[valid_mask].reset_index(drop=True)
        if config.split_strategy == "group_stratified":
            groups = _make_group_ids(raw, config.group_columns)
        else:
            groups = np.array([f"row:{index}" for index in range(len(raw))])
        record_ids = (
            raw["NF_SOURCE_FILE"].astype(str) + "::" + raw["NF_SOURCE_ROW"].astype(str)
        ).to_numpy(dtype=str)
        features = _numeric_features(
            raw, label_column, spec.name, requested_features=requested
        )

        feature_frames[spec.name] = features
        labels_by_domain[spec.name] = labels
        groups_by_domain[spec.name] = groups
        record_ids_by_domain[spec.name] = record_ids
        source_files[spec.name] = [str(path) for path in files]
        rows_observed[spec.name] = observed
        source_metadata[spec.name] = _source_file_metadata(
            files, config.hash_source_files
        )
        sample_fingerprints[spec.name] = _sample_fingerprint(record_ids, labels)

    feature_sets = [set(frame.columns) for frame in feature_frames.values()]
    intersection = set.intersection(*feature_sets)
    if requested is not None:
        missing = {
            name: sorted(set(requested) - set(frame.columns))
            for name, frame in feature_frames.items()
        }
        missing = {name: columns for name, columns in missing.items() if columns}
        if missing:
            raise ValueError(f"Explicit common_features are missing: {missing}")
        common_features = requested
    else:
        # Preserve the first standardized dataset's feature ordering while
        # enforcing intersection membership across every domain.
        first = next(iter(feature_frames.values()))
        common_features = [column for column in first.columns if column in intersection]
    if len(common_features) < 2:
        by_domain = {
            name: list(frame.columns) for name, frame in feature_frames.items()
        }
        raise ValueError(
            "Fewer than two common numeric features remain across all datasets: "
            f"{by_domain}"
        )

    unscaled: dict[str, DomainSplit] = {}
    for spec in config.datasets:
        values = feature_frames[spec.name][common_features].to_numpy(
            dtype=np.float32, copy=True
        )
        labels = labels_by_domain[spec.name]
        groups = groups_by_domain[spec.name]
        record_ids = record_ids_by_domain[spec.name]
        # These positions index the exact record ordering hashed by
        # sample_fingerprint_sha256. They remain stable even if feature-level
        # duplicates are removed before the train/validation/test split.
        sample_positions = np.arange(len(values), dtype=np.int64)
        if config.drop_feature_duplicates:
            key_frame = pd.DataFrame(values)
            key_frame["__label__"] = labels
            keep = ~key_frame.duplicated(keep="first").to_numpy()
            feature_duplicates_removed[spec.name] = int((~keep).sum())
            values, labels, groups, record_ids, sample_positions = (
                values[keep],
                labels[keep],
                groups[keep],
                record_ids[keep],
                sample_positions[keep],
            )
            if np.bincount(labels, minlength=2).min() < 3:
                raise ValueError(
                    f"{spec.name}: fewer than 3 rows per class remain after "
                    "feature-level deduplication"
                )
        else:
            feature_duplicates_removed[spec.name] = 0
        if config.split_strategy == "group_stratified":
            train_idx, val_idx, test_idx = _stratified_group_indices(
                labels,
                groups,
                config.train_fraction,
                config.validation_fraction,
                seed=_domain_seed(config.seed, spec.name, "data-split"),
            )
        else:
            train_idx, val_idx, test_idx = _stratified_indices(
                labels,
                config.train_fraction,
                config.validation_fraction,
                seed=_domain_seed(config.seed, spec.name, "data-split"),
            )
        fingerprinted_sample_positions[spec.name] = {
            "train": sample_positions[train_idx],
            "validation": sample_positions[val_idx],
            "test": sample_positions[test_idx],
        }
        feature_overlap[spec.name] = {
            "validation_rows_with_feature_vector_in_train": _feature_overlap(
                values, train_idx, val_idx
            ),
            "test_rows_with_feature_vector_in_train": _feature_overlap(
                values, train_idx, test_idx
            ),
        }
        rates = {
            "train": float(labels[train_idx].mean()),
            "validation": float(labels[val_idx].mean()),
            "test": float(labels[test_idx].mean()),
        }
        split_rate_gaps[spec.name] = {
            "validation_minus_train": rates["validation"] - rates["train"],
            "test_minus_train": rates["test"] - rates["train"],
        }
        worst_gap = max(abs(gap) for gap in split_rate_gaps[spec.name].values())
        if worst_gap > config.split_rate_warning_threshold:
            warnings.warn(
                f"{spec.name}: attack rate differs by {worst_gap:.2f} between "
                f"train and validation/test ({rates}); a heavy-hitter endpoint "
                "group is probably dominating one split. Consider stronger "
                "group_columns or a different sample.",
                RuntimeWarning,
                stacklevel=2,
            )
        overlap_max = max(feature_overlap[spec.name].values())
        if overlap_max > 0.05 and not config.drop_feature_duplicates:
            warnings.warn(
                f"{spec.name}: {overlap_max:.1%} of held-out rows have a feature "
                "vector identical to a training row. Test utility is inflated and "
                "the membership attack is deflated; set drop_feature_duplicates=True "
                "or report this overlap alongside results.",
                RuntimeWarning,
                stacklevel=2,
            )
        unscaled[spec.name] = DomainSplit(
            name=spec.name,
            x_train=values[train_idx],
            y_train=labels[train_idx],
            x_val=values[val_idx],
            y_val=labels[val_idx],
            x_test=values[test_idx],
            y_test=labels[test_idx],
            train_indices=train_idx,
            val_indices=val_idx,
            test_indices=test_idx,
            train_record_ids=record_ids[train_idx],
            val_record_ids=record_ids[val_idx],
            test_record_ids=record_ids[test_idx],
            train_group_ids=groups[train_idx],
            val_group_ids=groups[val_idx],
            test_group_ids=groups[test_idx],
        )

    if config.scaler in {"fixed_log", "none"}:
        # A dummy array makes the no-op fit API explicit without ever combining
        # or inspecting domain values.
        scaler_rows = np.empty((0, len(common_features)), dtype=np.float32)
    else:
        scaler_rows = _fit_rows(
            (split.x_train for split in unscaled.values()),
            config.scaler_fit_rows,
            config.seed,
        )
    scaler = FeatureScaler(config.scaler, config.seed).fit(scaler_rows)
    domains: dict[str, DomainSplit] = {}
    for name, split in unscaled.items():
        domains[name] = DomainSplit(
            name=name,
            x_train=scaler.transform(split.x_train),
            y_train=split.y_train,
            x_val=scaler.transform(split.x_val),
            y_val=split.y_val,
            x_test=scaler.transform(split.x_test),
            y_test=split.y_test,
            train_indices=split.train_indices,
            val_indices=split.val_indices,
            test_indices=split.test_indices,
            train_record_ids=split.train_record_ids,
            val_record_ids=split.val_record_ids,
            test_record_ids=split.test_record_ids,
            train_group_ids=split.train_group_ids,
            val_group_ids=split.val_group_ids,
            test_group_ids=split.test_group_ids,
        )

    prepared = PreparedData(
        domains=domains,
        common_features=common_features,
        scaler=scaler,
        source_files=source_files,
        rows_observed=rows_observed,
        source_metadata=source_metadata,
        sample_fingerprints=sample_fingerprints,
        duplicates_removed=duplicates_removed,
        feature_duplicates_removed=feature_duplicates_removed,
        feature_overlap=feature_overlap,
        split_rate_gaps=split_rate_gaps,
    )

    if output_dir is not None:
        destination = Path(output_dir)
        destination.mkdir(parents=True, exist_ok=True)
        feature_payload = {
            "feature_count": len(common_features),
            "features": common_features,
            "domain_feature_counts_before_intersection": {
                name: len(frame.columns) for name, frame in feature_frames.items()
            },
            "dropped_noncommon_features": {
                name: [
                    column for column in frame.columns if column not in common_features
                ]
                for name, frame in feature_frames.items()
            },
            "scaler": config.scaler,
            "feature_schema_scope": (
                "predeclared" if requested is not None else "observed_intersection"
            ),
            "transform_scope": (
                "stateless_data_independent"
                if config.scaler in {"fixed_log", "none"}
                else "fitted_combined_training_partitions_exploratory_only"
            ),
            "claim_scope": (
                "full_configured_model_pipeline"
                if config.scaler in {"fixed_log", "none"} and requested is not None
                else "model_weights_only"
            ),
        }
        (destination / "common_features.json").write_text(
            json.dumps(feature_payload, indent=2), encoding="utf-8"
        )

        manifest: dict[str, object] = {
            "seed": config.seed,
            "fractions": {
                "train": config.train_fraction,
                "validation": config.validation_fraction,
                "test": config.test_fraction,
            },
            "split_strategy": config.split_strategy,
            "group_columns": config.group_columns,
            "drop_exact_duplicates": config.drop_exact_duplicates,
            "drop_feature_duplicates": config.drop_feature_duplicates,
            "split_rate_warning_threshold": config.split_rate_warning_threshold,
            "domains": {},
        }
        domain_manifest = manifest["domains"]
        assert isinstance(domain_manifest, dict)
        for name, split in domains.items():
            domain_manifest[name] = {
                "rows_observed_before_sampling": rows_observed[name],
                "duplicates_removed_after_sampling": duplicates_removed[name],
                "feature_duplicates_removed": feature_duplicates_removed[name],
                "held_out_feature_vector_overlap_with_train": feature_overlap[name],
                "attack_rate_gap_vs_train": split_rate_gaps[name],
                "rows_after_sampling": int(
                    len(split.y_train) + len(split.y_val) + len(split.y_test)
                ),
                "source_files": source_files[name],
                "source_file_metadata": source_metadata[name],
                "sample_fingerprint_sha256": sample_fingerprints[name],
                "sample_fingerprint_scope": (
                    "post-sampling/post-exact-dedup/post-label-filter, "
                    "before feature-level deduplication"
                ),
                "split_index_reference": "post-feature-deduplication domain array",
                "fingerprinted_sample_position_reference": (
                    "positions in the record ordering hashed by sample_fingerprint_sha256"
                ),
                "splits": {
                    "train": {
                        "rows": len(split.y_train),
                        "attack_rate": float(split.y_train.mean()),
                    },
                    "validation": {
                        "rows": len(split.y_val),
                        "attack_rate": float(split.y_val.mean()),
                    },
                    "test": {
                        "rows": len(split.y_test),
                        "attack_rate": float(split.y_test.mean()),
                    },
                },
            }
            np.savez_compressed(
                destination / f"{_safe_name(name)}_split_indices.npz",
                train=split.train_indices,
                validation=split.val_indices,
                test=split.test_indices,
                train_fingerprinted_sample_positions=fingerprinted_sample_positions[name]["train"],
                validation_fingerprinted_sample_positions=fingerprinted_sample_positions[name]["validation"],
                test_fingerprinted_sample_positions=fingerprinted_sample_positions[name]["test"],
                train_labels=split.y_train,
                validation_labels=split.y_val,
                test_labels=split.y_test,
                train_record_ids=split.train_record_ids,
                validation_record_ids=split.val_record_ids,
                test_record_ids=split.test_record_ids,
                train_group_ids=split.train_group_ids,
                validation_group_ids=split.val_group_ids,
                test_group_ids=split.test_group_ids,
            )
        (destination / "split_manifest.json").write_text(
            json.dumps(manifest, indent=2), encoding="utf-8"
        )
        try:
            import joblib

            # Saving the wrapper class directly is fragile when this code is
            # executed from a notebook because its module is ``__main__``.
            # The sklearn transformer itself is importable and portable.
            joblib.dump(
                {
                    "kind": scaler.kind,
                    "seed": scaler.seed,
                    "formula": (
                        "tanh(sign(x) * log1p(abs(x)) / 10)"
                        if scaler.kind == "fixed_log"
                        else None
                    ),
                    "transformer": scaler.transformer,
                },
                destination / "scaler.joblib",
            )
        except ImportError as exc:
            raise RuntimeError(
                "joblib (installed with scikit-learn) is required"
            ) from exc

    return prepared


## 4. MLP and numerical-feature Transformer

**What the following block does:** This cell defines two classifiers with the same input and output contract. `MLPClassifier` passes each row through linear layers, LayerNorm, GELU activations, and dropout. The second model gives every scalar feature its own learned token projection, applies Transformer self-attention across feature tokens, mean-pools the tokens, and classifies the result. This is an Numerical-Feature Transformer in which every scalar feature receives a learned token projection before self-attention. Both networks output two raw logits—one for benign and one for attack—and later training uses softmax-compatible cross-entropy. The helper functions construct a model and report its parameter and checkpoint sizes; this cell performs no optimization.

In [ ]:
"""MLP and FT-style numerical-feature Transformer classifiers."""

from __future__ import annotations

import torch
from torch import nn



class MLPClassifier(nn.Module):
    """Explicit MLP replacement for the notebook's kernel-one CNN baseline."""

    def __init__(
        self,
        input_dim: int,
        hidden_dims: list[int],
        latent_dim: int,
        dropout: float,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        previous = input_dim
        for width in hidden_dims:
            layers.extend(
                [
                    nn.Linear(previous, width),
                    nn.LayerNorm(width),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ]
            )
            previous = width
        layers.extend(
            [
                nn.Linear(previous, latent_dim),
                nn.LayerNorm(latent_dim),
                nn.GELU(),
            ]
        )
        self.encoder = nn.Sequential(*layers)
        self.classifier = nn.Linear(latent_dim, 2)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward_features(self, values: torch.Tensor) -> torch.Tensor:
        return self.encoder(values)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.forward_features(values))


class NumericalFeatureTokenizer(nn.Module):
    """Turn each scalar continuous feature into its own learned token."""

    def __init__(self, input_dim: int, d_token: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, d_token))
        self.bias = nn.Parameter(torch.empty(input_dim, d_token))
        nn.init.normal_(self.weight, std=0.02)
        nn.init.normal_(self.bias, std=0.02)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return values.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class NumericalFeatureTransformerClassifier(nn.Module):
    """FT-style numerical-feature Transformer for continuous/numeric-coded inputs.

    Every input is represented as a scalar here, including numeric-coded
    categories such as protocol. Each scalar receives a learned feature-specific
    projection, self-attention models cross-feature interactions, and mean
    pooling yields a fixed-size representation.
    """

    def __init__(
        self,
        input_dim: int,
        latent_dim: int,
        d_token: int,
        n_heads: int,
        n_layers: int,
        ffn_factor: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.tokenizer = NumericalFeatureTokenizer(input_dim, d_token)
        layer = nn.TransformerEncoderLayer(
            d_model=d_token,
            nhead=n_heads,
            dim_feedforward=d_token * ffn_factor,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            layer, num_layers=n_layers, enable_nested_tensor=False
        )
        self.final_token_norm = nn.LayerNorm(d_token)
        self.encoder_out = nn.Sequential(
            nn.Linear(d_token, latent_dim),
            nn.LayerNorm(latent_dim),
            nn.GELU(),
        )
        self.classifier = nn.Linear(latent_dim, 2)
        self.reset_parameters()

    def reset_parameters(self) -> None:
        # TransformerEncoderLayer has already initialized its parameters.  Use
        # the same stable initialization as the MLP for the added projections.
        for module in (self.encoder_out[0], self.classifier):
            assert isinstance(module, nn.Linear)
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward_features(self, values: torch.Tensor) -> torch.Tensor:
        tokens = self.transformer(self.tokenizer(values))
        pooled = self.final_token_norm(tokens).mean(dim=1)
        return self.encoder_out(pooled)

    def forward(self, values: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.forward_features(values))


def build_model(name: str, input_dim: int, config: ModelConfig) -> nn.Module:
    normalized = name.strip().lower()
    if normalized == "mlp":
        return MLPClassifier(
            input_dim=input_dim,
            hidden_dims=config.mlp_hidden_dims,
            latent_dim=config.latent_dim,
            dropout=config.dropout,
        )
    if normalized == "numerical_feature_transformer":
        return NumericalFeatureTransformerClassifier(
            input_dim=input_dim,
            latent_dim=config.latent_dim,
            d_token=config.nft_d_token,
            n_heads=config.nft_heads,
            n_layers=config.nft_layers,
            ffn_factor=config.nft_ffn_factor,
            dropout=config.dropout,
        )
    raise ValueError(
        f"Unknown architecture {name!r}; choose mlp or numerical_feature_transformer"
    )


def parameter_count(model: nn.Module, trainable_only: bool = False) -> int:
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if not trainable_only or parameter.requires_grad
    )


def state_nbytes(state: dict[str, torch.Tensor]) -> int:
    return sum(tensor.numel() * tensor.element_size() for tensor in state.values())


## 5. Pooled multi-domain training and update logging

**What the following block does:** This cell defines reproducible training. It creates domain-homogeneous mini-batches so every optimization step can be attributed to exactly one dataset. Row permutations, schedule priorities, and dropout seeds are derived independently for each domain and batch; filtering a forgotten domain therefore leaves retained batches in the same order with the same examples and dropout masks in the scratch counterfactual. Strict runs use ordinary SGD with zero momentum and zero weight decay plus fixed, predeclared cross-entropy weights. This avoids AdamW moment buffers, regularization terms, and data-derived class weights that cannot be cleanly assigned to one domain.

During original training, the code stores the literal parameter difference from every step in that batch's domain trace. Initialization plus all traces must reconstruct the final checkpoint, which is checked later. A fixed final epoch prevents a future forgotten domain from influencing checkpoint choice through early stopping. Validation is run once at the end in strict mode, and optimization time is recorded separately from validation time so the efficiency comparison is like-for-like. The cell also defines prediction and checkpoint-copying helpers; actual training starts in Section 12.

In [ ]:
"""Deterministic multi-domain training with per-domain update accounting."""

from __future__ import annotations

import hashlib
import math
import random
import time
from collections.abc import Iterator, Mapping, Sequence
from dataclasses import dataclass

import numpy as np
import torch
from torch import nn


TensorState = dict[str, torch.Tensor]
DomainDeltas = dict[str, TensorState]


def seed_everything(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except TypeError:  # older torch
            torch.use_deterministic_algorithms(True)
        if torch.backends.cudnn.is_available():
            torch.backends.cudnn.benchmark = False
            torch.backends.cudnn.deterministic = True


def resolve_device(requested: str) -> torch.device:
    if requested == "auto":
        if torch.cuda.is_available():
            return torch.device("cuda")
        mps = getattr(torch.backends, "mps", None)
        if mps is not None and mps.is_available():
            return torch.device("mps")
        return torch.device("cpu")
    device = torch.device(requested)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    if device.type == "mps":
        mps = getattr(torch.backends, "mps", None)
        if mps is None or not mps.is_available():
            raise RuntimeError("MPS was requested but is unavailable")
    return device


def synchronize(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


def cpu_state_dict(model: nn.Module) -> TensorState:
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def _cpu_deltas(deltas: DomainDeltas) -> DomainDeltas:
    return {
        domain: {name: tensor.detach().cpu().clone() for name, tensor in state.items()}
        for domain, state in deltas.items()
    }


def load_state_dict(model: nn.Module, state: Mapping[str, torch.Tensor]) -> None:
    model.load_state_dict({name: tensor.clone() for name, tensor in state.items()})


def fixed_class_weights(config: TrainingConfig, device: torch.device) -> torch.Tensor:
    """Return predeclared loss weights that no dataset can influence."""

    return torch.tensor(config.fixed_class_weights, dtype=torch.float32, device=device)


def make_optimizer(
    parameters: Iterator[nn.Parameter] | Sequence[nn.Parameter],
    config: TrainingConfig,
    learning_rate: float | None = None,
) -> torch.optim.Optimizer:
    """Construct the configured optimizer and keep strict SGD explicit."""

    lr = config.learning_rate if learning_rate is None else learning_rate
    if config.optimizer == "sgd":
        return torch.optim.SGD(
            parameters,
            lr=lr,
            momentum=config.sgd_momentum,
            weight_decay=config.weight_decay,
        )
    if config.optimizer == "adamw":
        return torch.optim.AdamW(parameters, lr=lr, weight_decay=config.weight_decay)
    raise ValueError(f"Unknown optimizer: {config.optimizer}")


def _stable_seed(*parts: object) -> int:
    payload = "\x1f".join(str(part) for part in parts).encode("utf-8")
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "little")


def _stratified_fraction_indices(
    labels: np.ndarray, fraction: float, rng: np.random.Generator
) -> np.ndarray:
    if fraction >= 1.0:
        indices = np.arange(len(labels), dtype=np.int64)
        rng.shuffle(indices)
        return indices
    parts: list[np.ndarray] = []
    for label in (0, 1):
        candidates = np.flatnonzero(labels == label)
        count = max(1, round(len(candidates) * fraction))
        parts.append(
            rng.choice(candidates, size=min(count, len(candidates)), replace=False)
        )
    indices = np.concatenate(parts).astype(np.int64, copy=False)
    rng.shuffle(indices)
    return indices


def iter_domain_batches(
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    batch_size: int,
    seed: int,
    sampling: str = "proportional",
    fraction: float = 1.0,
) -> Iterator[tuple[str, np.ndarray, np.ndarray, int]]:
    """Yield domain-homogeneous batches in a shuffled multi-domain schedule.

    Homogeneous batches are required for attributing each optimizer update to a
    domain.  ``proportional`` traverses every selected row once.  ``balanced``
    cycles smaller domains until each domain contributes the same batch count.
    """

    chunks: dict[str, list[np.ndarray]] = {}
    for name in domain_names:
        rng = np.random.default_rng(_stable_seed(seed, "rows", name, fraction))
        chosen = _stratified_fraction_indices(domains[name].y_train, fraction, rng)
        chunks[name] = [
            chosen[start : start + batch_size]
            for start in range(0, len(chosen), batch_size)
        ]

    schedule: list[tuple[str, int]] = []
    if sampling == "proportional":
        for name in domain_names:
            schedule.extend((name, batch) for batch in range(len(chunks[name])))
    elif sampling == "balanced":
        max_batches = max(len(chunks[name]) for name in domain_names)
        for name in domain_names:
            for batch in range(max_batches):
                schedule.append((name, batch % len(chunks[name])))
    else:
        raise ValueError(f"Unknown domain sampling mode: {sampling}")
    # Each batch receives a data-independent priority.  Filtering one domain
    # from a full schedule therefore leaves the retained batches in exactly the
    # same relative order as a leave-one-domain-out scratch run.
    schedule.sort(
        key=lambda item: (
            _stable_seed(seed, "schedule", item[0], item[1]),
            item[0],
            item[1],
        )
    )

    for name, batch_number in schedule:
        selected = chunks[name][batch_number]
        batch_seed = _stable_seed(seed, "batch", name, batch_number) % (2**63 - 1)
        yield (
            name,
            domains[name].x_train[selected],
            domains[name].y_train[selected],
            batch_seed,
        )


@dataclass
class TrainingResult:
    state_dict: TensorState
    domain_deltas: DomainDeltas | None
    history: list[dict[str, float]]
    elapsed_seconds: float
    optimization_seconds: float
    validation_seconds: float
    validation_evaluations: int
    optimizer_steps: int
    samples_seen: int
    best_epoch: int
    best_validation_loss: float
    checkpoint_bytes: int
    delta_bytes: int


@torch.inference_mode()
def validation_loss(
    model: nn.Module,
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    criterion: nn.Module,
    device: torch.device,
    batch_size: int,
) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    count = 0
    for name in domain_names:
        values = domains[name].x_val
        labels = domains[name].y_val
        for start in range(0, len(labels), batch_size):
            x = torch.from_numpy(values[start : start + batch_size]).to(device)
            y = torch.from_numpy(labels[start : start + batch_size]).long().to(device)
            logits = model(x)
            loss = criterion(logits, y)
            batch_count = len(y)
            total_loss += float(loss.item()) * batch_count
            correct += int((logits.argmax(dim=1) == y).sum().item())
            count += batch_count
    return total_loss / max(count, 1), correct / max(count, 1)


def train_model(
    model: nn.Module,
    domains: Mapping[str, DomainSplit],
    config: TrainingConfig,
    device: torch.device,
    seed: int,
    domain_names: Sequence[str] | None = None,
    track_domain_deltas: bool = False,
    deterministic: bool = True,
) -> TrainingResult:
    """Train on the union of selected domains and retain the selected checkpoint.

    When tracking is enabled, every pure-domain SGD parameter delta is added
    to that domain's cumulative buffer.  At any checkpoint the model parameters
    equal initialization plus the sum of these buffers (up to floating-point
    accumulation).  Subtracting one buffer is the amnesiac approximation used
    by :mod:`netflow_unlearning.unlearning`.
    """

    selected = list(domain_names or domains.keys())
    if not selected:
        raise ValueError("At least one training domain is required")
    seed_everything(seed, deterministic)
    model.to(device)
    optimizer = make_optimizer(model.parameters(), config)
    criterion = nn.CrossEntropyLoss(weight=fixed_class_weights(config, device))

    parameter_map = dict(model.named_parameters())
    deltas: DomainDeltas | None = None
    if track_domain_deltas:
        deltas = {
            domain: {
                name: torch.zeros_like(parameter, device=device)
                for name, parameter in parameter_map.items()
            }
            for domain in selected
        }

    best_state: TensorState | None = None
    best_deltas: DomainDeltas | None = None
    best_loss = math.inf
    best_epoch = 0
    stale_epochs = 0
    steps = 0
    samples = 0
    history: list[dict[str, float]] = []
    optimization_seconds = 0.0
    validation_seconds = 0.0
    validation_evaluations = 0

    synchronize(device)
    started = time.perf_counter()
    for epoch in range(1, config.epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_samples = 0
        epoch_optimization_started = time.perf_counter()
        for domain, x_np, y_np, batch_seed in iter_domain_batches(
            domains,
            selected,
            batch_size=config.batch_size,
            seed=seed + epoch * 1_000_003,
            sampling=config.domain_sampling,
        ):
            # A stable per-batch seed pairs dropout masks between the original
            # and retained-only counterfactual schedules.
            torch.manual_seed(batch_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(batch_seed)
            x = torch.from_numpy(x_np).to(device)
            y = torch.from_numpy(y_np).long().to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            if config.gradient_clip_norm > 0:
                nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)

            before: dict[str, torch.Tensor] | None = None
            if deltas is not None:
                before = {
                    name: parameter.detach().clone()
                    for name, parameter in parameter_map.items()
                }
            optimizer.step()
            if deltas is not None and before is not None:
                with torch.no_grad():
                    for name, parameter in parameter_map.items():
                        deltas[domain][name].add_(parameter.detach() - before[name])

            batch_count = len(y)
            epoch_loss += float(loss.item()) * batch_count
            epoch_samples += batch_count
            steps += 1
            samples += batch_count
        synchronize(device)
        optimization_seconds += time.perf_counter() - epoch_optimization_started

        should_validate = (
            config.checkpoint_selection == "early_stopping" or epoch == config.epochs
        )
        if should_validate:
            validation_started = time.perf_counter()
            val_loss, val_accuracy = validation_loss(
                model, domains, selected, criterion, device, config.batch_size
            )
            synchronize(device)
            validation_seconds += time.perf_counter() - validation_started
            validation_evaluations += 1
        else:
            val_loss, val_accuracy = math.nan, math.nan
        history.append(
            {
                "epoch": float(epoch),
                "training_loss": epoch_loss / max(epoch_samples, 1),
                "validation_loss": val_loss,
                "validation_accuracy": val_accuracy,
            }
        )
        if config.checkpoint_selection == "final":
            # Selecting the original epoch with a validation domain that is
            # later forgotten would itself retain that domain's influence.  A
            # fixed epoch budget avoids this non-parameter deletion channel.
            if epoch == config.epochs:
                best_loss = val_loss
                best_epoch = epoch
                best_state = cpu_state_dict(model)
                best_deltas = _cpu_deltas(deltas) if deltas is not None else None
        elif val_loss < best_loss - config.min_delta:
            best_loss = val_loss
            best_epoch = epoch
            best_state = cpu_state_dict(model)
            best_deltas = _cpu_deltas(deltas) if deltas is not None else None
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= config.patience:
                break

    synchronize(device)
    elapsed = time.perf_counter() - started
    if best_state is None:
        raise RuntimeError("Training completed without producing a checkpoint")
    load_state_dict(model, best_state)
    delta_bytes = 0
    if best_deltas is not None:
        delta_bytes = sum(state_nbytes(state) for state in best_deltas.values())
    return TrainingResult(
        state_dict=best_state,
        domain_deltas=best_deltas,
        history=history,
        elapsed_seconds=elapsed,
        optimization_seconds=optimization_seconds,
        validation_seconds=validation_seconds,
        validation_evaluations=validation_evaluations,
        optimizer_steps=steps,
        samples_seen=samples,
        best_epoch=best_epoch,
        best_validation_loss=best_loss,
        checkpoint_bytes=state_nbytes(best_state),
        delta_bytes=delta_bytes,
    )


@torch.inference_mode()
def predict_proba(
    model: nn.Module,
    values: np.ndarray,
    device: torch.device,
    batch_size: int = 1_024,
) -> np.ndarray:
    model.eval()
    model.to(device)
    outputs: list[np.ndarray] = []
    for start in range(0, len(values), batch_size):
        x = torch.from_numpy(values[start : start + batch_size]).to(device)
        probabilities = torch.softmax(model(x), dim=1)
        outputs.append(probabilities.detach().cpu().numpy())
    return np.concatenate(outputs, axis=0)


## 6. Published baselines and the proposed NetFlow method

**What the following block does:** This cell implements the post-hoc unlearning paths and two DC-SSD component ablations with one common result format. First, `rollback_repair` subtracts the forgotten domain's logged SGD updates and then performs the small retained-data repair used in the earlier notebook; it is retained as a negative baseline because large interleaved deletions exposed its path-dependence. Second, canonical SSD follows Foster et al. (AAAI 2024): at the untouched original model it computes unweighted cross-entropy Fisher importance on every pooled training row and every forgotten training row, selects parameters satisfying \(F_f>\alpha F_D\), and multiplies only those parameters by \(\min(\lambda F_D/F_f,1)\), with no optimizer or repair. Third, the proposed DC-SSD computes Fisher separately for benign and attack rows in each dataset, averages the two classes equally, compares the forgotten-domain tensor with an equal-domain retained mean, and applies the same SSD dampening equation; this reduces the risk that the very different attack rates of the four NetFlow datasets masquerade as domain specificity. Finally, SCRUB+R follows Kurmanji et al. (NeurIPS 2023): a frozen original teacher and identical student alternate full forgotten-set divergence maximization with full retained-set distillation plus cross-entropy minimization, save each epoch, and rewind to the checkpoint whose forgotten-training error is closest to the final forgotten-validation error. Every path records exact gradient exposures, unique raw-data access, timing, temporary storage, parameter change, and method-specific diagnostics.

The SSD equation and SCRUB objectives/rewind are paper-based, but the architectures, binary domain deletion, domain-homogeneous batches, hyperparameter sweeps, and shortened SCRUB learning-rate schedule are NetFlow adaptations, not exact reproductions of the image experiments. SSD uses the authors' batch-gradient-squared importance proxy, not per-example Fisher; batch size and composition are therefore part of the reported protocol. Sweep values are predeclared, untuned starting points. Do not choose them using test scores or the gold model.

Canonical SSD source: [Foster et al., AAAI 2024](https://ojs.aaai.org/index.php/AAAI/article/view/29092). SCRUB+R source: [Kurmanji et al., NeurIPS 2023](https://proceedings.nips.cc/paper_files/paper/2023/file/062d711fb777322e2152435459e6e9d9-Paper-Conference.pdf). DC-SSD is an explicitly labelled adaptation rather than a claim of certified or exact unlearning.


**Minimal DC-SSD ablation:** `dc_ssd_no_contrast` retains class-balanced Fisher but builds the reference from all domains (including the forgotten domain). `dc_ssd_no_class_balance` retains the retained-only domain contrast and the exact same class-conditional Fisher estimator and batches, but weights benign and attack tensors by their observed training fractions instead of 0.5/0.5. Keeping those class-pure batches fixed is important: mixing classes inside a batch would also change gradient cancellation, confounding the class-weighting ablation. Each bank is measured separately and visits every training row once; only the final class mixture weights differ. These comparisons isolate the proposed components at the configured settings without changing the dampening rule.


In [ ]:
"""Published and NetFlow-specific approximate domain-unlearning methods."""

from __future__ import annotations

import math
import time
import warnings
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn


FisherState = dict[str, torch.Tensor]


@dataclass
class FisherEstimate:
    """A diagonal Fisher estimate plus transparent data-use accounting."""

    values: FisherState
    elapsed_seconds: float
    batches: int
    samples: int

    @property
    def storage_bytes(self) -> int:
        return state_nbytes(self.values)


@dataclass
class MethodResult:
    """Common return type used by every approximate unlearning method."""

    method: str
    state_dict: TensorState
    elapsed_seconds: float
    optimization_seconds: float
    optimizer_steps: int
    gradient_sample_exposures: int
    unique_forget_samples: int
    unique_retain_samples: int
    selection_inference_samples: int = 0
    temporary_storage_bytes: int = 0
    history: list[dict[str, float]] = field(default_factory=list)
    diagnostics: dict[str, object] = field(default_factory=dict)


def _parameter_change_diagnostics(
    original: Mapping[str, torch.Tensor],
    changed: Mapping[str, torch.Tensor],
) -> dict[str, float]:
    squared = 0.0
    maximum = 0.0
    for name, old_value in original.items():
        difference = changed[name].detach().cpu().float() - old_value.detach().cpu().float()
        squared += float(torch.sum(difference * difference).item())
        if difference.numel():
            maximum = max(maximum, float(difference.abs().max().item()))
    return {
        "parameter_change_l2": float(math.sqrt(squared)),
        "parameter_change_max_abs": maximum,
    }


def rollback_state(
    original_state: Mapping[str, torch.Tensor],
    forget_delta: Mapping[str, torch.Tensor],
    scale: float = 1.0,
) -> TensorState:
    """Subtract cumulative SGD updates attributed to one domain."""

    rolled: TensorState = {}
    for name, value in original_state.items():
        result = value.detach().cpu().clone()
        if name in forget_delta:
            delta = forget_delta[name].detach().cpu().to(result.dtype)
            if delta.shape != result.shape:
                raise ValueError(
                    f"Delta shape mismatch for {name}: {delta.shape} vs {result.shape}"
                )
            result.sub_(delta, alpha=scale)
        rolled[name] = result
    extra = set(forget_delta) - set(original_state)
    if extra:
        raise ValueError(f"Forget delta contains unknown parameters: {sorted(extra)}")
    return rolled


def run_rollback_repair(
    model: nn.Module,
    original_state: Mapping[str, torch.Tensor],
    domain_deltas: DomainDeltas,
    forget_domain: str,
    domains: Mapping[str, DomainSplit],
    training_config: TrainingConfig,
    config: UnlearningConfig,
    device: torch.device,
    seed: int,
) -> MethodResult:
    """Run the earlier amnesiac rollback as a documented negative baseline."""

    if forget_domain not in domain_deltas:
        raise KeyError(f"No logged updates for forgotten domain {forget_domain!r}")
    retained = [name for name in domains if name != forget_domain]
    seed_everything(seed)
    synchronize(device)
    started = time.perf_counter()
    rolled = rollback_state(
        original_state,
        domain_deltas[forget_domain],
        scale=config.rollback_scale,
    )
    load_state_dict(model, rolled)
    model.to(device)

    optimizer_steps = 0
    samples_seen = 0
    optimization_seconds = 0.0
    if config.repair_epochs > 0:
        optimizer = make_optimizer(
            model.parameters(),
            training_config,
            learning_rate=config.repair_learning_rate,
        )
        criterion = nn.CrossEntropyLoss(
            weight=fixed_class_weights(training_config, device)
        )
        optimization_started = time.perf_counter()
        for epoch in range(1, config.repair_epochs + 1):
            model.train()
            for _, x_np, y_np, batch_seed in iter_domain_batches(
                domains,
                retained,
                batch_size=training_config.batch_size,
                seed=seed + epoch * 1_000_033,
                sampling=training_config.domain_sampling,
                fraction=config.repair_fraction,
            ):
                torch.manual_seed(batch_seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(batch_seed)
                x = torch.from_numpy(x_np).to(device)
                y = torch.from_numpy(y_np).long().to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(x), y)
                loss.backward()
                if training_config.gradient_clip_norm > 0:
                    nn.utils.clip_grad_norm_(
                        model.parameters(), training_config.gradient_clip_norm
                    )
                optimizer.step()
                optimizer_steps += 1
                samples_seen += len(y)
        synchronize(device)
        optimization_seconds = time.perf_counter() - optimization_started

    synchronize(device)
    elapsed = time.perf_counter() - started
    state = cpu_state_dict(model)
    repair_indices: dict[str, set[int]] = {name: set() for name in retained}
    for epoch in range(1, config.repair_epochs + 1):
        epoch_seed = seed + epoch * 1_000_033
        for name in retained:
            rng = np.random.default_rng(
                _stable_seed(epoch_seed, "rows", name, config.repair_fraction)
            )
            selected = _stratified_fraction_indices(
                domains[name].y_train, config.repair_fraction, rng
            )
            repair_indices[name].update(int(index) for index in selected)
    unique_retain = sum(len(indices) for indices in repair_indices.values())
    diagnostics: dict[str, object] = {
        "protocol": "amnesiac update subtraction plus custom retained repair",
        "rollback_scale": config.rollback_scale,
        "repair_epochs": config.repair_epochs,
        "repair_fraction": config.repair_fraction,
        **_parameter_change_diagnostics(original_state, state),
    }
    return MethodResult(
        method="rollback_repair",
        state_dict=state,
        elapsed_seconds=elapsed,
        optimization_seconds=optimization_seconds,
        optimizer_steps=optimizer_steps,
        gradient_sample_exposures=samples_seen,
        unique_forget_samples=0,
        unique_retain_samples=unique_retain if config.repair_epochs else 0,
        # A deployment must retain every domain's trace to be able to forget
        # any of them, so the whole trace is the honest storage figure.
        temporary_storage_bytes=sum(
            state_nbytes(state) for state in domain_deltas.values()
        ),
        diagnostics=diagnostics,
    )


def compute_diagonal_fisher(
    model: nn.Module,
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    batch_size: int,
    device: torch.device,
    class_label: int | None = None,
) -> FisherEstimate:
    """Use the public SSD batch-gradient-squared importance estimator.

    The estimator uses unweighted cross-entropy, squares the gradient of each
    batch-mean loss, and averages those squared gradients across batches.  Every
    selected training row is visited exactly once; validation and test rows are
    never used. Evaluation mode disables dropout during importance estimation.
    Domain-aligned batches are identical in the full and forgotten estimates.
    This is an importance proxy, NOT the mean of per-example squared gradients;
    changing batch size or composition changes the estimator.
    """

    if class_label not in {None, 0, 1}:
        raise ValueError("class_label must be None, 0, or 1")
    selected = list(domain_names)
    if not selected:
        raise ValueError("Fisher estimation requires at least one domain")

    model.to(device)
    previous_mode = model.training
    model.eval()
    importance = {
        name: torch.zeros_like(parameter, device=device)
        for name, parameter in model.named_parameters()
    }
    criterion = nn.CrossEntropyLoss()
    batches = 0
    samples = 0
    synchronize(device)
    started = time.perf_counter()

    for domain_name in selected:
        split = domains[domain_name]
        if class_label is None:
            indices = np.arange(len(split.y_train), dtype=np.int64)
        else:
            indices = np.flatnonzero(split.y_train == class_label).astype(
                np.int64, copy=False
            )
            if not len(indices):
                raise ValueError(
                    f"{domain_name!r} has no class-{class_label} training rows; "
                    "class-balanced DC-SSD cannot be computed faithfully"
                )

        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start : start + batch_size]
            x = torch.from_numpy(split.x_train[batch_indices]).to(device)
            y = torch.from_numpy(split.y_train[batch_indices]).long().to(device)
            model.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            with torch.no_grad():
                for name, parameter in model.named_parameters():
                    if parameter.grad is not None:
                        importance[name].add_(parameter.grad.detach().square())
            batches += 1
            samples += len(y)

    if batches == 0:
        raise RuntimeError("Fisher estimation produced no batches")
    with torch.no_grad():
        for tensor in importance.values():
            tensor.div_(float(batches))
            if not torch.isfinite(tensor).all() or bool((tensor < 0).any()):
                raise FloatingPointError("Fisher importance must be finite and nonnegative")
    synchronize(device)
    elapsed = time.perf_counter() - started
    model.zero_grad(set_to_none=True)
    if previous_mode:
        model.train()
    values = {name: tensor.detach().cpu().clone() for name, tensor in importance.items()}
    return FisherEstimate(values, elapsed, batches, samples)


def compute_domain_class_fisher_bank(
    model: nn.Module,
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    batch_size: int,
    device: torch.device,
    *,
    class_balanced: bool = True,
) -> tuple[dict[str, FisherEstimate], dict[str, object]]:
    """Cache a balanced or empirical-weighted class-conditional Fisher bank.

    Both variants use the same per-class rows, batches, squared-gradient
    estimator, and original model. Only the final class mixture weights
    change, isolating class weighting from within-batch cancellation.
    Every training row is visited once in either separately timed bank.
    """

    bank: dict[str, FisherEstimate] = {}
    class_sample_counts: dict[str, dict[str, int]] = {}
    class_mixture_weights: dict[str, dict[str, float]] = {}
    started = time.perf_counter()
    for domain_name in domain_names:
        by_class: dict[int, FisherEstimate] = {}
        for label in (0, 1):
            by_class[label] = compute_diagonal_fisher(
                model,
                domains,
                [domain_name],
                batch_size,
                device,
                class_label=label,
            )
        total_samples = by_class[0].samples + by_class[1].samples
        weights = (
            (0.5, 0.5)
            if class_balanced
            else (
                by_class[0].samples / total_samples,
                by_class[1].samples / total_samples,
            )
        )
        combined_values = {
            name: weights[0] * by_class[0].values[name].float()
            + weights[1] * by_class[1].values[name].float()
            for name in by_class[0].values
        }
        bank[domain_name] = FisherEstimate(
            values=combined_values,
            elapsed_seconds=(
                by_class[0].elapsed_seconds + by_class[1].elapsed_seconds
            ),
            batches=by_class[0].batches + by_class[1].batches,
            samples=by_class[0].samples + by_class[1].samples,
        )
        class_sample_counts[domain_name] = {
            "benign": by_class[0].samples,
            "attack": by_class[1].samples,
        }
        class_mixture_weights[domain_name] = {
            "benign": weights[0],
            "attack": weights[1],
        }
    elapsed = time.perf_counter() - started
    storage = sum(
        estimate.storage_bytes for estimate in bank.values()
    )
    samples = sum(
        estimate.samples for estimate in bank.values()
    )
    batches = sum(
        estimate.batches for estimate in bank.values()
    )
    details = {
        "elapsed_seconds": elapsed,
        "storage_bytes": storage,
        "samples_seen": samples,
        "batches": batches,
        "domains": len(bank),
        "classes_per_domain": 2,
        "class_sample_counts": class_sample_counts,
        "class_mixture_weights": class_mixture_weights,
        "class_weighting": "equal 0.5/0.5" if class_balanced else "empirical training fractions",
        "estimator": "class-conditional mean squared batch-mean CE gradient",
    }
    return bank, details


def compute_domain_fisher_bank(
    model: nn.Module,
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    batch_size: int,
    device: torch.device,
) -> tuple[dict[str, FisherEstimate], dict[str, object]]:
    """Cache empirical-weighted class-conditional tensors for the ablation."""

    return compute_domain_class_fisher_bank(
        model, domains, domain_names, batch_size, device, class_balanced=False
    )


def _mean_fisher(states: Sequence[Mapping[str, torch.Tensor]]) -> FisherState:
    if not states:
        raise ValueError("Cannot average an empty Fisher collection")
    names = list(states[0])
    if any(list(state) != names for state in states[1:]):
        raise ValueError("Fisher parameter names or order differ")
    return {
        name: torch.stack([state[name].float() for state in states], dim=0).mean(0)
        for name in names
    }


def _max_fisher(states: Sequence[Mapping[str, torch.Tensor]]) -> FisherState:
    if not states:
        raise ValueError("Cannot maximize an empty Fisher collection")
    names = list(states[0])
    if any(list(state) != names for state in states[1:]):
        raise ValueError("Fisher parameter names or order differ")
    return {
        name: torch.stack([state[name].float() for state in states], dim=0).amax(0)
        for name in names
    }


def _balanced_domain_fisher(
    by_class: Mapping[int, FisherEstimate],
) -> FisherState:
    if set(by_class) != {0, 1}:
        raise ValueError("DC-SSD requires Fisher estimates for both binary classes")
    return _mean_fisher([by_class[0].values, by_class[1].values])


def _apply_ssd_rule(
    model: nn.Module,
    reference_importance: Mapping[str, torch.Tensor],
    forget_importance: Mapping[str, torch.Tensor],
    alpha: float,
    dampening_lambda: float,
    epsilon: float,
) -> dict[str, object]:
    """Apply SSD selection and dampening without ever amplifying a weight."""

    selected_total = 0
    changed_total = 0
    parameter_total = 0
    selected_by_parameter: dict[str, int] = {}
    parameter_size_by_parameter: dict[str, int] = {}
    ratios_for_summary: list[torch.Tensor] = []
    factors_for_summary: list[torch.Tensor] = []
    reference_parts: list[torch.Tensor] = []
    forget_parts: list[torch.Tensor] = []

    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name not in reference_importance or name not in forget_importance:
                raise KeyError(f"Missing Fisher tensor for parameter {name!r}")
            reference = reference_importance[name].to(parameter.device).float()
            forget = forget_importance[name].to(parameter.device).float()
            if reference.shape != parameter.shape or forget.shape != parameter.shape:
                raise ValueError(f"Fisher shape mismatch for {name}")
            selected = forget > alpha * reference
            # Match the published SSD update exactly on selected coordinates.
            # Selection guarantees ``forget > 0``, so no stabilizing epsilon is
            # needed in the actual dampening ratio.  Unselected coordinates stay 1.
            factor = torch.ones_like(forget)
            factor[selected] = torch.clamp(
                dampening_lambda * reference[selected] / forget[selected],
                min=0.0,
                max=1.0,
            )
            before = parameter.detach().clone()
            # Boolean advanced indexing returns a copy in PyTorch, so explicit
            # assignment is required for the dampening to reach the parameter.
            parameter[selected] = (
                parameter[selected] * factor[selected].to(parameter.dtype)
            )
            changed = selected & (parameter != before)
            count = int(selected.sum().item())
            selected_by_parameter[name] = count
            parameter_size_by_parameter[name] = parameter.numel()
            selected_total += count
            changed_total += int(changed.sum().item())
            parameter_total += parameter.numel()
            finite_ratio = torch.where(
                reference > 0,
                forget / reference.clamp_min(torch.finfo(reference.dtype).tiny),
                torch.where(forget > 0, torch.full_like(forget, float("inf")), torch.zeros_like(forget)),
            )
            ratios_for_summary.append(finite_ratio.detach().cpu().reshape(-1))
            reference_parts.append(reference.detach().cpu().reshape(-1))
            forget_parts.append(forget.detach().cpu().reshape(-1))
            if count:
                factors_for_summary.append(factor[selected].detach().cpu().reshape(-1))

    ratios = torch.cat(ratios_for_summary)
    finite_ratios = ratios[torch.isfinite(ratios)]
    selected_factors = (
        torch.cat(factors_for_summary) if factors_for_summary else torch.empty(0)
    )
    reference_vector = torch.cat(reference_parts).double()
    forget_vector = torch.cat(forget_parts).double()
    fisher_cosine = float(
        torch.dot(reference_vector, forget_vector).item()
        / max(
            float(torch.linalg.vector_norm(reference_vector).item())
            * float(torch.linalg.vector_norm(forget_vector).item()),
            epsilon,
        )
    )
    if selected_total == 0:
        warnings.warn(
            f"SSD selected zero parameters at alpha={alpha}. The returned model "
            "is intentionally left unchanged; this is a result, not a hidden fallback.",
            RuntimeWarning,
            stacklevel=2,
        )
    return {
        "alpha": alpha,
        "lambda": dampening_lambda,
        "selected_parameter_count": selected_total,
        "selected_parameter_fraction": selected_total / max(parameter_total, 1),
        "effectively_changed_parameter_count": changed_total,
        "selection_status": (
            "no_parameters_selected" if selected_total == 0
            else "selected_but_no_weights_changed" if changed_total == 0
            else "weights_changed"
        ),
        "total_parameter_count": parameter_total,
        "selected_by_parameter": selected_by_parameter,
        "parameter_size_by_parameter": parameter_size_by_parameter,
        "reference_forget_fisher_cosine_similarity": fisher_cosine,
        "finite_ratio_max": (
            float(finite_ratios.max().item()) if finite_ratios.numel() else math.nan
        ),
        "finite_ratio_median": (
            float(finite_ratios.median().item()) if finite_ratios.numel() else math.nan
        ),
        "dampening_factor_min": (
            float(selected_factors.min().item()) if selected_factors.numel() else 1.0
        ),
        "dampening_factor_median": (
            float(selected_factors.median().item()) if selected_factors.numel() else 1.0
        ),
        "dampening_factor_max": (
            float(selected_factors.max().item()) if selected_factors.numel() else 1.0
        ),
    }


def ssd_threshold_preflight(
    domains: Mapping[str, DomainSplit], config: UnlearningConfig
) -> list[dict[str, object]]:
    """Flag unreachable thresholds for this domain-aligned batch estimator.

    Since F_D is an average of nonnegative squared batch gradients and contains
    the exact forgotten batches, F_f <= (B_D / B_f) F_D coordinatewise.
    This is a batch-count bound, not a sample-count bound. It does not imply
    that a threshold below the bound selects any parameters or forgets well.
    """

    if "ssd_canonical" not in config.methods:
        return []
    batch_counts = {
        name: math.ceil(len(split.y_train) / config.fisher_batch_size)
        for name, split in domains.items()
    }
    if any(count == 0 for count in batch_counts.values()):
        raise ValueError("SSD requires nonempty training partitions")
    total_batches = sum(batch_counts.values())
    rows = []
    for label, method, overrides in config.expanded_methods():
        if method != "ssd_canonical":
            continue
        alpha = float(overrides.get("alpha", config.ssd_alpha))
        for name, count in batch_counts.items():
            bound = total_batches / count
            rows.append({
                "method": label,
                "forgotten_domain": name,
                "alpha": alpha,
                "full_fisher_batches": total_batches,
                "forget_fisher_batches": count,
                "batch_normalized_ratio_upper_bound": bound,
                "structural_noop": alpha >= bound,
            })
    return rows


def run_canonical_ssd(
    model: nn.Module,
    original_state: Mapping[str, torch.Tensor],
    full_fisher: FisherEstimate,
    forget_domain: str,
    domains: Mapping[str, DomainSplit],
    config: UnlearningConfig,
    device: torch.device,
    alpha: float | None = None,
    method_label: str = "ssd_canonical",
) -> MethodResult:
    """Run canonical AAAI SSD using full D and full D_f train partitions.

    ``alpha`` overrides ``config.ssd_alpha`` for sweep variants; the published
    selection/dampening equation itself is unchanged.
    """

    alpha_used = config.ssd_alpha if alpha is None else float(alpha)

    load_state_dict(model, original_state)
    model.to(device)
    synchronize(device)
    started = time.perf_counter()
    forget_fisher = compute_diagonal_fisher(
        model,
        domains,
        [forget_domain],
        config.fisher_batch_size,
        device,
    )
    expected_batches = sum(
        math.ceil(len(split.y_train) / config.fisher_batch_size)
        for split in domains.values()
    )
    if full_fisher.batches != expected_batches:
        raise ValueError("Full Fisher cache does not match this domain/batch configuration")
    batch_ratio_bound = full_fisher.batches / forget_fisher.batches
    diagnostics = _apply_ssd_rule(
        model,
        full_fisher.values,
        forget_fisher.values,
        alpha_used,
        config.ssd_lambda,
        config.fisher_epsilon,
    )
    synchronize(device)
    elapsed = time.perf_counter() - started
    state = cpu_state_dict(model)
    forget_fraction = len(domains[forget_domain].y_train) / max(
        sum(len(split.y_train) for split in domains.values()), 1
    )
    diagnostics.update(
        {
            "protocol": "canonical SSD (Foster et al., AAAI 2024)",
            "alpha_is_configured_default": alpha_used == config.ssd_alpha,
            "alpha_matches_published_cifar_resnet_setting": alpha_used == 10.0,
            "fisher_scope": "100% pooled train cache plus 100% forgotten train online",
            "forget_training_fraction": forget_fraction,
            "batch_normalized_ratio_upper_bound": batch_ratio_bound,
            "full_fisher_batches": full_fisher.batches,
            "structural_noop": alpha_used >= batch_ratio_bound,
            "forget_fisher_batches": forget_fisher.batches,
            "forget_fisher_samples": forget_fisher.samples,
            **_parameter_change_diagnostics(original_state, state),
        }
    )
    return MethodResult(
        method=method_label,
        state_dict=state,
        elapsed_seconds=elapsed,
        optimization_seconds=0.0,
        optimizer_steps=0,
        gradient_sample_exposures=forget_fisher.samples,
        unique_forget_samples=forget_fisher.samples,
        unique_retain_samples=0,
        temporary_storage_bytes=forget_fisher.storage_bytes,
        diagnostics=diagnostics,
    )


def run_dc_ssd(
    model: nn.Module,
    original_state: Mapping[str, torch.Tensor],
    fisher_bank: Mapping[str, FisherEstimate],
    forget_domain: str,
    config: UnlearningConfig,
    device: torch.device,
    *,
    method_label: str = "dc_ssd",
    class_balanced: bool = True,
    contrastive: bool = True,
) -> MethodResult:
    """Run DC-SSD or a single-component ablation using a prepared Fisher bank."""

    retained = [name for name in fisher_bank if name != forget_domain]
    if not retained:
        raise ValueError("DC-SSD needs at least one retained domain")
    reference_domains = retained if contrastive else list(fisher_bank)
    no_contrast_bound = (
        float(len(reference_domains)) if config.dc_ssd_reference == "mean" else 1.0
    )
    load_state_dict(model, original_state)
    model.to(device)
    synchronize(device)
    started = time.perf_counter()
    forget_importance = fisher_bank[forget_domain].values
    reference_importances = [fisher_bank[name].values for name in reference_domains]
    if config.dc_ssd_reference == "mean":
        reference = _mean_fisher(reference_importances)
    else:
        reference = _max_fisher(reference_importances)
    diagnostics = _apply_ssd_rule(
        model,
        reference,
        forget_importance,
        config.dc_ssd_alpha,
        config.dc_ssd_lambda,
        config.fisher_epsilon,
    )
    synchronize(device)
    elapsed = time.perf_counter() - started
    state = cpu_state_dict(model)
    bank_storage = sum(estimate.storage_bytes for estimate in fisher_bank.values())
    if class_balanced and contrastive:
        protocol = "proposed class-balanced domain-contrastive SSD"
    elif class_balanced:
        protocol = "DC-SSD ablation: class-balanced Fisher without retained-only contrast"
    elif contrastive:
        protocol = "DC-SSD ablation: retained-domain contrast without class balancing"
    else:
        raise ValueError("At least one DC-SSD component must be enabled")
    diagnostics.update(
        {
            "protocol": protocol,
            "ablation_class_balanced": class_balanced,
            "ablation_contrastive": contrastive,
            "no_contrast_ratio_upper_bound": no_contrast_bound if not contrastive else None,
            "structural_noop": bool(
                not contrastive and config.dc_ssd_alpha >= no_contrast_bound
            ),
            "class_weighting": "equal 0.5/0.5" if class_balanced else "empirical training fractions within domain",
            "fisher_estimator": "class-conditional mean squared batch-mean CE gradient",
            "domain_weighting": "equal across reference domains",
            "reference_domain_scope": "retained only" if contrastive else "all domains including forgotten",
            "retained_reference": config.dc_ssd_reference,
            "online_raw_retain_rows": 0,
            "online_raw_forget_rows": 0,
            "cache_policy_note": (
                "The shared bank is kept only to benchmark independent deletion "
                "scenarios; a deployment must purge the forgotten domain entry."
            ),
            **_parameter_change_diagnostics(original_state, state),
        }
    )
    return MethodResult(
        method=method_label,
        state_dict=state,
        elapsed_seconds=elapsed,
        optimization_seconds=0.0,
        optimizer_steps=0,
        gradient_sample_exposures=0,
        unique_forget_samples=0,
        unique_retain_samples=0,
        temporary_storage_bytes=bank_storage,
        diagnostics=diagnostics,
    )

def _scrub_kl(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    temperature: float,
) -> torch.Tensor:
    """Teacher-to-student distillation divergence used by SCRUB."""

    return F.kl_div(
        F.log_softmax(student_logits / temperature, dim=1),
        F.softmax(teacher_logits / temperature, dim=1),
        reduction="batchmean",
    ) * (temperature**2)


@torch.inference_mode()
def _classification_error(
    model: nn.Module,
    values: np.ndarray,
    labels: np.ndarray,
    device: torch.device,
    batch_size: int,
) -> float:
    model.eval()
    model.to(device)
    incorrect = 0
    for start in range(0, len(labels), batch_size):
        x = torch.from_numpy(values[start : start + batch_size]).to(device)
        y = torch.from_numpy(labels[start : start + batch_size]).long().to(device)
        incorrect += int((model(x).argmax(1) != y).sum().item())
    return incorrect / max(len(labels), 1)


def _make_scrub_optimizer(
    parameters: object, config: UnlearningConfig, learning_rate: float | None = None
) -> torch.optim.Optimizer:
    lr = config.scrub_learning_rate if learning_rate is None else float(learning_rate)
    if config.scrub_optimizer == "sgd":
        return torch.optim.SGD(
            parameters,
            lr=lr,
            momentum=config.scrub_momentum,
            weight_decay=config.scrub_weight_decay,
        )
    return torch.optim.Adam(
        parameters,
        lr=lr,
        weight_decay=config.scrub_weight_decay,
    )


def run_scrub_r(
    student: nn.Module,
    teacher: nn.Module,
    original_state: Mapping[str, torch.Tensor],
    forget_domain: str,
    domains: Mapping[str, DomainSplit],
    training_config: TrainingConfig,
    config: UnlearningConfig,
    device: torch.device,
    seed: int,
    learning_rate: float | None = None,
    method_label: str = "scrub_r",
) -> MethodResult:
    """Run original SCRUB min/max optimization and its privacy rewind rule.

    ``learning_rate`` overrides ``config.scrub_learning_rate`` for sweep rows.
    """

    lr_used = config.scrub_learning_rate if learning_rate is None else float(learning_rate)

    retained = [name for name in domains if name != forget_domain]
    if not retained:
        raise ValueError("SCRUB needs at least one retained domain")
    seed_everything(seed)
    load_state_dict(student, original_state)
    load_state_dict(teacher, original_state)
    student.to(device)
    teacher.to(device)
    teacher.eval()
    for parameter in teacher.parameters():
        parameter.requires_grad_(False)

    optimizer = _make_scrub_optimizer(student.parameters(), config, lr_used)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=config.scrub_lr_milestones,
        gamma=config.scrub_lr_decay_factor,
    )
    criterion = nn.CrossEntropyLoss()
    checkpoints: list[TensorState] = []
    history: list[dict[str, float]] = []
    optimizer_steps = 0
    gradient_exposures = 0
    selection_exposures = 0
    optimization_seconds = 0.0

    synchronize(device)
    started = time.perf_counter()
    for outer_step in range(1, config.scrub_steps + 1):
        used_learning_rate = float(optimizer.param_groups[0]["lr"])
        max_kl_sum = 0.0
        max_samples = 0
        if outer_step <= config.scrub_max_steps:
            student.train()
            phase_started = time.perf_counter()
            for _, x_np, y_np, batch_seed in iter_domain_batches(
                domains,
                [forget_domain],
                batch_size=config.scrub_forget_batch_size,
                seed=seed + outer_step * 1_000_081,
                sampling="proportional",
                fraction=1.0,
            ):
                torch.manual_seed(batch_seed)
                if torch.cuda.is_available():
                    torch.cuda.manual_seed_all(batch_seed)
                x = torch.from_numpy(x_np).to(device)
                optimizer.zero_grad(set_to_none=True)
                with torch.no_grad():
                    teacher_logits = teacher(x)
                student_logits = student(x)
                divergence = _scrub_kl(
                    student_logits, teacher_logits, config.scrub_temperature
                )
                if not torch.isfinite(divergence):
                    raise FloatingPointError("Non-finite SCRUB forget divergence")
                (-divergence).backward()
                optimizer.step()
                optimizer_steps += 1
                gradient_exposures += len(x)
                max_kl_sum += float(divergence.detach().item()) * len(x)
                max_samples += len(x)
            synchronize(device)
            optimization_seconds += time.perf_counter() - phase_started

        student.train()
        min_kl_sum = 0.0
        min_ce_sum = 0.0
        min_samples = 0
        phase_started = time.perf_counter()
        for _, x_np, y_np, batch_seed in iter_domain_batches(
            domains,
            retained,
            batch_size=config.scrub_retain_batch_size,
            seed=seed + outer_step * 1_000_099,
            sampling="proportional",
            fraction=1.0,
        ):
            torch.manual_seed(batch_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(batch_seed)
            x = torch.from_numpy(x_np).to(device)
            y = torch.from_numpy(y_np).long().to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                teacher_logits = teacher(x)
            student_logits = student(x)
            divergence = _scrub_kl(
                student_logits, teacher_logits, config.scrub_temperature
            )
            classification = criterion(student_logits, y)
            objective = (
                config.scrub_alpha * divergence
                + config.scrub_gamma * classification
            )
            if not torch.isfinite(objective):
                raise FloatingPointError("Non-finite SCRUB retain objective")
            objective.backward()
            optimizer.step()
            optimizer_steps += 1
            gradient_exposures += len(y)
            min_kl_sum += float(divergence.detach().item()) * len(y)
            min_ce_sum += float(classification.detach().item()) * len(y)
            min_samples += len(y)
        synchronize(device)
        optimization_seconds += time.perf_counter() - phase_started
        scheduler.step()

        split = domains[forget_domain]
        forget_train_error = _classification_error(
            student,
            split.x_train,
            split.y_train,
            device,
            max(config.scrub_forget_batch_size, config.scrub_retain_batch_size) * 4,
        )
        selection_exposures += len(split.y_train)
        checkpoints.append(cpu_state_dict(student))
        history.append(
            {
                "step": float(outer_step),
                "learning_rate": used_learning_rate,
                "forget_max_kl": max_kl_sum / max(max_samples, 1),
                "retain_min_kl": min_kl_sum / max(min_samples, 1),
                "retain_cross_entropy": min_ce_sum / max(min_samples, 1),
                "forget_train_error": forget_train_error,
            }
        )

    split = domains[forget_domain]
    final_validation_error = _classification_error(
        student,
        split.x_val,
        split.y_val,
        device,
        max(config.scrub_forget_batch_size, config.scrub_retain_batch_size) * 4,
    )
    selection_exposures += len(split.y_val)
    chosen_index = int(
        np.argmin(
            [
                abs(row["forget_train_error"] - final_validation_error)
                for row in history
            ]
        )
    )
    chosen_step = chosen_index + 1
    load_state_dict(student, checkpoints[chosen_index])
    history[chosen_index]["selected_by_rewind"] = 1.0
    for index, row in enumerate(history):
        if index != chosen_index:
            row["selected_by_rewind"] = 0.0
        row["final_same_domain_validation_error"] = final_validation_error

    synchronize(device)
    elapsed = time.perf_counter() - started
    state = cpu_state_dict(student)
    for name, value in teacher.state_dict().items():
        if not torch.equal(value.detach().cpu(), original_state[name].detach().cpu()):
            raise RuntimeError(f"Frozen SCRUB teacher changed at {name}")
    if any(parameter.grad is not None for parameter in teacher.parameters()):
        raise RuntimeError("Frozen SCRUB teacher unexpectedly accumulated gradients")
    if any(not torch.isfinite(value).all() for value in state.values()):
        raise FloatingPointError("SCRUB produced non-finite model state")
    checkpoint_bytes = state_nbytes(state)
    diagnostics: dict[str, object] = {
        "protocol": "SCRUB+R (Kurmanji et al., NeurIPS 2023)",
        "teacher_frozen": True,
        "forget_max_steps": config.scrub_max_steps,
        "total_min_steps": config.scrub_steps,
        "chosen_rewind_step": chosen_step,
        "final_same_domain_validation_error": final_validation_error,
        "chosen_forget_train_error": history[chosen_index]["forget_train_error"],
        "temperature": config.scrub_temperature,
        "alpha_retain_kl": config.scrub_alpha,
        "gamma_retain_ce": config.scrub_gamma,
        "optimizer": config.scrub_optimizer,
        "learning_rate": lr_used,
        "learning_rate_is_configured_default": lr_used == config.scrub_learning_rate,
        "learning_rate_matches_published_cifar_setting": lr_used == 5e-4,
        "lr_milestones": config.scrub_lr_milestones,
        "schedule_scope": "NetFlow short-run adaptation; not exact image-experiment reproduction",
        "weight_decay": config.scrub_weight_decay,
        "momentum": config.scrub_momentum if config.scrub_optimizer == "sgd" else 0.0,
        "forget_batch_size": config.scrub_forget_batch_size,
        "retain_batch_size": config.scrub_retain_batch_size,
        "gradient_clipping": False,
        "rewind_reference_uses": "forgotten-domain validation split, never test",
        **_parameter_change_diagnostics(original_state, state),
    }
    return MethodResult(
        method=method_label,
        state_dict=state,
        elapsed_seconds=elapsed,
        optimization_seconds=optimization_seconds,
        optimizer_steps=optimizer_steps,
        gradient_sample_exposures=gradient_exposures,
        unique_forget_samples=len(split.y_train),
        unique_retain_samples=sum(len(domains[name].y_train) for name in retained),
        selection_inference_samples=selection_exposures,
        temporary_storage_bytes=(len(checkpoints) + 1) * checkpoint_bytes,
        history=history,
        diagnostics=diagnostics,
    )


## 7. Utility, counterfactual similarity, and privacy audits

**What the following block does:** This cell defines utility, counterfactual-similarity, representation-similarity, and privacy evaluation families. `classification_metrics` computes accuracy, balanced accuracy, attack precision/recall/F1, ROC AUC, average precision, log loss, and the confusion matrix. `prediction_similarity` compares outputs with the retrained reference, while linear CKA compares penultimate representations on the same test records. These comparisons matter because forgetting means approaching the no-domain counterfactual, not forcing forgotten-domain accuracy to chance.

For privacy, the cell selects forgotten-domain training records as putative members and disjoint test records as nonmembers, with identical benign/attack counts; the exact same sampled indices are reused for every model. The predeclared loss-score attack reports AUC, advantage, and TPR at a fixed FPR. For grouped data its confidence intervals resample entire observed conversation groups, not independent flows, and reweight each draw to the original matched class mix. Candidate-minus-scratch AUC intervals use identical cluster draws and weights for both models. The secondary logistic output attack uses group-disjoint fitting and evaluation halves with the same class/membership reweighting, so related flows cannot occur on both sides. If groups or valid bootstrap draws are insufficient, the affected intervals or learned-attack scores are missing and an explicit status explains why; grouped audits never silently fall back to an IID-row calculation. Calls without group IDs remain supported but are explicitly labelled IID-row audits. Confidence intervals are conditional on the sampled audit records and the chosen conversation-group definition; they do not cover shared-host dependence across different groups or uncertainty from new corpus sampling. These empirical attacks can reveal residual membership signal, but failure to detect signal is not a mathematical or differential-privacy guarantee.

The loss-score advantage and fixed-FPR TPR are summaries of the audit ROC curve, not performance of an externally calibrated deployment threshold. Train/test cohort differences can also create membership signal, so interpret attacks relative to the retained-only gold model. The cell additionally computes linear CKA on matched penultimate test representations; constant representations have undefined CKA, and high CKA alone is not proof of deletion. See [Kornblith et al., ICML 2019](https://proceedings.mlr.press/v97/kornblith19a.html).


In [ ]:
"""Utility metrics, gold-model similarity, and a same-domain MIA."""

from __future__ import annotations

import numpy as np

EPSILON = 1e-12


def classification_metrics(
    y_true: np.ndarray, probabilities: np.ndarray
) -> dict[str, float]:
    """Return binary intrusion-detection metrics with attack as positive class."""

    try:
        from sklearn.metrics import (
            accuracy_score,
            average_precision_score,
            balanced_accuracy_score,
            confusion_matrix,
            f1_score,
            log_loss,
            precision_score,
            recall_score,
            roc_auc_score,
        )
    except ImportError as exc:
        raise RuntimeError("scikit-learn is required for evaluation") from exc

    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.ndim != 2 or probabilities.shape[1] != 2:
        raise ValueError(
            f"Expected N x 2 probabilities, received {probabilities.shape}"
        )
    if len(y_true) != len(probabilities):
        raise ValueError("Labels and probabilities have different lengths")
    probabilities = np.clip(probabilities, EPSILON, 1.0)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    positive = np.clip(probabilities[:, 1], EPSILON, 1.0 - EPSILON)
    predictions = (positive >= 0.5).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, predictions)),
        "precision_attack": float(
            precision_score(y_true, predictions, pos_label=1, zero_division=0)
        ),
        "recall_attack": float(
            recall_score(y_true, predictions, pos_label=1, zero_division=0)
        ),
        "f1_attack": float(f1_score(y_true, predictions, pos_label=1, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, positive)),
        "average_precision": float(average_precision_score(y_true, positive)),
        "log_loss": float(log_loss(y_true, probabilities, labels=[0, 1])),
        "true_negative": float(tn),
        "false_positive": float(fp),
        "false_negative": float(fn),
        "true_positive": float(tp),
        "samples": float(len(y_true)),
        "attack_rate": float(np.mean(y_true)),
    }


def prediction_similarity(candidate: np.ndarray, gold: np.ndarray) -> dict[str, float]:
    """Measure functional closeness to a scratch leave-one-domain-out model."""

    candidate = np.clip(np.asarray(candidate, dtype=np.float64), EPSILON, 1.0)
    gold = np.clip(np.asarray(gold, dtype=np.float64), EPSILON, 1.0)
    if candidate.shape != gold.shape:
        raise ValueError(f"Prediction shapes differ: {candidate.shape} vs {gold.shape}")
    candidate /= candidate.sum(axis=1, keepdims=True)
    gold /= gold.sum(axis=1, keepdims=True)
    midpoint = 0.5 * (candidate + gold)
    js_per_row = 0.5 * np.sum(candidate * np.log(candidate / midpoint), axis=1)
    js_per_row += 0.5 * np.sum(gold * np.log(gold / midpoint), axis=1)
    return {
        "prediction_agreement": float(
            np.mean(candidate.argmax(axis=1) == gold.argmax(axis=1))
        ),
        "mean_abs_attack_probability_gap": float(
            np.mean(np.abs(candidate[:, 1] - gold[:, 1]))
        ),
        "root_mean_square_probability_gap": float(
            np.sqrt(np.mean((candidate[:, 1] - gold[:, 1]) ** 2))
        ),
        "mean_jensen_shannon_divergence": float(np.mean(js_per_row)),
    }



def linear_cka(candidate_features: np.ndarray, gold_features: np.ndarray) -> float:
    """Linear centered-kernel alignment between matched penultimate representations."""

    candidate = np.asarray(candidate_features, dtype=np.float64)
    gold = np.asarray(gold_features, dtype=np.float64)
    if candidate.ndim != 2 or gold.ndim != 2 or candidate.shape[0] != gold.shape[0]:
        raise ValueError("CKA expects two 2-D feature matrices with the same row count")
    if candidate.shape[0] < 2:
        return float("nan")
    if not np.isfinite(candidate).all() or not np.isfinite(gold).all():
        raise ValueError("CKA inputs must be finite")
    candidate = candidate - candidate.mean(axis=0, keepdims=True)
    gold = gold - gold.mean(axis=0, keepdims=True)
    # CKA is invariant to isotropic scaling. Normalize before products so
    # a small but nonconstant representation is not mistaken for collapse.
    candidate_scale = float(np.max(np.abs(candidate), initial=0.0))
    gold_scale = float(np.max(np.abs(gold), initial=0.0))
    if candidate_scale == 0.0 or gold_scale == 0.0:
        return float("nan")
    candidate = candidate / candidate_scale
    gold = gold / gold_scale
    cross = candidate.T @ gold
    candidate_self = candidate.T @ candidate
    gold_self = gold.T @ gold
    numerator = float(np.square(cross).sum())
    denominator = float(
        np.linalg.norm(candidate_self, ord="fro") * np.linalg.norm(gold_self, ord="fro")
    )
    if denominator <= 0.0:
        return float("nan")
    return float(np.clip(numerator / denominator, 0.0, 1.0))

def matched_membership_indices(
    member_labels: np.ndarray,
    nonmember_labels: np.ndarray,
    max_samples_per_class: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, dict[int, int]]:
    rng = np.random.default_rng(seed)
    member_parts = []
    nonmember_parts = []
    counts: dict[int, int] = {}
    for label in (0, 1):
        members = np.flatnonzero(member_labels == label)
        nonmembers = np.flatnonzero(nonmember_labels == label)
        count = min(len(members), len(nonmembers), max_samples_per_class)
        if count < 2:
            raise ValueError(
                f"Membership attack needs >=2 member/nonmember rows for class {label}"
            )
        member_parts.append(rng.choice(members, size=count, replace=False))
        nonmember_parts.append(rng.choice(nonmembers, size=count, replace=False))
        counts[label] = count
    member_indices = np.concatenate(member_parts)
    nonmember_indices = np.concatenate(nonmember_parts)
    rng.shuffle(member_indices)
    rng.shuffle(nonmember_indices)
    return member_indices, nonmember_indices, counts


# Backward-compatible private spelling used by early notebook versions.
_matched_membership_indices = matched_membership_indices


def _loss_attack_statistics(
    member_loss: np.ndarray,
    nonmember_loss: np.ndarray,
    fixed_fpr: float,
    member_weights: np.ndarray | None = None,
    nonmember_weights: np.ndarray | None = None,
) -> dict[str, float]:
    from sklearn.metrics import roc_auc_score, roc_curve

    membership = np.concatenate(
        [np.ones(len(member_loss), dtype=np.int64),
         np.zeros(len(nonmember_loss), dtype=np.int64)]
    )
    score = np.concatenate([-member_loss, -nonmember_loss])
    weights = None
    if member_weights is not None or nonmember_weights is not None:
        if member_weights is None or nonmember_weights is None:
            raise ValueError("Provide both membership weight arrays or neither")
        weights = np.concatenate([member_weights, nonmember_weights])
    fpr, tpr, _ = roc_curve(membership, score, sample_weight=weights)
    valid = tpr[fpr <= fixed_fpr]
    return {
        "mia_loss_auc": float(roc_auc_score(membership, score, sample_weight=weights)),
        "mia_advantage": float(np.max(tpr - fpr)),
        "mia_tpr_at_fixed_fpr": float(valid.max()) if len(valid) else 0.0,
        "member_nonmember_loss_gap": float(
            np.average(nonmember_loss, weights=nonmember_weights)
            - np.average(member_loss, weights=member_weights)
        ),
    }


def _selected_membership_groups(
    member_group_ids: np.ndarray | None,
    nonmember_group_ids: np.ndarray | None,
    member_labels: np.ndarray,
    nonmember_labels: np.ndarray,
    member_indices: np.ndarray,
    nonmember_indices: np.ndarray,
) -> tuple[np.ndarray | None, np.ndarray | None]:
    """Validate full-split group IDs, then select exactly the audited records."""

    if (member_group_ids is None) != (nonmember_group_ids is None):
        raise ValueError("Provide both full-split group ID arrays or neither")
    if member_group_ids is None:
        return None, None
    selected = []
    for groups, labels, indices in (
        (member_group_ids, member_labels, member_indices),
        (nonmember_group_ids, nonmember_labels, nonmember_indices),
    ):
        groups = np.asarray(groups)
        if groups.ndim != 1 or len(groups) != len(labels):
            raise ValueError("Group ID arrays must align with the full label arrays")
        if any(value is None or str(value) in {"nan", "NaN", "<NA>"} for value in groups):
            raise ValueError("Membership group IDs must not contain missing values")
        selected.append(groups[indices].astype(str))
    return selected[0], selected[1]


def _group_blocks(groups: np.ndarray) -> list[np.ndarray]:
    """Return row indices for each observed cluster without quadratic scanning."""

    _, inverse, counts = np.unique(groups, return_inverse=True, return_counts=True)
    order = np.argsort(inverse, kind="stable")
    return list(np.split(order, np.cumsum(counts)[:-1]))


def _matched_class_weights(labels: np.ndarray, class_priors: np.ndarray) -> np.ndarray:
    """Reweight a cluster draw/attack half to the original matched label mix.

    A cluster is never broken or subsampled: all its rows share its draw
    multiplicity. Post-stratification prevents changing class prevalence from
    masquerading as a membership signal. Both classes must be represented.
    """

    counts = np.bincount(labels, minlength=2)
    if np.any(counts == 0):
        raise ValueError("Both classes are required for class-matched audit weights")
    return (len(labels) * class_priors / counts)[labels]


def _membership_bootstrap_intervals(
    member_loss: np.ndarray,
    nonmember_loss: np.ndarray,
    member_labels: np.ndarray,
    nonmember_labels: np.ndarray,
    fixed_fpr: float,
    repetitions: int,
    confidence_level: float,
    seed: int,
    member_groups: np.ndarray | None,
    nonmember_groups: np.ndarray | None,
    reference_losses: tuple[np.ndarray, np.ndarray] | None = None,
) -> dict[str, object]:
    """Cluster bootstrap when IDs exist; explicitly IID otherwise.

    Conversation clusters are drawn independently within each membership
    partition, with replacement. Candidate and gold use the same draws and
    class weights. Missing class coverage is counted, not silently replaced
    by a row bootstrap. Intervals require at least 20 valid draws and at least
    90% of requested draws; the status and accepted/rejected counts are saved.
    """

    paired = reference_losses is not None
    prefix = "mia_gap_bootstrap" if paired else "mia_bootstrap"
    keys = (["mia_auc_gap_to_retrained"] if paired else [
        "mia_loss_auc", "mia_advantage", "mia_tpr_at_fixed_fpr",
        "member_nonmember_loss_gap",
    ])
    result: dict[str, object] = {
        f"{key}_ci_{bound}": float("nan")
        for key in keys for bound in ("lower", "upper")
    }
    grouped = member_groups is not None
    if grouped != (nonmember_groups is not None):
        raise ValueError("Provide both selected group arrays or neither")
    result.update({
        f"{prefix}_unit": "conversation_group" if grouped else "iid_row",
        f"{prefix}_status": "disabled" if repetitions <= 0 else "pending",
        f"{prefix}_valid_repetitions": 0,
        f"{prefix}_invalid_class_coverage_repetitions": 0,
    })
    if repetitions <= 0:
        return result

    class_priors = np.bincount(member_labels, minlength=2) / len(member_labels)
    member_classes = [np.flatnonzero(member_labels == label) for label in (0, 1)]
    nonmember_classes = [np.flatnonzero(nonmember_labels == label) for label in (0, 1)]
    if any(len(indices) < 2 for indices in member_classes + nonmember_classes):
        result[f"{prefix}_status"] = "unavailable_insufficient_rows_per_class"
        return result
    if grouped:
        member_blocks = _group_blocks(member_groups)
        nonmember_blocks = _group_blocks(nonmember_groups)
        result[f"{prefix}_member_groups"] = len(member_blocks)
        result[f"{prefix}_nonmember_groups"] = len(nonmember_blocks)
        per_class_group_counts = [
            len(np.unique(groups[labels == label]))
            for groups, labels in ((member_groups, member_labels),
                                   (nonmember_groups, nonmember_labels))
            for label in (0, 1)
        ]
        if min(per_class_group_counts) < 2:
            result[f"{prefix}_status"] = "unavailable_insufficient_groups_per_class"
            return result

    rng = np.random.default_rng(seed)
    bootstrap = {key: [] for key in keys}
    invalid = 0
    for _ in range(repetitions):
        if grouped:
            member_draw = np.concatenate([
                member_blocks[index]
                for index in rng.integers(len(member_blocks), size=len(member_blocks))
            ])
            nonmember_draw = np.concatenate([
                nonmember_blocks[index]
                for index in rng.integers(len(nonmember_blocks), size=len(nonmember_blocks))
            ])
        else:
            member_draw = np.concatenate([
                rng.choice(indices, len(indices), replace=True) for indices in member_classes
            ])
            nonmember_draw = np.concatenate([
                rng.choice(indices, len(indices), replace=True) for indices in nonmember_classes
            ])
        member_draw_labels = member_labels[member_draw]
        nonmember_draw_labels = nonmember_labels[nonmember_draw]
        if min(np.bincount(member_draw_labels, minlength=2)) == 0 or min(
            np.bincount(nonmember_draw_labels, minlength=2)
        ) == 0:
            invalid += 1
            continue
        member_weights = _matched_class_weights(member_draw_labels, class_priors)
        nonmember_weights = _matched_class_weights(nonmember_draw_labels, class_priors)
        measured = _loss_attack_statistics(
            member_loss[member_draw], nonmember_loss[nonmember_draw], fixed_fpr,
            member_weights, nonmember_weights,
        )
        if paired:
            reference = _loss_attack_statistics(
                reference_losses[0][member_draw], reference_losses[1][nonmember_draw],
                fixed_fpr, member_weights, nonmember_weights,
            )
            measured = {"mia_auc_gap_to_retrained":
                        measured["mia_loss_auc"] - reference["mia_loss_auc"]}
        for key in keys:
            bootstrap[key].append(measured[key])
    valid_count = repetitions - invalid
    result[f"{prefix}_valid_repetitions"] = valid_count
    result[f"{prefix}_invalid_class_coverage_repetitions"] = invalid
    if valid_count < max(20, int(np.ceil(0.9 * repetitions))):
        result[f"{prefix}_status"] = "unavailable_insufficient_valid_repetitions"
        return result
    result[f"{prefix}_status"] = (
        "ok" if invalid == 0 else "conditional_on_both_classes_present"
    )
    tail = (1.0 - confidence_level) / 2.0
    for key, values in bootstrap.items():
        lower, upper = np.quantile(values, [tail, 1.0 - tail])
        result[f"{key}_ci_lower"] = float(lower)
        result[f"{key}_ci_upper"] = float(upper)
    return result


def _learned_output_attack(
    member_labels: np.ndarray,
    member_probabilities: np.ndarray,
    nonmember_labels: np.ndarray,
    nonmember_probabilities: np.ndarray,
    fixed_fpr: float,
    seed: int,
    member_groups: np.ndarray | None = None,
    nonmember_groups: np.ndarray | None = None,
) -> dict[str, object]:
    """Fit/evaluate the output attack on group-disjoint halves when IDs exist."""

    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score, roc_curve
    from sklearn.model_selection import GroupShuffleSplit, train_test_split

    labels = np.concatenate([member_labels, nonmember_labels]).astype(np.int64)
    probabilities = np.concatenate([member_probabilities, nonmember_probabilities])
    membership = np.concatenate([
        np.ones(len(member_labels), dtype=np.int64),
        np.zeros(len(nonmember_labels), dtype=np.int64),
    ])
    strata = 2 * membership + labels
    grouped = member_groups is not None
    if grouped != (nonmember_groups is not None):
        raise ValueError("Provide both selected group arrays or neither")
    result: dict[str, object] = {
        "mia_learned_auc": float("nan"),
        "mia_learned_advantage": float("nan"),
        "mia_learned_tpr_at_fixed_fpr": float("nan"),
        "mia_learned_attack_train_rows": 0,
        "mia_learned_attack_test_rows": 0,
        "mia_learned_split_unit": "conversation_group" if grouped else "iid_row",
        "mia_learned_status": "pending",
    }
    if grouped:
        # Membership prefixes avoid accidental collisions between unrelated
        # train/test group namespaces. Whole conversations stay in one half.
        groups = np.concatenate([
            np.char.add("member::", np.asarray(member_groups, dtype=str)),
            np.char.add("nonmember::", np.asarray(nonmember_groups, dtype=str)),
        ])
        if min(len(np.unique(groups[strata == s])) for s in range(4)) < 2:
            result["mia_learned_status"] = "unavailable_insufficient_groups_per_class"
            return result
        splitter = GroupShuffleSplit(n_splits=100, test_size=0.5, random_state=seed)
        legal_split = next((
            (train, test) for train, test in splitter.split(labels, groups=groups)
            if set(strata[train]) == {0, 1, 2, 3}
            and set(strata[test]) == {0, 1, 2, 3}
        ), None)
        if legal_split is None:
            result["mia_learned_status"] = "unavailable_no_group_disjoint_class_coverage"
            return result
        train_idx, test_idx = legal_split
        if set(groups[train_idx]) & set(groups[test_idx]):
            raise RuntimeError("Learned attack group split leaked a conversation")
        result["mia_learned_attack_train_groups"] = len(np.unique(groups[train_idx]))
        result["mia_learned_attack_test_groups"] = len(np.unique(groups[test_idx]))
        result["mia_learned_group_overlap"] = 0
    else:
        if min(np.bincount(strata, minlength=4)) < 2:
            result["mia_learned_status"] = "unavailable_insufficient_rows_per_class"
            return result
        train_idx, test_idx = train_test_split(
            np.arange(len(labels)), test_size=0.5, random_state=seed, stratify=strata,
        )

    true_confidence = probabilities[np.arange(len(labels)), labels]
    other_confidence = probabilities[np.arange(len(labels)), 1 - labels]
    loss = -np.log(np.clip(true_confidence, EPSILON, 1.0))
    entropy = -np.sum(probabilities * np.log(np.clip(probabilities, EPSILON, 1.0)), axis=1)
    margin = true_confidence - other_confidence
    features = np.column_stack([loss, true_confidence, entropy, margin])
    class_priors = np.bincount(member_labels, minlength=2) / len(member_labels)

    def audit_weights(indices: np.ndarray) -> np.ndarray:
        counts = np.bincount(strata[indices], minlength=4)
        target = np.tile(class_priors, 2) * 0.5
        return (len(indices) * target / counts)[strata[indices]]

    # Each half is reweighted to equal member/nonmember mass and the original
    # matched class mix; unequal cluster sizes cannot create a class-prior cue.
    attack = LogisticRegression(max_iter=1_000, random_state=seed)
    attack.fit(features[train_idx], membership[train_idx], sample_weight=audit_weights(train_idx))
    score = attack.predict_proba(features[test_idx])[:, 1]
    truth = membership[test_idx]
    weights = audit_weights(test_idx)
    fpr, tpr, _ = roc_curve(truth, score, sample_weight=weights)
    valid = tpr[fpr <= fixed_fpr]
    result.update({
        "mia_learned_auc": float(roc_auc_score(truth, score, sample_weight=weights)),
        "mia_learned_advantage": float(np.max(tpr - fpr)),
        "mia_learned_tpr_at_fixed_fpr": float(valid.max()) if len(valid) else 0.0,
        "mia_learned_attack_train_rows": len(train_idx),
        "mia_learned_attack_test_rows": len(test_idx),
        "mia_learned_status": "ok",
    })
    return result


def loss_membership_attack(
    member_labels: np.ndarray,
    member_probabilities: np.ndarray,
    nonmember_labels: np.ndarray,
    nonmember_probabilities: np.ndarray,
    max_samples_per_class: int,
    fixed_fpr: float,
    seed: int,
    bootstrap_repetitions: int = 1_000,
    confidence_level: float = 0.95,
    member_indices: np.ndarray | None = None,
    nonmember_indices: np.ndarray | None = None,
    member_group_ids: np.ndarray | None = None,
    nonmember_group_ids: np.ndarray | None = None,
) -> dict[str, object]:
    """Same-domain, class-matched loss MIA with explicit audit dependence.

    Full-split group arrays must align with the original label arrays. When
    supplied, intervals resample entire observed conversation clusters and the
    secondary learned attack holds out whole groups. Insufficient groups yield
    missing intervals/scores plus a status, never an implicit IID fallback.
    Without IDs the row-based audit remains available and is labelled IID.
    """

    if (member_indices is None) != (nonmember_indices is None):
        raise ValueError("Provide both matched index arrays or neither")
    if member_indices is None:
        member_idx, nonmember_idx, counts = matched_membership_indices(
            member_labels, nonmember_labels, max_samples_per_class, seed
        )
    else:
        member_idx = np.asarray(member_indices, dtype=np.int64)
        nonmember_idx = np.asarray(nonmember_indices, dtype=np.int64)
        if len(member_idx) != len(nonmember_idx):
            raise ValueError("Matched membership groups must have equal size")
        counts = {label: int(np.sum(member_labels[member_idx] == label)) for label in (0, 1)}
        if any(counts[label] != int(np.sum(nonmember_labels[nonmember_idx] == label))
               for label in (0, 1)):
            raise ValueError("Provided membership indices are not class matched")
    selected_member_labels = member_labels[member_idx]
    selected_nonmember_labels = nonmember_labels[nonmember_idx]
    member_groups, nonmember_groups = _selected_membership_groups(
        member_group_ids, nonmember_group_ids, member_labels, nonmember_labels,
        member_idx, nonmember_idx,
    )
    member_probs = np.clip(member_probabilities[member_idx], EPSILON, 1.0)
    nonmember_probs = np.clip(nonmember_probabilities[nonmember_idx], EPSILON, 1.0)
    member_loss = -np.log(member_probs[np.arange(len(member_idx)), selected_member_labels])
    nonmember_loss = -np.log(nonmember_probs[np.arange(len(nonmember_idx)), selected_nonmember_labels])
    result = {
        **_loss_attack_statistics(member_loss, nonmember_loss, fixed_fpr),
        "fixed_fpr": float(fixed_fpr),
        "member_mean_loss": float(member_loss.mean()),
        "nonmember_mean_loss": float(nonmember_loss.mean()),
        "samples_per_membership_group": float(len(member_loss)),
        "benign_samples_per_group": float(counts[0]),
        "attack_samples_per_group": float(counts[1]),
        "bootstrap_repetitions": int(bootstrap_repetitions),
        "confidence_level": float(confidence_level),
    }
    result.update(_membership_bootstrap_intervals(
        member_loss, nonmember_loss, selected_member_labels, selected_nonmember_labels,
        fixed_fpr, bootstrap_repetitions, confidence_level, seed + 97_409,
        member_groups, nonmember_groups,
    ))
    result.update(_learned_output_attack(
        selected_member_labels, member_probs, selected_nonmember_labels,
        nonmember_probs, fixed_fpr, seed + 271, member_groups, nonmember_groups,
    ))
    return result


def paired_loss_mia_auc_gap(
    member_labels: np.ndarray,
    candidate_member_probabilities: np.ndarray,
    reference_member_probabilities: np.ndarray,
    nonmember_labels: np.ndarray,
    candidate_nonmember_probabilities: np.ndarray,
    reference_nonmember_probabilities: np.ndarray,
    member_indices: np.ndarray,
    nonmember_indices: np.ndarray,
    fixed_fpr: float,
    bootstrap_repetitions: int,
    confidence_level: float,
    seed: int,
    member_group_ids: np.ndarray | None = None,
    nonmember_group_ids: np.ndarray | None = None,
) -> dict[str, object]:
    """Candidate-minus-gold AUC, sharing cluster draws and class weights."""

    member_indices = np.asarray(member_indices, dtype=np.int64)
    nonmember_indices = np.asarray(nonmember_indices, dtype=np.int64)
    selected_member_labels = member_labels[member_indices]
    selected_nonmember_labels = nonmember_labels[nonmember_indices]
    member_groups, nonmember_groups = _selected_membership_groups(
        member_group_ids, nonmember_group_ids, member_labels, nonmember_labels,
        member_indices, nonmember_indices,
    )

    def losses(probabilities: np.ndarray, indices: np.ndarray, labels: np.ndarray):
        selected = np.clip(probabilities[indices], EPSILON, 1.0)
        return -np.log(selected[np.arange(len(indices)), labels])

    candidate_member_loss = losses(candidate_member_probabilities, member_indices, selected_member_labels)
    reference_member_loss = losses(reference_member_probabilities, member_indices, selected_member_labels)
    candidate_nonmember_loss = losses(candidate_nonmember_probabilities, nonmember_indices, selected_nonmember_labels)
    reference_nonmember_loss = losses(reference_nonmember_probabilities, nonmember_indices, selected_nonmember_labels)
    candidate_auc = _loss_attack_statistics(candidate_member_loss, candidate_nonmember_loss, fixed_fpr)["mia_loss_auc"]
    reference_auc = _loss_attack_statistics(reference_member_loss, reference_nonmember_loss, fixed_fpr)["mia_loss_auc"]
    result = {"mia_auc_gap_to_retrained": candidate_auc - reference_auc}
    result.update(_membership_bootstrap_intervals(
        candidate_member_loss, candidate_nonmember_loss, selected_member_labels,
        selected_nonmember_labels, fixed_fpr, bootstrap_repetitions, confidence_level,
        seed, member_groups, nonmember_groups,
        reference_losses=(reference_member_loss, reference_nonmember_loss),
    ))
    return result


## 8. Resource monitoring

**What the following block does:** This cell defines a context manager that synchronizes the selected device, measures wall time, samples process resident memory, and records CUDA peak allocation when that counter is available. It reports absolute peaks and increments over memory held at entry. CUDA memory has status `measured`; MPS peaks are missing (`unavailable`); CPU accelerator memory is missing (`not_applicable`). Missing values are saved as blank CSV fields or JSON null, never as zero consumption. Process RSS is not isolated per-model memory and the sampler can miss short-lived peaks. The method functions also count optimizer steps, gradient exposures, unique examples, and temporary tensor storage. The optimization-scope speedup compares online method time with scratch optimization, excluding downstream evaluation; a separate monitor-scope ratio includes setup and transfers for both. Offline Fisher preparation and trace storage are reported separately, and the full original-training time must not be misrepresented as incremental tracing overhead. These are measured costs on the selected hardware, not FLOP counts or a universal speed guarantee.


In [ ]:
"""Wall-clock, CPU RSS, and accelerator peak-memory measurement."""

from __future__ import annotations

import os
import platform
import threading
import time
from dataclasses import dataclass

import torch



def _rss_bytes() -> int:
    try:
        import psutil

        return int(psutil.Process(os.getpid()).memory_info().rss)
    except ImportError:
        pass
    try:
        import resource  # POSIX only; absent on Windows
    except ImportError:
        return 0
    usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # macOS reports bytes; Linux and most BSDs report KiB.
    return int(usage if platform.system() == "Darwin" else usage * 1024)


@dataclass
class ResourceUsage:
    elapsed_seconds: float
    rss_start_bytes: int
    rss_peak_bytes: int
    rss_peak_increment_bytes: int
    accelerator_start_bytes: int | None
    accelerator_peak_bytes: int | None
    accelerator_peak_increment_bytes: int | None
    accelerator_memory_status: str


class ResourceMonitor:
    """Sample process RSS while an operation executes.

    The reported RSS increment is relative to memory already held at entry, so
    model/data allocations common to unlearning and retraining are not charged
    twice.  CUDA peak allocation is reset at entry.  MPS currently lacks an
    equivalent reliable peak counter: unavailable values are None, never zero.
    CPU accelerator memory is not applicable; CUDA values are measured.
    """

    def __init__(self, device: torch.device, sample_interval: float = 0.02) -> None:
        self.device = device
        self.sample_interval = sample_interval
        self.result: ResourceUsage | None = None
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self._start_rss = 0
        self._peak_rss = 0
        self._started = 0.0
        self._accelerator_start: int | None = None

    def _sample(self) -> None:
        while not self._stop.wait(self.sample_interval):
            self._peak_rss = max(self._peak_rss, _rss_bytes())

    def __enter__(self) -> ResourceMonitor:
        synchronize(self.device)
        self._accelerator_start = None
        if self.device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(self.device)
            self._accelerator_start = int(torch.cuda.memory_allocated(self.device))
        self._start_rss = _rss_bytes()
        self._peak_rss = self._start_rss
        self._stop.clear()
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()
        self._started = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback) -> None:
        synchronize(self.device)
        elapsed = time.perf_counter() - self._started
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=max(1.0, self.sample_interval * 5))
        self._peak_rss = max(self._peak_rss, _rss_bytes())
        accelerator: int | None = None
        memory_status = "not_applicable" if self.device.type == "cpu" else "unavailable"
        if self.device.type == "cuda":
            accelerator = int(torch.cuda.max_memory_allocated(self.device))
            memory_status = "measured"
        self.result = ResourceUsage(
            elapsed_seconds=elapsed,
            rss_start_bytes=self._start_rss,
            rss_peak_bytes=self._peak_rss,
            rss_peak_increment_bytes=max(0, self._peak_rss - self._start_rss),
            accelerator_start_bytes=self._accelerator_start,
            accelerator_peak_bytes=accelerator,
            accelerator_peak_increment_bytes=(
                max(0, accelerator - self._accelerator_start)
                if accelerator is not None and self._accelerator_start is not None
                else None
            ),
            accelerator_memory_status=memory_status,
        )


## 9. Full multi-method leave-one-domain-out experiment

**What the following block does:** This orchestration cell prepares the data once, then trains one original model for each architecture and seed. It computes the pooled Fisher cache once when canonical SSD is enabled and a separate class-balanced per-domain Fisher bank once when DC-SSD is enabled, while verifying that neither calculation changes the original checkpoint. For each forgotten dataset it runs every enabled method independently from that exact checkpoint, retrains one gold model from the exact original initialization using only the other domains, and evaluates all candidates on every test domain. Results are written in long format so algorithms are never averaged together: classification tables include per-domain and retained-macro utility, similarity tables compare predictions with retraining, membership tables compare class-matched forgotten training members with untouched forgotten test nonmembers, efficiency tables separate offline preparation from online deletion, and diagnostic files expose selected-weight fractions, Fisher overlap, rewind epochs, and parameter changes. The code also checks finite probabilities, immutable teachers/original states, exact delta reconstruction, method-specific checkpoints, and the expected row count and unique key of every final table.


In [ ]:
"""Comparative leave-one-NetFlow-domain-out unlearning experiment."""

from __future__ import annotations

import gc
import hashlib
import warnings
import json
import math
import platform
import re
import sys
from collections.abc import Mapping, Sequence
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch


PERFORMANCE_KEYS = (
    "accuracy",
    "balanced_accuracy",
    "precision_attack",
    "recall_attack",
    "f1_attack",
    "roc_auc",
    "average_precision",
    "log_loss",
)


def _safe_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("_")


def _scenario_seed(base_seed: int, domain: str, purpose: str) -> int:
    payload = f"{base_seed}\x1f{domain}\x1f{purpose}".encode()
    return int.from_bytes(hashlib.blake2b(payload, digest_size=4).digest(), "little")


def _json_dump(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def _write_rows(path: Path, rows: Sequence[Mapping[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(path, index=False)


def _aggregate_summary_across_seeds(
    rows: Sequence[Mapping[str, object]],
) -> list[dict[str, object]]:
    """Aggregate each method separately; never mix distinct algorithms."""

    frame = pd.DataFrame(rows)
    group_columns = ["architecture", "forgotten_domain", "method"]
    value_columns = [
        column
        for column in frame.select_dtypes(include=[np.number]).columns
        if column != "seed"
    ]
    output: list[dict[str, object]] = []
    for keys, group in frame.groupby(group_columns, sort=False):
        row: dict[str, object] = {
            "architecture": keys[0],
            "forgotten_domain": keys[1],
            "method": keys[2],
            "seed_count": int(group["seed"].nunique()),
        }
        for column in value_columns:
            values = group[column].astype(float).to_numpy()
            finite = values[np.isfinite(values)]
            row[f"{column}_valid_seed_count"] = int(len(finite))
            if not len(finite):
                row[f"{column}_mean"] = math.nan
                row[f"{column}_std"] = math.nan
                row[f"{column}_sem"] = math.nan
                row[f"{column}_ci95_lower"] = math.nan
                row[f"{column}_ci95_upper"] = math.nan
                continue
            mean = float(np.mean(finite))
            count = len(finite)
            if count >= 2:
                standard_deviation = float(np.std(finite, ddof=1))
                standard_error = standard_deviation / math.sqrt(count)
                from scipy.stats import t

                critical = float(t.ppf(0.975, df=count - 1))
            else:
                standard_deviation = math.nan
                standard_error = math.nan
                critical = math.nan
            row[f"{column}_mean"] = mean
            row[f"{column}_std"] = standard_deviation
            row[f"{column}_sem"] = standard_error
            row[f"{column}_ci95_lower"] = mean - critical * standard_error
            row[f"{column}_ci95_upper"] = mean + critical * standard_error
        output.append(row)
    return output


def _save_checkpoint(
    path: Path,
    state_dict: Mapping[str, torch.Tensor],
    metadata: Mapping[str, object],
    domain_deltas: DomainDeltas | None = None,
    initial_state: Mapping[str, torch.Tensor] | None = None,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload: dict[str, object] = {
        "state_dict": dict(state_dict),
        "metadata": dict(metadata),
    }
    if domain_deltas is not None:
        payload["domain_deltas"] = domain_deltas
    if initial_state is not None:
        payload["initial_state_dict"] = dict(initial_state)
    torch.save(payload, path)


def _resource_dict(prefix: str, usage: ResourceUsage | None) -> dict[str, object]:
    if usage is None:
        return {
            f"{prefix}_wall_seconds": math.nan,
            f"{prefix}_rss_start_bytes": math.nan,
            f"{prefix}_rss_peak_bytes": math.nan,
            f"{prefix}_rss_peak_increment_bytes": math.nan,
            f"{prefix}_accelerator_start_bytes": None,
            f"{prefix}_accelerator_peak_bytes": None,
            f"{prefix}_accelerator_peak_increment_bytes": None,
            f"{prefix}_accelerator_memory_status": "not_measured",
        }
    return {
        f"{prefix}_wall_seconds": usage.elapsed_seconds,
        f"{prefix}_rss_start_bytes": usage.rss_start_bytes,
        f"{prefix}_rss_peak_bytes": usage.rss_peak_bytes,
        f"{prefix}_rss_peak_increment_bytes": usage.rss_peak_increment_bytes,
        f"{prefix}_accelerator_start_bytes": usage.accelerator_start_bytes,
        f"{prefix}_accelerator_peak_bytes": usage.accelerator_peak_bytes,
        f"{prefix}_accelerator_peak_increment_bytes": (
            usage.accelerator_peak_increment_bytes
        ),
        f"{prefix}_accelerator_memory_status": usage.accelerator_memory_status,
    }


def _delta_reconstruction_error(
    initial: Mapping[str, torch.Tensor],
    final: Mapping[str, torch.Tensor],
    deltas: DomainDeltas,
) -> dict[str, float]:
    squared_error = 0.0
    squared_reference = 0.0
    max_abs = 0.0
    for name, start in initial.items():
        if name not in next(iter(deltas.values())):
            continue
        reconstructed = start.float().clone()
        for domain_delta in deltas.values():
            reconstructed.add_(domain_delta[name].float())
        target = final[name].float()
        error = reconstructed - target
        squared_error += float(torch.sum(error * error).item())
        squared_reference += float(torch.sum(target * target).item())
        if error.numel():
            max_abs = max(max_abs, float(error.abs().max().item()))
    l2 = math.sqrt(squared_error)
    return {
        "delta_reconstruction_l2": l2,
        "delta_reconstruction_relative_l2": l2
        / max(math.sqrt(squared_reference), 1e-12),
        "delta_reconstruction_max_abs": max_abs,
    }


def _aggregate_retained_rows(
    base: Mapping[str, object],
    model_name: str,
    per_domain_metrics: Mapping[str, Mapping[str, float]],
    retained: Sequence[str],
) -> dict[str, object]:
    row: dict[str, object] = dict(base)
    row.update(
        {
            "model": model_name,
            "evaluation_domain": "RETAINED_MACRO",
            "evaluation_scope": "retained_macro",
        }
    )
    for key in PERFORMANCE_KEYS:
        row[key] = float(np.mean([per_domain_metrics[name][key] for name in retained]))
    for key in (
        "true_negative",
        "false_positive",
        "false_negative",
        "true_positive",
        "samples",
    ):
        row[key] = float(np.sum([per_domain_metrics[name][key] for name in retained]))
    total_samples = sum(per_domain_metrics[name]["samples"] for name in retained)
    row["attack_rate"] = float(
        sum(
            per_domain_metrics[name]["attack_rate"]
            * per_domain_metrics[name]["samples"]
            for name in retained
        )
        / max(total_samples, 1.0)
    )
    return row


def _clear_accelerator(device: torch.device) -> None:
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()


def _training_metadata(result: TrainingResult) -> dict[str, object]:
    return {
        "elapsed_seconds": result.elapsed_seconds,
        "optimization_seconds": result.optimization_seconds,
        "validation_seconds": result.validation_seconds,
        "validation_evaluations": result.validation_evaluations,
        "optimizer_steps": result.optimizer_steps,
        "samples_seen": result.samples_seen,
        "best_epoch": result.best_epoch,
        "best_validation_loss": result.best_validation_loss,
        "checkpoint_bytes": result.checkpoint_bytes,
        "domain_delta_bytes": result.delta_bytes,
    }


def _assert_state_equal(
    observed: Mapping[str, torch.Tensor],
    expected: Mapping[str, torch.Tensor],
    context: str,
) -> None:
    if set(observed) != set(expected):
        raise RuntimeError(f"{context}: state keys differ")
    for name, value in observed.items():
        if not torch.equal(value.detach().cpu(), expected[name].detach().cpu()):
            raise RuntimeError(f"{context}: original state changed at {name}")


def _predict_state(
    architecture: str,
    state: Mapping[str, torch.Tensor],
    input_dim: int,
    model_config: ModelConfig,
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    forget_domain: str,
    device: torch.device,
    batch_size: int,
) -> tuple[dict[str, np.ndarray], np.ndarray]:
    """Load one state, predict every test domain and forgotten training rows."""

    model = build_model(architecture, input_dim, model_config)
    load_state_dict(model, state)
    test_probabilities = {
        name: predict_proba(model, domains[name].x_test, device, batch_size)
        for name in domain_names
    }
    forgotten_train_probabilities = predict_proba(
        model, domains[forget_domain].x_train, device, batch_size
    )
    model.to("cpu")
    del model
    _clear_accelerator(device)
    for name, probabilities in test_probabilities.items():
        if not np.isfinite(probabilities).all():
            raise FloatingPointError(f"Non-finite test probability for {name}")
        if not np.allclose(probabilities.sum(1), 1.0, atol=1e-5):
            raise FloatingPointError(f"Probabilities do not sum to one for {name}")
    return test_probabilities, forgotten_train_probabilities


@torch.inference_mode()
def _extract_penultimate_features(
    model: torch.nn.Module,
    values: np.ndarray,
    device: torch.device,
    batch_size: int,
) -> np.ndarray:
    """Extract the representation immediately before the final classifier."""

    if not hasattr(model, "forward_features"):
        raise TypeError("Model must expose forward_features() for representation CKA")
    model.eval()
    model.to(device)
    chunks: list[np.ndarray] = []
    for start in range(0, len(values), batch_size):
        x = torch.from_numpy(values[start : start + batch_size]).to(device)
        features = model.forward_features(x)
        chunks.append(features.detach().cpu().numpy())
    if not chunks:
        return np.empty((0, 0), dtype=np.float32)
    result = np.concatenate(chunks, axis=0)
    if not np.isfinite(result).all():
        raise FloatingPointError("Non-finite penultimate representation")
    return result


def _feature_state(
    architecture: str,
    state: Mapping[str, torch.Tensor],
    input_dim: int,
    model_config: ModelConfig,
    domains: Mapping[str, DomainSplit],
    domain_names: Sequence[str],
    device: torch.device,
    batch_size: int,
) -> dict[str, np.ndarray]:
    """Load one state and extract test-set penultimate representations by domain."""

    model = build_model(architecture, input_dim, model_config)
    load_state_dict(model, state)
    representations = {
        name: _extract_penultimate_features(model, domains[name].x_test, device, batch_size)
        for name in domain_names
    }
    model.to("cpu")
    del model
    _clear_accelerator(device)
    return representations


def _retained_validation_log_loss(
    architecture: str,
    state: Mapping[str, torch.Tensor],
    input_dim: int,
    model_config: ModelConfig,
    domains: Mapping[str, DomainSplit],
    retained: Sequence[str],
    device: torch.device,
    batch_size: int,
) -> float:
    """Mean cross-entropy on retained-domain validation rows (never test)."""

    from sklearn.metrics import log_loss

    model = build_model(architecture, input_dim, model_config)
    load_state_dict(model, state)
    labels = np.concatenate([domains[name].y_val for name in retained])
    probabilities = np.concatenate(
        [predict_proba(model, domains[name].x_val, device, batch_size) for name in retained]
    )
    model.to("cpu")
    del model
    _clear_accelerator(device)
    probabilities = np.clip(probabilities, 1e-12, 1.0)
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    return float(log_loss(labels, probabilities, labels=[0, 1]))


IDENTITY_CHANGE_THRESHOLD = 1e-6


def _method_preparation_accounting(
    method: str,
    full_fisher: FisherEstimate | None,
    full_fisher_usage: ResourceUsage | None,
    dc_details: Mapping[str, object] | None,
    dc_usage: ResourceUsage | None,
    unbalanced_dc_details: Mapping[str, object] | None,
    unbalanced_dc_usage: ResourceUsage | None,
    delta_bytes: int,
) -> dict[str, object]:
    if method == "ssd_canonical" and full_fisher is not None:
        return {
            "offline_preparation_seconds": full_fisher.elapsed_seconds,
            "offline_preparation_samples": full_fisher.samples,
            "offline_preparation_batches": full_fisher.batches,
            "offline_preparation_storage_bytes": full_fisher.storage_bytes,
            "offline_preparation_kind": "full-training Fisher cache",
            **_resource_dict("offline_preparation", full_fisher_usage),
        }
    if method in {"dc_ssd", "dc_ssd_no_contrast"} and dc_details is not None:
        return {
            "offline_preparation_seconds": dc_details["elapsed_seconds"],
            "offline_preparation_samples": dc_details["samples_seen"],
            "offline_preparation_batches": dc_details["batches"],
            "offline_preparation_storage_bytes": dc_details["storage_bytes"],
            "offline_preparation_kind": "class-balanced per-domain Fisher bank",
            **_resource_dict("offline_preparation", dc_usage),
        }
    if method == "dc_ssd_no_class_balance" and unbalanced_dc_details is not None:
        return {
            "offline_preparation_seconds": unbalanced_dc_details["elapsed_seconds"],
            "offline_preparation_samples": unbalanced_dc_details["samples_seen"],
            "offline_preparation_batches": unbalanced_dc_details["batches"],
            "offline_preparation_storage_bytes": unbalanced_dc_details["storage_bytes"],
            "offline_preparation_kind": "empirical-weighted class-conditional per-domain Fisher bank",
            **_resource_dict("offline_preparation", unbalanced_dc_usage),
        }
    if method == "rollback_repair":
        return {
            "offline_preparation_seconds": math.nan,
            "offline_preparation_samples": 0,
            "offline_preparation_batches": 0,
            "offline_preparation_storage_bytes": delta_bytes,
            "offline_preparation_kind": (
                "domain update trace; incremental tracing time not separately measured"
            ),
            **_resource_dict("offline_preparation", None),
        }
    return {
        "offline_preparation_seconds": 0.0,
        "offline_preparation_samples": 0,
        "offline_preparation_batches": 0,
        "offline_preparation_storage_bytes": 0,
        "offline_preparation_kind": "none",
        **_resource_dict("offline_preparation", None),
    }


def run_experiment(config: ExperimentConfig, run_dir: str | Path | None = None) -> Path:
    """Run both architectures, all deletion domains, and every enabled method."""

    config.validate()
    if run_dir is None:
        stamp = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
        destination = Path(config.runtime.output_dir) / stamp
    else:
        destination = Path(run_dir)
    destination = destination.expanduser().resolve()
    destination.mkdir(parents=True, exist_ok=False)
    config.to_json(destination / "resolved_config.json")

    device = resolve_device(config.runtime.device)
    environment = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "device": str(device),
        "cuda_device": (
            torch.cuda.get_device_name(device) if device.type == "cuda" else None
        ),
        "feature_schema_scope": (
            "predeclared" if config.data.common_features is not None else "observed"
        ),
        "transform_scope": (
            "stateless_data_independent"
            if config.data.scaler in {"fixed_log", "none"}
            else "learned_exploratory"
        ),
        "privacy_statement": "Membership attacks are empirical audits, not proofs.",
        "method_scope": (
            "Published SSD/SCRUB mechanisms are adapted to numerical NetFlow models; "
            "DC-SSD is the explicitly proposed method."
        ),
    }
    _json_dump(destination / "environment.json", environment)

    print(f"[data] preparing {len(config.data.datasets)} domains")
    prepared = prepare_data(config.data, destination / "data")
    domain_names = list(prepared.domains)
    ssd_preflight = ssd_threshold_preflight(prepared.domains, config.unlearning)
    _json_dump(destination / "ssd_threshold_preflight.json", ssd_preflight)
    unreachable = [row for row in ssd_preflight if row["structural_noop"]]
    if unreachable:
        warnings.warn(
            f"{len(unreachable)} SSD threshold/domain pairs cannot select weights "
            "under this batch estimator (up to rounding). They are retained as "
            "explicit identity controls; see ssd_threshold_preflight.json. "
            "No hyperparameter is changed automatically.",
            RuntimeWarning,
            stacklevel=2,
        )
    print(
        f"[data] {prepared.input_dim} predeclared common features; domains: "
        + ", ".join(domain_names)
    )

    all_metric_rows: list[dict[str, object]] = []
    all_similarity_rows: list[dict[str, object]] = []
    all_mia_rows: list[dict[str, object]] = []
    all_efficiency_rows: list[dict[str, object]] = []
    all_summary_rows: list[dict[str, object]] = []
    all_diagnostic_rows: list[dict[str, object]] = []
    expanded_methods = config.unlearning.expanded_methods()
    method_labels = [label for label, _, _ in expanded_methods]
    print(f"[methods] {method_labels}")

    for seed in config.training.seeds:
        for architecture in config.models.architectures:
            architecture_dir = destination / architecture / f"seed_{seed}"
            architecture_dir.mkdir(parents=True, exist_ok=True)
            print(f"\n[original] architecture={architecture} seed={seed}")

            seed_everything(seed, config.runtime.deterministic)
            original_model = build_model(
                architecture, prepared.input_dim, config.models
            )
            initial_state = cpu_state_dict(original_model)
            trace_updates = "rollback_repair" in config.unlearning.methods
            with ResourceMonitor(device) as original_monitor:
                original_result = train_model(
                    original_model,
                    prepared.domains,
                    config.training,
                    device,
                    seed,
                    domain_names=domain_names,
                    track_domain_deltas=trace_updates,
                    deterministic=config.runtime.deterministic,
                )
            assert original_monitor.result is not None
            delta_check: dict[str, float] = {}
            if trace_updates:
                if original_result.domain_deltas is None:
                    raise RuntimeError("Rollback baseline requires domain deltas")
                delta_check = _delta_reconstruction_error(
                    initial_state,
                    original_result.state_dict,
                    original_result.domain_deltas,
                )
                if delta_check["delta_reconstruction_relative_l2"] > 1e-4:
                    raise RuntimeError(
                        "Per-domain update bookkeeping failed reconstruction check: "
                        f"{delta_check}"
                    )

            original_metadata = {
                "architecture": architecture,
                "seed": seed,
                "input_dim": prepared.input_dim,
                "features": prepared.common_features,
                "training": _training_metadata(original_result),
                "delta_check": delta_check,
            }
            _save_checkpoint(
                architecture_dir / "original.pt",
                original_result.state_dict,
                original_metadata,
                domain_deltas=original_result.domain_deltas,
                initial_state=initial_state,
            )
            _write_rows(
                architecture_dir / "original_training_history.csv",
                original_result.history,
            )
            _json_dump(architecture_dir / "original_metadata.json", original_metadata)

            original_test_probs = {
                name: predict_proba(
                    original_model,
                    prepared.domains[name].x_test,
                    device,
                    config.training.batch_size * 4,
                )
                for name in domain_names
            }
            original_model.to("cpu")
            _clear_accelerator(device)
            original_train_probs: dict[str, np.ndarray] = {}

            full_fisher: FisherEstimate | None = None
            full_fisher_usage: ResourceUsage | None = None
            if "ssd_canonical" in config.unlearning.methods:
                print(f"[prepare] canonical SSD full Fisher: {architecture}, seed={seed}")
                with ResourceMonitor(device) as fisher_monitor:
                    full_fisher = compute_diagonal_fisher(
                        original_model,
                        prepared.domains,
                        domain_names,
                        config.unlearning.fisher_batch_size,
                        device,
                    )
                full_fisher_usage = fisher_monitor.result
                _assert_state_equal(
                    original_model.state_dict(),
                    original_result.state_dict,
                    "full Fisher preparation",
                )
                original_model.to("cpu")
                _clear_accelerator(device)

            fisher_bank: dict[str, FisherEstimate] | None = None
            dc_details: dict[str, object] | None = None
            dc_usage: ResourceUsage | None = None
            if {"dc_ssd", "dc_ssd_no_contrast"} & set(config.unlearning.methods):
                print(f"[prepare] class-balanced domain Fisher bank: {architecture}, seed={seed}")
                with ResourceMonitor(device) as dc_monitor:
                    fisher_bank, dc_details = compute_domain_class_fisher_bank(
                        original_model,
                        prepared.domains,
                        domain_names,
                        config.unlearning.fisher_batch_size,
                        device,
                    )
                dc_usage = dc_monitor.result
                _assert_state_equal(
                    original_model.state_dict(),
                    original_result.state_dict,
                    "DC-SSD Fisher preparation",
                )
                original_model.to("cpu")
                _clear_accelerator(device)

            unbalanced_fisher_bank: dict[str, FisherEstimate] | None = None
            unbalanced_dc_details: dict[str, object] | None = None
            unbalanced_dc_usage: ResourceUsage | None = None
            if "dc_ssd_no_class_balance" in config.unlearning.methods:
                print(f"[prepare] empirical-weighted class-conditional Fisher bank: {architecture}, seed={seed}")
                with ResourceMonitor(device) as unbalanced_dc_monitor:
                    unbalanced_fisher_bank, unbalanced_dc_details = compute_domain_fisher_bank(
                        original_model,
                        prepared.domains,
                        domain_names,
                        config.unlearning.fisher_batch_size,
                        device,
                    )
                unbalanced_dc_usage = unbalanced_dc_monitor.result
                _assert_state_equal(
                    original_model.state_dict(),
                    original_result.state_dict,
                    "empirical-weighted class-conditional DC-SSD Fisher preparation",
                )
                original_model.to("cpu")
                _clear_accelerator(device)

            preparation_metadata = {
                "canonical_ssd": (
                    {
                        "elapsed_seconds": full_fisher.elapsed_seconds,
                        "samples_seen": full_fisher.samples,
                        "batches": full_fisher.batches,
                        "storage_bytes": full_fisher.storage_bytes,
                    }
                    if full_fisher is not None
                    else None
                ),
                "dc_ssd_class_balanced_bank": dc_details,
                "dc_ssd_unbalanced_bank": unbalanced_dc_details,
                "warning": (
                    "Fisher caches are data-derived artifacts. A deployment must purge "
                    "the forgotten domain's cached entry after a deletion."
                ),
            }
            _json_dump(architecture_dir / "fisher_preparation.json", preparation_metadata)

            for forget_domain in domain_names:
                retained = [name for name in domain_names if name != forget_domain]
                scenario_dir = architecture_dir / f"forget_{_safe_name(forget_domain)}"
                scenario_dir.mkdir(parents=True, exist_ok=True)
                print(
                    f"[forget] architecture={architecture} seed={seed} "
                    f"domain={forget_domain}"
                )

                method_results: dict[str, MethodResult] = {}
                method_resources: dict[str, ResourceUsage] = {}
                method_base: dict[str, str] = {}
                for method_label, method, overrides in expanded_methods:
                    print(f"  [method] {method_label}")
                    method_base[method_label] = method
                    seed_everything(seed, config.runtime.deterministic)
                    candidate = build_model(
                        architecture, prepared.input_dim, config.models
                    )
                    if method == "rollback_repair":
                        if original_result.domain_deltas is None:
                            raise RuntimeError("Missing update trace for rollback")
                        with ResourceMonitor(device) as method_monitor:
                            result = run_rollback_repair(
                                candidate,
                                original_result.state_dict,
                                original_result.domain_deltas,
                                forget_domain,
                                prepared.domains,
                                config.training,
                                config.unlearning,
                                device,
                                _scenario_seed(seed, forget_domain, method),
                            )
                    elif method == "ssd_canonical":
                        if full_fisher is None:
                            raise RuntimeError("Canonical SSD full Fisher was not prepared")
                        with ResourceMonitor(device) as method_monitor:
                            result = run_canonical_ssd(
                                candidate,
                                original_result.state_dict,
                                full_fisher,
                                forget_domain,
                                prepared.domains,
                                config.unlearning,
                                device,
                                alpha=overrides.get("alpha"),
                                method_label=method_label,
                            )
                    elif method in {"dc_ssd", "dc_ssd_no_contrast"}:
                        if fisher_bank is None:
                            raise RuntimeError("Class-balanced DC-SSD Fisher bank was not prepared")
                        with ResourceMonitor(device) as method_monitor:
                            result = run_dc_ssd(
                                candidate,
                                original_result.state_dict,
                                fisher_bank,
                                forget_domain,
                                config.unlearning,
                                device,
                                method_label=method_label,
                                class_balanced=True,
                                contrastive=(method == "dc_ssd"),
                            )
                    elif method == "dc_ssd_no_class_balance":
                        if unbalanced_fisher_bank is None:
                            raise RuntimeError("Empirical-weighted class-conditional Fisher bank was not prepared")
                        with ResourceMonitor(device) as method_monitor:
                            result = run_dc_ssd(
                                candidate,
                                original_result.state_dict,
                                unbalanced_fisher_bank,
                                forget_domain,
                                config.unlearning,
                                device,
                                method_label=method_label,
                                class_balanced=False,
                                contrastive=True,
                            )
                    elif method == "scrub_r":
                        teacher = build_model(
                            architecture, prepared.input_dim, config.models
                        )
                        with ResourceMonitor(device) as method_monitor:
                            result = run_scrub_r(
                                candidate,
                                teacher,
                                original_result.state_dict,
                                forget_domain,
                                prepared.domains,
                                config.training,
                                config.unlearning,
                                device,
                                _scenario_seed(seed, forget_domain, method),
                                learning_rate=overrides.get("learning_rate"),
                                method_label=method_label,
                            )
                        teacher.to("cpu")
                        del teacher
                    else:
                        raise ValueError(f"Unknown unlearning method {method!r}")
                    assert method_monitor.result is not None
                    if result.method != method_label:
                        raise RuntimeError(
                            f"Method returned label {result.method!r}, expected {method_label!r}"
                        )
                    if any(not torch.isfinite(value).all() for value in result.state_dict.values()):
                        raise FloatingPointError(f"{method} produced non-finite weights")
                    _assert_state_equal(
                        original_result.state_dict,
                        cpu_state_dict(original_model),
                        f"after {method_label}",
                    )
                    change = float(result.diagnostics.get("parameter_change_l2", math.nan))
                    identity = all(
                        torch.equal(value, original_result.state_dict[name])
                        for name, value in result.state_dict.items()
                    )
                    result.diagnostics["is_identity_map"] = identity
                    result.diagnostics["is_near_identity"] = bool(change < IDENTITY_CHANGE_THRESHOLD)
                    if identity:
                        warnings.warn(
                            f"{method_label} left the model exactly unchanged "
                            f"(parameter_change_l2={change:.3g}). Interpret this "
                            "identity result against original-versus-gold differences.",
                            RuntimeWarning,
                            stacklevel=2,
                        )
                    method_results[method_label] = result
                    method_resources[method_label] = method_monitor.result
                    candidate.to("cpu")
                    del candidate
                    _clear_accelerator(device)

                # Gold standard: same exact initialization and training protocol,
                # but the forgotten domain is absent from both training and validation.
                seed_everything(seed, config.runtime.deterministic)
                gold_model = build_model(
                    architecture, prepared.input_dim, config.models
                )
                load_state_dict(gold_model, initial_state)
                _assert_state_equal(
                    gold_model.state_dict(), initial_state, "gold initialization"
                )
                with ResourceMonitor(device) as retraining_monitor:
                    retraining_result = train_model(
                        gold_model,
                        prepared.domains,
                        config.training,
                        device,
                        seed,
                        domain_names=retained,
                        track_domain_deltas=False,
                        deterministic=config.runtime.deterministic,
                    )
                assert retraining_monitor.result is not None
                gold_model.to("cpu")
                _clear_accelerator(device)

                _save_checkpoint(
                    scenario_dir / "retrained_gold.pt",
                    retraining_result.state_dict,
                    {
                        "architecture": architecture,
                        "seed": seed,
                        "forgotten_domain": forget_domain,
                        "retained_domains": retained,
                        "training": _training_metadata(retraining_result),
                    },
                )
                _write_rows(
                    scenario_dir / "retrained_training_history.csv",
                    retraining_result.history,
                )
                for method, result in method_results.items():
                    _save_checkpoint(
                        scenario_dir / f"{method}.pt",
                        result.state_dict,
                        {
                            "architecture": architecture,
                            "seed": seed,
                            "forgotten_domain": forget_domain,
                            "method": method,
                            "elapsed_seconds": result.elapsed_seconds,
                            "diagnostics": result.diagnostics,
                        },
                    )
                    _json_dump(
                        scenario_dir / f"{method}_diagnostics.json",
                        result.diagnostics,
                    )
                    if result.history:
                        _write_rows(
                            scenario_dir / f"{method}_history.csv", result.history
                        )

                if forget_domain not in original_train_probs:
                    original_train_probs[forget_domain] = predict_proba(
                        original_model,
                        prepared.domains[forget_domain].x_train,
                        device,
                        config.training.batch_size * 4,
                    )
                    original_model.to("cpu")
                    _clear_accelerator(device)

                test_probabilities: dict[str, dict[str, np.ndarray]] = {
                    "original": original_test_probs
                }
                train_probabilities: dict[str, np.ndarray] = {
                    "original": original_train_probs[forget_domain]
                }
                for method, result in method_results.items():
                    test_probs, train_probs = _predict_state(
                        architecture,
                        result.state_dict,
                        prepared.input_dim,
                        config.models,
                        prepared.domains,
                        domain_names,
                        forget_domain,
                        device,
                        config.training.batch_size * 4,
                    )
                    test_probabilities[method] = test_probs
                    train_probabilities[method] = train_probs
                gold_test_probs, gold_train_probs = _predict_state(
                    architecture,
                    retraining_result.state_dict,
                    prepared.input_dim,
                    config.models,
                    prepared.domains,
                    domain_names,
                    forget_domain,
                    device,
                    config.training.batch_size * 4,
                )
                test_probabilities["retrained"] = gold_test_probs
                train_probabilities["retrained"] = gold_train_probs

                base: dict[str, object] = {
                    "architecture": architecture,
                    "seed": seed,
                    "forgotten_domain": forget_domain,
                }
                candidate_states: dict[str, Mapping[str, torch.Tensor]] = {
                    "original": original_result.state_dict,
                    **{label: result.state_dict for label, result in method_results.items()},
                    "retrained": retraining_result.state_dict,
                }
                retained_val_log_loss = {
                    name: _retained_validation_log_loss(
                        architecture,
                        state,
                        prepared.input_dim,
                        config.models,
                        prepared.domains,
                        retained,
                        device,
                        config.training.batch_size * 4,
                    )
                    for name, state in candidate_states.items()
                }
                model_order = ["original", *method_labels, "retrained"]
                scenario_metric_rows: list[dict[str, object]] = []
                metric_lookup: dict[str, dict[str, dict[str, float]]] = {}
                for model_name in model_order:
                    by_domain = test_probabilities[model_name]
                    metric_lookup[model_name] = {}
                    for evaluation_domain in domain_names:
                        measured = classification_metrics(
                            prepared.domains[evaluation_domain].y_test,
                            by_domain[evaluation_domain],
                        )
                        metric_lookup[model_name][evaluation_domain] = measured
                        row = dict(base)
                        row.update(
                            {
                                "model": model_name,
                                "evaluation_domain": evaluation_domain,
                                "evaluation_scope": (
                                    "forgotten"
                                    if evaluation_domain == forget_domain
                                    else "retained"
                                ),
                                **measured,
                            }
                        )
                        scenario_metric_rows.append(row)
                    scenario_metric_rows.append(
                        _aggregate_retained_rows(
                            base, model_name, metric_lookup[model_name], retained
                        )
                    )

                gold_test_features = _feature_state(
                    architecture,
                    retraining_result.state_dict,
                    prepared.input_dim,
                    config.models,
                    prepared.domains,
                    domain_names,
                    device,
                    config.training.batch_size * 4,
                )
                scenario_similarity_rows: list[dict[str, object]] = []
                for candidate_name in model_order[:-1]:
                    candidate_test_features = _feature_state(
                        architecture,
                        candidate_states[candidate_name],
                        prepared.input_dim,
                        config.models,
                        prepared.domains,
                        domain_names,
                        device,
                        config.training.batch_size * 4,
                    )
                    for evaluation_domain in domain_names:
                        similarity = prediction_similarity(
                            test_probabilities[candidate_name][evaluation_domain],
                            test_probabilities["retrained"][evaluation_domain],
                        )
                        representation_cka = linear_cka(
                            candidate_test_features[evaluation_domain],
                            gold_test_features[evaluation_domain],
                        )
                        candidate_metrics = metric_lookup[candidate_name][evaluation_domain]
                        gold_metrics = metric_lookup["retrained"][evaluation_domain]
                        row = dict(base)
                        row.update(
                            {
                                "candidate_model": candidate_name,
                                "evaluation_domain": evaluation_domain,
                                "evaluation_scope": (
                                    "forgotten"
                                    if evaluation_domain == forget_domain
                                    else "retained"
                                ),
                                **similarity,
                                "linear_cka_to_retrained": representation_cka,
                                "f1_gap_to_retrained": candidate_metrics["f1_attack"]
                                - gold_metrics["f1_attack"],
                                "roc_auc_gap_to_retrained": candidate_metrics["roc_auc"]
                                - gold_metrics["roc_auc"],
                                "log_loss_gap_to_retrained": candidate_metrics["log_loss"]
                                - gold_metrics["log_loss"],
                            }
                        )
                        scenario_similarity_rows.append(row)
                    del candidate_test_features
                del gold_test_features

                member_indices, nonmember_indices, _ = matched_membership_indices(
                    prepared.domains[forget_domain].y_train,
                    prepared.domains[forget_domain].y_test,
                    config.attack.max_samples_per_class,
                    _scenario_seed(seed, forget_domain, "mia-sample"),
                )
                scenario_mia_rows: list[dict[str, object]] = []
                for model_name in model_order:
                    attack = loss_membership_attack(
                        prepared.domains[forget_domain].y_train,
                        train_probabilities[model_name],
                        prepared.domains[forget_domain].y_test,
                        test_probabilities[model_name][forget_domain],
                        config.attack.max_samples_per_class,
                        config.attack.fixed_fpr,
                        _scenario_seed(seed, forget_domain, "mia-bootstrap"),
                        bootstrap_repetitions=config.attack.bootstrap_repetitions,
                        confidence_level=config.attack.confidence_level,
                        member_indices=member_indices,
                        nonmember_indices=nonmember_indices,
                        member_group_ids=(
                            prepared.domains[forget_domain].train_group_ids
                            if config.data.split_strategy == "group_stratified" else None
                        ),
                        nonmember_group_ids=(
                            prepared.domains[forget_domain].test_group_ids
                            if config.data.split_strategy == "group_stratified" else None
                        ),
                    )
                    row = dict(base)
                    if config.data.split_strategy == "group_stratified" and (
                        attack["mia_bootstrap_unit"] != "conversation_group"
                        or attack["mia_learned_split_unit"] != "conversation_group"
                    ):
                        raise RuntimeError("Grouped data requires a group-aware membership audit")
                    row.update({"model": model_name, **attack})
                    scenario_mia_rows.append(row)
                mia_lookup = {str(row["model"]): row for row in scenario_mia_rows}
                mia_lookup["retrained"].update(
                    {
                        "mia_auc_gap_to_retrained": 0.0,
                        "mia_auc_gap_to_retrained_ci_lower": 0.0,
                        "mia_auc_gap_to_retrained_ci_upper": 0.0,
                    }
                )
                for model_name in model_order[:-1]:
                    paired_gap = paired_loss_mia_auc_gap(
                        prepared.domains[forget_domain].y_train,
                        train_probabilities[model_name],
                        train_probabilities["retrained"],
                        prepared.domains[forget_domain].y_test,
                        test_probabilities[model_name][forget_domain],
                        test_probabilities["retrained"][forget_domain],
                        member_indices,
                        nonmember_indices,
                        config.attack.fixed_fpr,
                        config.attack.bootstrap_repetitions,
                        config.attack.confidence_level,
                        _scenario_seed(seed, forget_domain, "mia-paired-gap"),
                        member_group_ids=(
                            prepared.domains[forget_domain].train_group_ids
                            if config.data.split_strategy == "group_stratified" else None
                        ),
                        nonmember_group_ids=(
                            prepared.domains[forget_domain].test_group_ids
                            if config.data.split_strategy == "group_stratified" else None
                        ),
                    )
                    if config.data.split_strategy == "group_stratified" and (
                        paired_gap["mia_gap_bootstrap_unit"] != "conversation_group"
                    ):
                        raise RuntimeError("Paired membership intervals must resample groups")
                    mia_lookup[model_name].update(paired_gap)

                retained_macro = {
                    str(row["model"]): row
                    for row in scenario_metric_rows
                    if row["evaluation_scope"] == "retained_macro"
                }
                forgotten_rows = {
                    str(row["model"]): row
                    for row in scenario_metric_rows
                    if row["evaluation_scope"] == "forgotten"
                }
                scenario_efficiency_rows: list[dict[str, object]] = []
                scenario_summary_rows: list[dict[str, object]] = []
                for method, result in method_results.items():
                    preparation = _method_preparation_accounting(
                        method_base[method],
                        full_fisher,
                        full_fisher_usage,
                        dc_details,
                        dc_usage,
                        unbalanced_dc_details,
                        unbalanced_dc_usage,
                        original_result.delta_bytes,
                    )
                    efficiency: dict[str, object] = {
                        **base,
                        "method": method,
                        "device": str(device),
                        "parameter_count": parameter_count(original_model),
                        "checkpoint_bytes": original_result.checkpoint_bytes,
                        "original_training_optimizer_steps": original_result.optimizer_steps,
                        "original_training_samples_seen": original_result.samples_seen,
                        "online_optimizer_steps": result.optimizer_steps,
                        "online_gradient_sample_exposures": result.gradient_sample_exposures,
                        "online_unique_forget_samples": result.unique_forget_samples,
                        "online_unique_retain_samples": result.unique_retain_samples,
                        "online_selection_inference_samples": result.selection_inference_samples,
                        "online_temporary_storage_bytes": result.temporary_storage_bytes,
                        "online_seconds": result.elapsed_seconds,
                        "online_optimization_seconds": result.optimization_seconds,
                        "full_retraining_optimizer_steps": retraining_result.optimizer_steps,
                        "full_retraining_samples_seen": retraining_result.samples_seen,
                        "full_retraining_optimization_seconds": retraining_result.optimization_seconds,
                        "full_retraining_validation_seconds": retraining_result.validation_seconds,
                        **preparation,
                        **_resource_dict("original_training", original_monitor.result),
                        **_resource_dict("online_unlearning", method_resources[method]),
                        **_resource_dict("full_retraining", retraining_monitor.result),
                    }
                    efficiency["online_speedup_vs_retraining_optimization"] = float(
                        retraining_result.optimization_seconds
                        / max(result.elapsed_seconds, 1e-12)
                    )
                    efficiency["monitor_scope_speedup_vs_retraining"] = (
                        retraining_monitor.result.elapsed_seconds
                        / max(method_resources[method].elapsed_seconds, 1e-12)
                    )
                    efficiency["gradient_exposure_reduction_fraction"] = float(
                        1.0
                        - result.gradient_sample_exposures
                        / max(retraining_result.samples_seen, 1)
                    )
                    scenario_efficiency_rows.append(efficiency)

                    forgotten_similarity = next(
                        row
                        for row in scenario_similarity_rows
                        if row["candidate_model"] == method
                        and row["evaluation_domain"] == forget_domain
                    )
                    retained_similarity_rows = [
                        row
                        for row in scenario_similarity_rows
                        if row["candidate_model"] == method
                        and row["evaluation_domain"] in retained
                    ]
                    diagnostics = result.diagnostics
                    summary: dict[str, object] = {
                        **base,
                        "method": method,
                        "base_method": method_base[method],
                        "original_retained_validation_log_loss": retained_val_log_loss["original"],
                        "method_retained_validation_log_loss": retained_val_log_loss[method],
                        "retrained_retained_validation_log_loss": retained_val_log_loss["retrained"],
                        "is_identity_map": diagnostics.get("is_identity_map", False),
                        "is_near_identity": diagnostics.get("is_near_identity", False),
                        "ssd_selection_status": diagnostics.get("selection_status"),
                        "ssd_structural_noop": diagnostics.get("structural_noop"),
                        "ssd_batch_ratio_upper_bound": diagnostics.get(
                            "batch_normalized_ratio_upper_bound", math.nan
                        ),
                        "original_retained_macro_f1": retained_macro["original"]["f1_attack"],
                        "method_retained_macro_f1": retained_macro[method]["f1_attack"],
                        "retrained_retained_macro_f1": retained_macro["retrained"]["f1_attack"],
                        "original_retained_macro_roc_auc": retained_macro["original"]["roc_auc"],
                        "method_retained_macro_roc_auc": retained_macro[method]["roc_auc"],
                        "retrained_retained_macro_roc_auc": retained_macro["retrained"]["roc_auc"],
                        "original_forgotten_f1": forgotten_rows["original"]["f1_attack"],
                        "method_forgotten_f1": forgotten_rows[method]["f1_attack"],
                        "retrained_forgotten_f1": forgotten_rows["retrained"]["f1_attack"],
                        "original_forgotten_roc_auc": forgotten_rows["original"]["roc_auc"],
                        "method_forgotten_roc_auc": forgotten_rows[method]["roc_auc"],
                        "retrained_forgotten_roc_auc": forgotten_rows["retrained"]["roc_auc"],
                        "original_mia_auc": mia_lookup["original"]["mia_loss_auc"],
                        "method_mia_auc": mia_lookup[method]["mia_loss_auc"],
                        "retrained_mia_auc": mia_lookup["retrained"]["mia_loss_auc"],
                        "method_abs_mia_auc_gap_from_retrained": abs(
                            float(mia_lookup[method]["mia_auc_gap_to_retrained"])
                        ),
                        "method_forgotten_prediction_agreement_with_retrained": forgotten_similarity[
                            "prediction_agreement"
                        ],
                        "method_forgotten_js_divergence_from_retrained": forgotten_similarity[
                            "mean_jensen_shannon_divergence"
                        ],
                        "method_forgotten_linear_cka_to_retrained": forgotten_similarity[
                            "linear_cka_to_retrained"
                        ],
                        "method_retained_macro_linear_cka_to_retrained": float(
                            np.mean([row["linear_cka_to_retrained"] for row in retained_similarity_rows])
                        ),
                        "method_forgotten_abs_f1_gap_from_retrained": abs(
                            float(forgotten_similarity["f1_gap_to_retrained"])
                        ),
                        "online_seconds": result.elapsed_seconds,
                        "full_retraining_optimization_seconds": retraining_result.optimization_seconds,
                        "online_speedup_vs_retraining_optimization": efficiency[
                            "online_speedup_vs_retraining_optimization"
                        ],
                        "selected_parameter_fraction": diagnostics.get(
                            "selected_parameter_fraction", math.nan
                        ),
                        "fisher_cosine_similarity": diagnostics.get(
                            "reference_forget_fisher_cosine_similarity", math.nan
                        ),
                        "parameter_change_l2": diagnostics.get(
                            "parameter_change_l2", math.nan
                        ),
                    }
                    scenario_summary_rows.append(summary)

                    diagnostic_row = {
                        **base,
                        "method": method,
                        "base_method": method_base[method],
                        **{
                            key: value
                            for key, value in diagnostics.items()
                            if not isinstance(value, (dict, list))
                        },
                    }
                    all_diagnostic_rows.append(diagnostic_row)

                _write_rows(
                    scenario_dir / "classification_metrics.csv", scenario_metric_rows
                )
                _write_rows(
                    scenario_dir / "gold_similarity.csv", scenario_similarity_rows
                )
                _write_rows(
                    scenario_dir / "membership_inference.csv", scenario_mia_rows
                )
                _write_rows(scenario_dir / "efficiency.csv", scenario_efficiency_rows)
                _write_rows(scenario_dir / "summary.csv", scenario_summary_rows)

                if config.runtime.save_predictions:
                    arrays: dict[str, np.ndarray] = {}
                    for model_name, by_domain in test_probabilities.items():
                        for domain_name, probabilities in by_domain.items():
                            arrays[f"{model_name}__{_safe_name(domain_name)}__test"] = probabilities
                    for model_name, probabilities in train_probabilities.items():
                        arrays[f"{model_name}__{_safe_name(forget_domain)}__train"] = probabilities
                    split = prepared.domains[forget_domain]
                    arrays["forgotten_train_record_ids"] = split.train_record_ids
                    arrays["forgotten_test_record_ids"] = split.test_record_ids
                    arrays["forgotten_train_labels"] = split.y_train
                    arrays["forgotten_test_labels"] = split.y_test
                    arrays["mia_member_indices"] = member_indices
                    arrays["mia_nonmember_indices"] = nonmember_indices
                    np.savez_compressed(scenario_dir / "predictions.npz", **arrays)

                all_metric_rows.extend(scenario_metric_rows)
                all_similarity_rows.extend(scenario_similarity_rows)
                all_mia_rows.extend(scenario_mia_rows)
                all_efficiency_rows.extend(scenario_efficiency_rows)
                all_summary_rows.extend(scenario_summary_rows)
                del method_results, test_probabilities, train_probabilities, gold_model
                _clear_accelerator(device)

            del original_model, original_test_probs, original_train_probs
            del full_fisher, fisher_bank, unbalanced_fisher_bank
            _clear_accelerator(device)

    deletion_scenarios = (
        len(config.models.architectures)
        * len(config.training.seeds)
        * len(domain_names)
    )
    method_scenarios = deletion_scenarios * len(method_labels)
    model_count = len(method_labels) + 2  # original + methods + gold
    expected_counts = {
        "classification": deletion_scenarios * model_count * (len(domain_names) + 1),
        "similarity": deletion_scenarios * (model_count - 1) * len(domain_names),
        "membership": deletion_scenarios * model_count,
        "efficiency": method_scenarios,
        "diagnostics": method_scenarios,
        "summary": method_scenarios,
    }
    row_collections = {
        "classification": all_metric_rows,
        "similarity": all_similarity_rows,
        "membership": all_mia_rows,
        "efficiency": all_efficiency_rows,
        "diagnostics": all_diagnostic_rows,
        "summary": all_summary_rows,
    }
    uniqueness_keys = {
        "classification": [
            "architecture", "seed", "forgotten_domain", "model", "evaluation_domain"
        ],
        "similarity": [
            "architecture", "seed", "forgotten_domain", "candidate_model",
            "evaluation_domain"
        ],
        "membership": ["architecture", "seed", "forgotten_domain", "model"],
        "efficiency": ["architecture", "seed", "forgotten_domain", "method"],
        "diagnostics": ["architecture", "seed", "forgotten_domain", "method"],
        "summary": ["architecture", "seed", "forgotten_domain", "method"],
    }
    for table_name, rows in row_collections.items():
        if len(rows) != expected_counts[table_name]:
            raise RuntimeError(
                f"{table_name} produced {len(rows)} rows; expected "
                f"{expected_counts[table_name]}"
            )
        if pd.DataFrame(rows).duplicated(uniqueness_keys[table_name]).any():
            raise RuntimeError(f"{table_name} contains duplicate scenario rows")

    _write_rows(destination / "all_classification_metrics.csv", all_metric_rows)
    _write_rows(destination / "all_gold_similarity.csv", all_similarity_rows)
    _write_rows(destination / "all_membership_inference.csv", all_mia_rows)
    _write_rows(destination / "all_efficiency.csv", all_efficiency_rows)
    _write_rows(destination / "all_method_diagnostics.csv", all_diagnostic_rows)
    _write_rows(destination / "summary.csv", all_summary_rows)
    _write_rows(
        destination / "summary_aggregated.csv",
        _aggregate_summary_across_seeds(all_summary_rows),
    )
    _json_dump(
        destination / "completed.json",
        {
            "completed_utc": datetime.now(timezone.utc).isoformat(),
            "architectures": config.models.architectures,
            "seeds": config.training.seeds,
            "forgotten_domains": domain_names,
            "methods": method_labels,
            "base_methods": config.unlearning.methods,
            "deletion_scenarios": deletion_scenarios,
            "method_scenarios": len(all_summary_rows),
        },
    )
    print(f"\n[done] results written to {destination}")
    return destination


## 10. Tiny synthetic domains for an end-to-end smoke test

**What the following block does:** This cell creates four small artificial NetFlow-like CSV files so the complete multi-method workflow can be checked without downloading the research corpora. The domains have different feature distributions but share one learnable binary task, labels deliberately use several common encodings, and domain-only columns test the exclusion rules. Smoke mode uses both architectures, every enabled unlearning method, small Fisher batches, two SCRUB steps, one seed, and short bootstrap attacks. The synthetic label is a deterministic function of `PROTOCOL` in every domain; this intentionally easy task checks execution and label handling, not memorization, realistic intrusion detection, or unlearning effectiveness. A passing smoke test cannot establish privacy or scientific success, and its scores must never be reported as research findings.


In [ ]:
"""Small four-domain generator used only to smoke-test the full pipeline."""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd



def make_synthetic_config(root: str | Path) -> ExperimentConfig:
    root = Path(root).resolve()
    data_dir = root / "synthetic_domains"
    data_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(42)
    specs = []
    names = ["NF-SYNTH-A", "NF-SYNTH-B", "NF-SYNTH-C", "NF-SYNTH-D"]
    for domain_index, name in enumerate(names):
        rows = 320
        protocol = rng.integers(0, 4, size=rows)
        packets = rng.poisson(12 + domain_index, size=rows) + 1
        in_bytes = packets * rng.lognormal(4.0 + 0.08 * domain_index, 0.5, rows)
        out_bytes = packets * rng.lognormal(3.6 - 0.04 * domain_index, 0.6, rows)
        duration = rng.lognormal(4.0 + 0.1 * domain_index, 0.7, rows)
        # A deliberately simple, balanced task makes smoke-test utility
        # nonzero for both architectures; these scores are not research results.
        labels = (protocol >= 2).astype(np.int64)
        frame = pd.DataFrame(
            {
                "IPV4_SRC_ADDR": [
                    f"10.{domain_index}.0.{i % 255}" for i in range(rows)
                ],
                "IPV4_DST_ADDR": [
                    f"172.16.{domain_index}.{i % 255}" for i in range(rows)
                ],
                "L4_SRC_PORT": rng.integers(1_024, 65_535, rows),
                "L4_DST_PORT": rng.integers(1, 65_535, rows),
                "PROTOCOL": protocol,
                "IN_BYTES": in_bytes,
                "IN_PKTS": packets,
                "OUT_BYTES": out_bytes,
                "OUT_PKTS": rng.poisson(9, size=rows) + 1,
                "TCP_FLAGS": rng.integers(0, 32, rows),
                "FLOW_DURATION_MILLISECONDS": duration,
                "SRC_TO_DST_IAT_MAX": rng.lognormal(2.0, 0.8, rows),
                "DST_TO_SRC_IAT_MAX": rng.lognormal(1.8, 0.9, rows),
                f"DOMAIN_ONLY_{domain_index}": rng.normal(size=rows),
            }
        )
        if domain_index == 0:
            frame["Label"] = labels
        elif domain_index == 1:
            frame["LABEL"] = np.where(labels == 1, "Attack", "Benign")
        elif domain_index == 2:
            frame["label"] = labels.astype(bool)
        else:
            frame["Label"] = np.where(labels == 1, "DDoS", "Normal")
        path = data_dir / f"{name}.csv"
        frame.to_csv(path, index=False)
        specs.append(DatasetConfig(name=name, path=str(path), sample_rows=None))

    return ExperimentConfig(
        data=DataConfig(
            datasets=specs,
            common_features=[
                "PROTOCOL",
                "IN_BYTES",
                "IN_PKTS",
                "OUT_BYTES",
                "OUT_PKTS",
                "TCP_FLAGS",
                "FLOW_DURATION",
                "SRC_TO_DST_IAT_MAX",
                "DST_TO_SRC_IAT_MAX",
                "BYTES_PER_PKT",
            ],
            sample_rows_per_dataset=None,
            csv_chunk_rows=1_000,
            scaler="fixed_log",
            scaler_fit_rows=None,
            allow_kaggle_download=False,
            split_strategy="group_stratified",
            hash_source_files=False,
            drop_feature_duplicates=True,
            strict_protocol=False,
        ),
        models=ModelConfig(
            architectures=["mlp", "numerical_feature_transformer"],
            latent_dim=8,
            mlp_hidden_dims=[16],
            dropout=0.0,
            nft_d_token=8,
            nft_heads=2,
            nft_layers=1,
            nft_ffn_factor=2,
        ),
        training=TrainingConfig(
            epochs=8,
            batch_size=64,
            learning_rate=2e-2,
            patience=2,
            seeds=[42],
            strict_unlearning_protocol=False,
        ),
        unlearning=UnlearningConfig(
            repair_epochs=1,
            repair_fraction=0.25,
            repair_learning_rate=5e-4,
            fisher_batch_size=64,
            scrub_steps=2,
            scrub_max_steps=1,
            scrub_forget_batch_size=64,
            scrub_retain_batch_size=64,
            scrub_lr_milestones=[1],
            ssd_alpha_sweep=[2.0],
            scrub_learning_rate_sweep=[5e-3],
        ),
        attack=AttackConfig(
            max_samples_per_class=30,
            fixed_fpr=0.05,
            bootstrap_repetitions=25,
        ),
        runtime=RuntimeConfig(
            output_dir=str(root / "synthetic_results"),
            device="cpu",
            deterministic=True,
        ),
    )


## 11. Edit the run here

**What the following block does:** This is the only cell most students need to edit. `MODE="smoke"` generates tiny local files and checks the whole notebook; `"quick"` uses the real Kaggle/local datasets with a 10,000-row cap, two original-training epochs, one seed, fewer attack bootstraps, and a shortened SCRUB schedule; `"full"` applies the strict comparison configuration with a uniform 250,000-row cap per source dataset and three seeds. `METHODS` can disable expensive baselines without changing their code. Local files are always tried first; when they are absent and downloads are allowed, the same Kaggle slugs and matching patterns used by the earlier notebook are applied. The printed summary makes the cap, architectures, domains, methods, seeds, device, and strict-protocol status visible before computation begins.


In [ ]:
from pathlib import Path
import tempfile

# --------------------------- EDIT THESE VALUES ---------------------------
MODE = "smoke"  # "smoke", "quick", or "full"
DATA_DIR = Path("datasets").resolve()
ALLOW_KAGGLE_DOWNLOAD = True
SAMPLE_ROWS_PER_DATASET = 250_000  # fixed uniform cap; do not call this all rows
SEEDS = [42, 1337, 2026]  # full mode requires at least three independent seeds
DROP_FEATURE_DUPLICATES = True  # dedupe identical (feature vector, label) rows per domain
SSD_ALPHA_SWEEP = [1.0, 2.0, 3.0, 5.0]  # each reported as its own canonical-SSD row
SCRUB_LR_SWEEP = [5e-3, 1e-2]  # each reported as its own SCRUB+R row
DEVICE = "auto"  # "auto", "cpu", "cuda", or "mps"
SAVE_PREDICTIONS = False
CSV_ENCODING = "utf-8"
METHODS = [
    "rollback_repair",
    "ssd_canonical",
    "dc_ssd_no_contrast",
    "dc_ssd_no_class_balance",
    "dc_ssd",
    "scrub_r",
]
# ------------------------------------------------------------------------


def make_real_config() -> ExperimentConfig:
    datasets = [
        DatasetConfig(
            name="NF-UNSW-NB15",
            path=str(DATA_DIR),
            file_pattern="**/*UNSW*N*15*v3*",
            kaggle_slug="seyhed/nf-unsw-nb15-v3",
            encoding=CSV_ENCODING,
            allow_multiple_files=False,
        ),
        DatasetConfig(
            name="NF-ToN-IoT",
            path=str(DATA_DIR),
            file_pattern="**/*ToN*IoT*v3*",
            kaggle_slug="seyhed/nf-ton-iot-v3",
            encoding=CSV_ENCODING,
            allow_multiple_files=False,
        ),
        DatasetConfig(
            name="NF-BoT-IoT",
            path=str(DATA_DIR),
            file_pattern="**/*BoT*IoT*v3*",
            kaggle_slug="ndayisabae/nf-bot-iot-v3",
            encoding=CSV_ENCODING,
            allow_multiple_files=False,
        ),
        DatasetConfig(
            name="NF-CSE-CIC-IDS2018",
            path=str(DATA_DIR),
            file_pattern="**/*CIC*IDS2018*v3*",
            kaggle_slug="seyhed/nf-cicids2018-v3",
            encoding=CSV_ENCODING,
            allow_multiple_files=False,
        ),
    ]
    return ExperimentConfig(
        data=DataConfig(
            datasets=datasets,
            common_features=PUBLIC_V3_MODEL_FEATURES.copy(),
            train_fraction=0.70,
            validation_fraction=0.15,
            test_fraction=0.15,
            seed=42,
            sample_rows_per_dataset=SAMPLE_ROWS_PER_DATASET,
            csv_chunk_rows=100_000,
            scaler="fixed_log",
            scaler_fit_rows=None,
            allow_kaggle_download=ALLOW_KAGGLE_DOWNLOAD,
            split_strategy="group_stratified",
            group_columns=["IPV4_SRC_ADDR", "IPV4_DST_ADDR", "PROTOCOL"],
            drop_exact_duplicates=True,
            drop_feature_duplicates=DROP_FEATURE_DUPLICATES,
            hash_source_files=True,
            strict_protocol=True,
        ),
        models=ModelConfig(
            architectures=["mlp", "numerical_feature_transformer"],
            latent_dim=32,
            mlp_hidden_dims=[128, 64],
            dropout=0.10,
            nft_d_token=16,
            nft_heads=4,
            nft_layers=2,
            nft_ffn_factor=4,
        ),
        training=TrainingConfig(
            epochs=20,
            batch_size=256,
            learning_rate=1e-2,
            optimizer="sgd",
            sgd_momentum=0.0,
            weight_decay=0.0,
            patience=5,
            min_delta=1e-4,
            fixed_class_weights=[1.0, 1.0],
            gradient_clip_norm=5.0,
            domain_sampling="proportional",
            checkpoint_selection="final",
            seeds=SEEDS,
            strict_unlearning_protocol=True,
        ),
        unlearning=UnlearningConfig(
            methods=METHODS.copy(),
            rollback_scale=1.0,
            repair_epochs=2,
            repair_fraction=0.25,
            repair_learning_rate=2e-3,
            fisher_batch_size=64,  # SSD authors' full-class loader default
            ssd_alpha=10.0,
            ssd_lambda=1.0,
            ssd_alpha_sweep=list(SSD_ALPHA_SWEEP),
            dc_ssd_alpha=2.0,
            dc_ssd_lambda=1.0,
            dc_ssd_reference="mean",
            scrub_steps=3,
            scrub_max_steps=2,
            scrub_forget_batch_size=32,  # SCRUB large-scale delete batch
            scrub_retain_batch_size=128,
            scrub_optimizer="sgd",
            scrub_learning_rate=5e-4,
            scrub_learning_rate_sweep=list(SCRUB_LR_SWEEP),
            scrub_momentum=0.9,
            scrub_weight_decay=5e-4,
            scrub_temperature=4.0,
            scrub_alpha=0.001,
            scrub_gamma=0.99,
            scrub_lr_milestones=[2],
            scrub_lr_decay_factor=0.1,
        ),
        attack=AttackConfig(
            max_samples_per_class=10_000,
            fixed_fpr=0.01,
            bootstrap_repetitions=1_000,
            confidence_level=0.95,
        ),
        runtime=RuntimeConfig(
            output_dir=str(Path("artifacts/netflow_unlearning").resolve()),
            device=DEVICE,
            save_predictions=SAVE_PREDICTIONS,
            deterministic=True,
        ),
    )


if MODE == "smoke":
    smoke_root = Path(tempfile.mkdtemp(prefix="netflow_unlearning_notebook_"))
    config = make_synthetic_config(smoke_root)
elif MODE in {"quick", "full"}:
    config = make_real_config()
    if MODE == "quick":
        config.data.sample_rows_per_dataset = 10_000
        config.data.hash_source_files = False
        config.data.strict_protocol = False
        config.training.epochs = 2
        config.training.seeds = config.training.seeds[:1]
        config.training.strict_unlearning_protocol = False
        config.unlearning.repair_epochs = 1
        config.unlearning.fisher_batch_size = 512
        config.unlearning.scrub_steps = 2
        config.unlearning.scrub_max_steps = 1
        config.unlearning.scrub_forget_batch_size = 256
        config.unlearning.scrub_retain_batch_size = 256
        config.unlearning.scrub_lr_milestones = [1]
        config.unlearning.ssd_alpha_sweep = [2.0]
        config.unlearning.scrub_learning_rate_sweep = [5e-3]
        config.attack.max_samples_per_class = 1_000
        config.attack.bootstrap_repetitions = 50
else:
    raise ValueError("MODE must be 'smoke', 'quick', or 'full'")

config.unlearning.methods = METHODS.copy()
config.validate()
print(f"Mode: {MODE}")
print(f"Architectures: {config.models.architectures}")
print(f"Datasets: {[dataset.name for dataset in config.data.datasets]}")
print(f"Methods: {config.unlearning.methods}")
print(f"Per-dataset row cap: {config.data.sample_rows_per_dataset}")
print(f"Seeds: {config.training.seeds}")
print(f"Device request: {config.runtime.device}")
print(f"Strict protocol checks (not scientific certification): {config.data.strict_protocol}")


## 12. Run every forget experiment

**What the following block does:** These two lines launch the full configured workflow. For each architecture and seed, the original model and reusable Fisher preparations are created once; every forgotten-domain scenario then runs each enabled approximate method from the same original state and trains one retained-only gold model. The output path is timestamped, so an existing run is never overwritten. Full mode is intentionally expensive—especially SCRUB+R and the four scratch retrainings—while smoke mode should be used first to catch environment or data-format problems.


In [ ]:
RESULT_DIR = run_experiment(config)
print(f"Completed run: {RESULT_DIR}")

## 13. Inspect the research tables

**What the following block does:** This cell loads the main long-format outputs. `summary.csv` has one row for every architecture, seed, forgotten domain, and approximate method, with original and retrained reference columns repeated for convenient comparison. `summary_aggregated.csv` reports means, standard deviations, standard errors, and 95% confidence intervals across seeds without mixing methods. `all_efficiency.csv` separates one-time Fisher/trace preparation from online deletion and records raw-data exposure, optimizer steps, memory, and speedup. `all_method_diagnostics.csv` exposes SSD selection fractions and Fisher similarity, SCRUB rewind decisions, and parameter-change magnitudes so a good-looking average cannot conceal an identity update or model collapse.


In [ ]:
summary = pd.read_csv(RESULT_DIR / "summary.csv")
summary_aggregated = pd.read_csv(RESULT_DIR / "summary_aggregated.csv")
efficiency = pd.read_csv(RESULT_DIR / "all_efficiency.csv")
method_diagnostics = pd.read_csv(RESULT_DIR / "all_method_diagnostics.csv")

display(summary.sort_values(["architecture", "forgotten_domain", "method"]))
display(summary_aggregated.sort_values(["architecture", "forgotten_domain", "method"]))
display(efficiency.sort_values(["architecture", "forgotten_domain", "method"]))
display(method_diagnostics.sort_values(["architecture", "forgotten_domain", "method"]))


## Interpretation reminder

A forgotten dataset's accuracy need not fall to chance: a model trained on the other NetFlow domains can legitimately generalize to its attacks. The correct target is closeness to retained-only retraining on forgotten-domain predictions, retained-domain utility, and membership behaviour. A membership AUC near 0.5 is not evidence of successful forgetting when retained F1 has collapsed, so always read privacy, utility, and scratch-similarity columns together.

Canonical SSD with \(\alpha=10\) cannot select weights for four approximately equal domains under this notebook's batch estimator. Because pooled importance includes exactly the same nonnegative squared batch gradients as forgotten importance, \(F_f/F_D\le B_D/B_f\), where \(B_D\) and \(B_f\) are the actual full and forgotten batch counts (not sample counts). The saved preflight flags unreachable thresholds before model training, and diagnostics record actual selection and exact identity separately from near-identity parameter changes. The explicit alpha sweep preserves both reachable settings and identity controls; being below the ceiling does not guarantee selection or effective forgetting. An identity result is not automatically good or bad: compare it with the untouched original's gap to retraining. SCRUB learning-rate sweep variants share the same random schedule, isolating learning rate rather than changing both settings and random seeds. These hyperparameters and the short SCRUB schedule are disclosed NetFlow adaptations, not tuned optima or an exact paper-experiment reproduction. All variants are reported: predeclare any selection rule before seeing test results, and never select on gold/test performance. Retained validation loss can select for utility, but cannot alone establish forgetting. DC-SSD compares the forgotten Fisher against explicit retained-domain importance and balances benign/attack classes; it remains a proposed local heuristic without a formal deletion guarantee.

SCRUB+R is not retain-free: its configured three min epochs traverse 100% of retained training data three times, while its first two max epochs traverse 100% of forgotten training data twice. DC-SSD is raw-retain-data-free only at online deletion time because its per-domain Fisher bank was prepared earlier; that cache is a data-derived artifact and the forgotten entry must be deleted in a real deployment. In this benchmarking notebook the shared bank is kept long enough to run each independent leave-one-domain-out scenario, and its full creation time and storage are reported.

Held-out rows whose feature vector also occurs in training inflate utility and deflate the membership attack; `split_manifest.json` reports that overlap per domain, and `drop_feature_duplicates=True` removes such rows before splitting. Large train-vs-test attack-rate gaps (also in the manifest) indicate a heavy-hitter endpoint group dominating one split.

Finally, the uniform 250,000-row cap means “whole-domain deletion” refers to every training row from that domain used by this experiment, not every row contained in the original Kaggle corpus. Remove the cap only after confirming that compute and storage are sufficient, and continue to report the exact sampled and split counts saved in the data provenance outputs.

Across-seed intervals describe training-seed variation on these fixed sampled partitions, not generalization to every possible corpus or split. The per-metric valid-seed count is saved; a single seed has no estimated across-seed uncertainty. The no-contrast ablation has an additional identity case: with a max reference including the forgotten domain and alpha at least one, no coordinate can be selected. Diagnostics flag it explicitly.

Before treating a full run as research evidence, check original and gold-model validation utility/convergence, the real-data split and overlap manifests, and all retained-utility/privacy/gold-similarity results together. Smoke mode validates implementation only. This notebook provides approximate empirical unlearning comparisons, not a guarantee of successful deletion.
